# Build GTFS File: LRT Jakarta Lin Selatan Fase 1A
---

**GTFS (General Transit Feed Specification)** adalah standar format data yang digunakan untuk mendeskripsikan informasi transportasi publik seperti rute, jadwal, dan halte/stasiun dalam bentuk file teks yang terstruktur. GTFS memungkinkan operator transportasi umum untuk mempublikasikan data jadwal dan rute mereka dalam format yang bisa dibaca oleh berbagai aplikasi dan platform secara seragam. Dengan GTFS, data bisa langsung diintegrasikan ke Google Maps, Apple Maps, atau aplikasi perjalanan lainnya tanpa konversi manual. Format ini memungkinkan pengembang untuk:

```
- Menampilkan rute & jadwal transit di aplikasi peta (Google Maps, Moovit, dll)
- Membangun aplikasi perencanaan perjalanan multimoda
- Menganalisis cakupan dan aksesibilitas layanan transportasi
- Berbagi data transit secara terbuka untuk publik
```

GTFS terdiri dari kumpulan file **CSV (Comma-Separated Values)** dengan ekstensi `.txt` yang dikompres menjadi satu file `.zip`. Setiap file memiliki skema kolom tertentu yang telah distandarisasi. GTFS terdiri dari dua Varian Utama yaitu:

- `GTFS Static`, data statis yang menggambarkan "rencana" layanan transportasi tidak berubah secara real-time. Biasanya diperbarui secara berkala (mingguan/bulanan) ketika ada perubahan jadwal atau rute.
- `GTFS Realtime`, data dinamis yang menggambarkan kondisi aktual layanan saat ini (Realtime). Menggunakan format Protocol Buffers (protobuf), diperbarui setiap beberapa detik hingga menit. Dengan GTFS-Realtime, Informasi keterlambatan, pembatalan, perubahan jadwal perjalanan dapat diketahui secara live, selain itu dapat mengetahui posisi GPS kendaraan secara real-time di peta. GTFS Realtime tidak bisa berdiri sendiri, membutuhkan GTFS Static sebagai acuan.

*Dokumentasi Resmi:* [GTFS Reference](https://gtfs.org/documentation/overview/)

---
Pada notebook ini akan dilakukan proses pembuatan GTFS Static untuk operasi layanan LRT Jakarta. Berikut ini file yang akan di-build pada notebook ini:

| File | Status | Deskripsi Singkat |
|------|--------|------------------|
| `agency.txt` | ✅ Required | Informasi operator (PT LRT Jakarta) |
| `stops.txt` | ✅ Required | Data halte/stasiun (nama, koordinat, zona) |
| `routes.txt` | ✅ Required | Data rute/layanan (warna, tipe, referensi agency) |
| `trips.txt` | ✅ Required | Data perjalanan spesifik (route, service_id, direction) |
| `stop_times.txt` | ✅ Required | Jadwal kedatangan/keberangkatan di tiap halte/stasiun |
| `calendar.txt` | ⚠️ Conditional | Jadwal operasional reguler (Senin-Minggu) |
| `fare_attributes.txt` | ❌ Optional | Informasi tarif/ticketing |
| `fare_rules.txt` | ❌ Optional | Aturan penerapan tarif |
| `shapes.txt` | ❌ Optional | Koordinat polyline untuk visualisasi rute di peta |

---

Setelah notebook selesai dijalankan, project ini akan memiliki struktur seperti ini:

```
gtfs_lrt_jakarta/
├── agency.txt
├── calendar.txt
├── fare_attributes.txt
├── fare_rules.txt
├── routes.txt
├── shapes.txt
├── stops.txt
├── stop_times.txt
└── trips.txt
```

**Field Types**

Berikut ini merupakan tipe data yang digunakan pada format file GTFS:

| Tipe Data | Deskripsi | Contoh | Catatan Penting |
|-----------|-----------|--------|----------------|
| **`Color`** | Warna yang dienkode sebagai angka heksadesimal 6-digit. | `FFFFFF` (putih), `000000` (hitam), `0039A6` (biru MTA NY) | Jangan sertakan tanda `#` di awal. Gunakan [htmlcolorcodes.com](https://htmlcolorcodes.com) untuk generate nilai valid. |
| **`Currency code`** | Kode mata uang alfabetis ISO 4217. | `IDR` (Rupiah), `USD` (Dolar AS), `EUR` (Euro) | Lihat daftar lengkap: [ISO 4217 Active Codes](https://en.wikipedia.org/wiki/ISO_4217#Active_codes) |
| **`Currency amount`** | Nilai desimal yang menunjukkan jumlah mata uang. | `14000.00`, `2.50` | Jumlah desimal mengikuti standar ISO 4217 untuk mata uang terkait. **Jangan proses sebagai `float`** — gunakan tipe `decimal`/`currency` untuk hindari error pembulatan finansial. |
| **`Date`** | Hari layanan dalam format `YYYYMMDD`. | `20260416` (16 April 2026) | Waktu dalam hari layanan boleh > `24:00:00`, sehingga satu hari layanan dapat mencakup informasi hari berikutnya. |
| **`Email`** | Alamat email valid. | `carla@lrtjakarta.co.id` | Gunakan format standar RFC 5322. |
| **`Enum`** | Opsi dari sekumpulan konstanta yang telah didefinisikan di kolom "Description". | `0` = Trem, `1` = Metro, `3` = Bus (untuk field `route_type`) | Selalu merujuk ke dokumentasi resmi untuk nilai yang diperbolehkan. |
| **`ID`** | Nilai ID internal (tidak untuk ditampilkan ke penumpang), berupa urutan karakter UTF-8 apa pun. | `LRTJ-ROUTE-01`, `STOP-PGD-001` | 🔹 Disarankan hanya gunakan karakter ASCII yang dapat dicetak.<br>🔹 **"Unique ID"**: Harus unik dalam satu file.<br>🔹 **"Foreign ID"**: Merujuk ke ID di file lain (misal: `routes.agency_id` merujuk ke `agency.agency_id`). |
| **`Language code`** | Kode bahasa IETF BCP 47. | `id` (Indonesia), `en` (Inggris), `en-US` (Inggris AS) | Panduan: [RFC 4646](http://www.rfc-editor.org/rfc/bcp/bcp47.txt) & [W3C Language Tags](https://www.w3.org/International/articles/language-tags/) |
| **`Latitude`** | Lintang WGS84 dalam desimal derajat. | `-6.2088` (Velodrome) | 📍 Rentang nilai: `-90.0` ≤ latitude ≤ `90.0` |
| **`Longitude`** | Bujur WGS84 dalam desimal derajat. | `106.8456` (Velodrome) | 📍 Rentang nilai: `-180.0` ≤ longitude ≤ `180.0` |
| **`Float`** | (*floating point*). | `3.14159`, `-0.001` | Hindari untuk nilai mata uang tetap gunakan `Currency amount` dengan tipe `decimal`. |
| **`Integer`** | Integer | `0`, `42`, `-7` | Cocok untuk counter, urutan, atau nilai diskrit. |
| **`Phone number`** | Nomor telepon. | `+62215000332`, `1500332` | Format bebas, tetapi disarankan gunakan format internasional E.164 untuk kompatibilitas global. |
| **`Time`** | Waktu dalam format `HH:MM:SS` (atau `H:MM:SS`). Diukur dari "noon minus 12h" hari layanan (efektif tengah malam, kecuali saat perubahan DST). | `14:30:00` (2:30 PM), `25:35:00` (1:35 AM hari berikutnya) | Untuk waktu setelah tengah malam hari layanan, gunakan nilai > `24:00:00`. Contoh: Kereta tiba pukul 01:15 dini hari → tulis `25:15:00`. |
| **`Local time`** | Waktu dalam format `HH:MM:SS` (atau `H:MM:SS`). Merepresentasikan waktu dinding (*wall-clock time*) sesuai zona waktu lokasi yang ditentukan. | `08:00:00` (jam 8 pagi waktu setempat) | Berbeda dengan `Time`, tipe ini tidak terkait hari layanan — murni waktu lokal absolut. |
| **`Text`** | String karakter UTF-8 yang ditujukan untuk ditampilkan dan harus dapat dibaca manusia. | `Stasiun Velodrome`, `Jakarta LRT` | Gunakan encoding `utf-8` saat ekspor file untuk dukung karakter khusus & aksara lokal. |
| **`Timezone`** | Zona waktu TZ dari [IANA Time Zone Database](https://www.iana.org/time-zones). | `Asia/Jakarta`, `America/New_York` | Nama zona waktu tidak pernah mengandung spasi, tapi boleh menggunakan underscore (`_`). Lihat daftar lengkap: [List of tz zones](https://en.wikipedia.org/wiki/List_of_tz_database_time_zones) |
| **`URL`** | URL lengkap yang menyertakan `http://` atau `https://`, dengan karakter khusus yang di-*escape* dengan benar. | `https://www.lrtjakarta.co.id/jadwal.html` | 🔗 Panduan pembuatan URL valid: [W3C URI Recommendations](http://www.w3.org/Addressing/URL/4_URI_Recommentations.html) |


Lebih langkap cek *dokumentasi resmi:* [GTFS Reference](https://gtfs.org/documentation/overview/)

---

## agency.txt (Required)

File `agency.txt` berisi informasi tentang operator transit yang menyediakan layanan dalam dataset GTFS ini. Untuk LRT Jakarta, file ini mendeskripsikan identitas PT LRT Jakarta sebagai penyedia layanan. Referensi data agency.txt ini diambil dari website resmi [LRT Jakarta](https://lrtjakarta.co.id/).

Direferensi oleh: `routes.txt` (melalui field `agency_id`)

---

**Skema kolom agency.txt**

| Field | Tipe Data | Status | Deskripsi |
|---|---|---|---|
| `agency_id` | Unique ID | **Kondisional** | ID unik pengenal merek/operator transportasi. **Wajib** jika dataset berisi lebih dari satu operator. Disarankan diisi meskipun hanya satu operator. |
| `agency_name` | Text | **Wajib** | Nama lengkap operator transportasi. |
| `agency_url` | URL | **Wajib** | Alamat website resmi operator. |
| `agency_timezone` | Timezone | **Wajib** | Zona waktu lokasi operator. Jika ada beberapa operator dalam satu dataset, semua harus menggunakan zona waktu yang sama. |
| `agency_lang` | Language Code | Opsional | Bahasa utama yang digunakan operator. Membantu aplikasi menerapkan aturan penulisan dan pengaturan bahasa yang tepat. |
| `agency_phone` | Phone Number | Opsional | Nomor telepon layanan operator. Boleh mengandung tanda baca pemisah angka.|
| `agency_fare_url` | URL | Opsional | URL halaman web tempat penumpang dapat membeli tiket atau melihat informasi tarif layanan. |
| `agency_email` | Email | Opsional | Alamat email layanan pelanggan yang aktif dipantau, sebagai kontak langsung bagi penumpang. |

### Panduan Pengisian `agency.txt` — LRT Jakarta Lin Selatan Fase 1A

---

**`agency_id`**

Diisi dengan `LRTJ` untuk layanan LRT Jakarta.

---

**`agency_name`**

Diisi dengan operator atau perusahaan yang mengoperasikan layanan LRT Jakarta yaitu `PT LRT Jakarta`

---

**`agency_url`**

Diisi dengan website resmi dari LRT Jakarta yaitu `https://lrtjakarta.co.id/`

---

**`agency_timezone`**

Diisi dengan zona waktu lokasi layanan LRT Jakarta yaitu `Asia/Jakarta`

---

**`agency_lang`**

Diisi dengan bahasa utama yang digunakan operator LRT Jakarta yaitu bahasa indonesia `id`

---

**`agency_phone`**

Diisi dengan nomor telephone call center dari operator LRT Jakarta yaitu `1500332`

---

**`agency_fare_url`**

Diisi dengan URL halaman web tempat penumpang dapat membeli tiket atau melihat informasi tarif layanan, pada LRT Jakarta bisa diakses pada website resmi LRT Jakarta pada bagian `https://www.lrtjakarta.co.id/informasi-tiket.html`

---

**`agency_email`**

Diisi dengan alamat email customer care dari operator layanan LRT Jakarta yaitu `carla@lrtjakarta.co.id`

In [1]:
import pandas as pd
import os
import csv

# Data agency LRT Jakarta
agency_data = {
    'agency_id': ['LRTJ'],
    'agency_name': ['PT LRT Jakarta'],
    'agency_url': ['https://lrtjakarta.co.id/'],
    'agency_timezone': ['Asia/Jakarta'],
    'agency_lang': ['id'],
    'agency_phone': ['02150899909'],
    'agency_fare_url': ['https://www.lrtjakarta.co.id/informasi-tiket.html'],
    'agency_email': ['carla@lrtjakarta.co.id']
}

df_agency = pd.DataFrame(agency_data)

df_agency.to_csv(
    "agency.txt",
    index=False,
    encoding='utf-8',        
    lineterminator='\n',         
    quoting=csv.QUOTE_MINIMAL   
)

# Tampilkan hasil
print("✅ File agency.txt berhasil dibuat!")
print("\n📄 Preview isi file:")
df_agency

✅ File agency.txt berhasil dibuat!

📄 Preview isi file:


,agency_id,agency_name,agency_url,agency_timezone,agency_lang,agency_phone,agency_fare_url,agency_email
0,LRTJ,PT LRT Jakarta,https://lrtjakarta.co.id/,Asia/Jakarta,id,02150899909,https://www.lrtjakarta.co.id/informasi-tiket.html,carla@lrtjakarta.co.id


## stops.txt (Conditionally Required)

File `stops.txt` berisi informasi tentang lokasi halte/stasiun tempat transportasi publik menaikkan atau menurunkan penumpang atau semua lokasi dalam jaringan transportasi mulai dari stasiun induk, peron, hingga pintu masuk/keluar. Untuk LRT Jakarta, file ini mendeskripsikan 6 stasiun Lin Selatan Fase 1A dari Stasiun Pegangsaan Dua hingga Stasiun Velodrome.

---

**Field yang Direferensikan di File Lain**

| Field | Direferensikan di | Sebagai |
|---|---|---|
| `stop_id` | `stop_times.txt` | Foreign ID — setiap baris stop_times merujuk ke stop tempat kendaraan berhenti |
| `stop_id` | `transfers.txt` | Foreign ID — mendefinisikan aturan transfer antara dua stop |
| `stop_id` | `pathways.txt` | Foreign ID — mendefinisikan jalur fisik (tangga, lift, koridor) antar stop/entrance |
| `stop_id` | `levels.txt` | Foreign ID — menghubungkan lantai/level ke stop tertentu |
| `stop_id` | `locations.geojson` | Foreign ID — area geografis untuk on-demand service |
| `zone_id` | `fare_rules.txt` | Foreign ID — menentukan tarif berdasarkan zona asal dan tujuan |
| `level_id` | `levels.txt` | Foreign ID — merujuk ke level/lantai di dalam stasiun |
| `parent_station` | `stops.txt` *(self-referencing)* | Foreign ID — stop/platform/entrance merujuk ke stasiun induknya |

---

**Skema Kolom stops.txt**

| Nama Kolom | Tipe Data | status | Deskripsi | Contoh Value |
|------------|-----------|--------|-----------|--------------|
| `stop_id` | ID (unique) | **Wajib** | ID unik untuk mengidentifikasi halte/stasiun. | `SPGD` |
| `stop_code` | Text | Opsional | Kode singkat lokasi yang ditampilkan ke penumpang misalnya di papan informasi. | `S01` |
| `stop_name` | Text | **Kondisional** | Nama lokasi sesuai yang tertera di jadwal, website, atau papan petunjuk resmi. **Wajib** untuk `location_type` 0, 1, dan 2. Opsional untuk tipe 3 dan 4. | `Pegangsaan Dua` |
| `tts_stop_name` | Text | Opsional | Versi `stop_name` yang mudah dibaca oleh sistem text-to-speech (phonemisasi). | |
| `stop_desc` | Text | Opsional | Deskripsi lokasi yang informatif. Tidak boleh duplikasi dari `stop_name`. | `Stasiun terminus dan terintegrasi dengan halte bus` |
| `stop_lat` | Latitude | **Kondisional** | Koordinat lintang lokasi (WGS84). Untuk platform/halte, koordinat harus merujuk ke tiang halte atau tempat penumpang naik — bukan ke jalur kendaraan. **Wajib** untuk tipe 0, 1, dan 2. | `-6.2832` |
| `stop_lon` | Longitude | **Kondisional** | Koordinat bujur lokasi (WGS84). Aturan sama dengan `stop_lat`. **Wajib** untuk tipe 0, 1, dan 2. | `106.7394` |
| `zone_id` | ID | Opsional | ID zona tarif untuk halte. Diabaikan jika lokasi adalah stasiun atau pintu masuk stasiun. | `ZONE-JAKTIM` |
| `stop_url` | URL | Opsional | URL informasi spesifik halte/stasiun. | `https://jakartamrt.co.id/layanan-pelanggan/stasiun/stasiun-lebak-bulus` |
| `location_type` | Enum | Opsional | Tipe lokasi: `0`=halte/platform, `1`=stasiun, `2`=pintu masuk, `3`=node, `4`=area boarding. Lebih detail lihat tabel jenis lokasi di bawah. | `1` |
| `parent_station` | Foreign ID → stops.stop_id | **Kondisional** | ID stasiun induk (untuk hierarki dalam stasiun) **Wajib** untuk tipe 2, 3, dan 4. Opsional untuk tipe 0. **Dilarang** diisi untuk tipe 1. | `SPGD` |
| `stop_timezone` | Timezone | Opsional | Zona waktu halte/stasiun (jika berbeda dari agency). | `Asia/Jakarta` |
| `wheelchair_boarding` | Enum | Opsional | Aksesibilitas kursi roda: `0`=tidak diketahui, `1`=tersedia, `2`=tidak tersedia. | `1` |
| `level_id` | Foreign ID → levels.level_id | Opsional | Referensi ke `levels.txt` untuk lantai dalam stasiun. | `LEVEL-G` |
| `platform_code` | Text | Opsional | Kode peron — hanya nomor/hurufnya saja, misalnya `1` atau `G`. Jangan sertakan kata "peron" atau "jalur". | `1` |
| `stop_access` | Enum | **Kondisional** | Cara akses ke platform dari luar. Lihat tabel nilai enum di bawah. **Dilarang** untuk tipe 1, 2, 3, dan 4, serta jika `parent_station` kosong. |

---

**Jenis Lokasi (`location_type`)**

| Nilai | Nama | Deskripsi |
|:---:|---|---|
| `0` atau kosong | Stop / Platform | Tempat penumpang naik/turun kendaraan. Disebut platform jika berada di dalam stasiun induk. |
| `1` | Station | Bangunan atau area fisik yang menampung satu atau lebih platform. |
| `2` | Entrance / Exit | Pintu masuk atau keluar stasiun dari jalan. Jika satu pintu terhubung ke beberapa stasiun, harus dipilih satu sebagai induk. |
| `3` | Generic Node | Lokasi dalam stasiun yang tidak termasuk tipe lain digunakan untuk menghubungkan jalur pejalan kaki (`pathways.txt`). |
| `4` | Boarding Area | Lokasi spesifik di dalam platform tempat penumpang naik/turun kendaraan. |

---

**Aksesibilitas Kursi Roda (`wheelchair_boarding`)**

| Konteks | Nilai | Deskripsi |
|---|:---:|---|
| **Stop tanpa induk** | `0` atau kosong | Tidak ada informasi aksesibilitas |
| | `1` | Beberapa kendaraan bisa diakses kursi roda |
| | `2` | Tidak bisa diakses kursi roda |
| **Stop anak (punya induk)** | `0` atau kosong | Mewarisi nilai dari stasiun induk (parent station) |
| | `1` | Ada jalur aksesibel dari luar stasiun ke platform ini |
| | `2` | Tidak ada jalur aksesibel ke platform ini |
| **Entrance / Exit** | `0` atau kosong | Mewarisi nilai dari stasiun induk (parent station) |
| | `1` | Pintu masuk aksesibel kursi roda |
| | `2` | Tidak ada jalur aksesibel dari pintu masuk ke platform |

---
**Akses Stop (`stop_access`)**

| Nilai | Deskripsi |
|:---:|---|
| `0` | Platform tidak bisa diakses langsung dari jalan — harus melalui pintu masuk stasiun atau jalur yang terdefinisi di `pathways.txt` |
| `1` | Aplikasi boleh memberi petunjuk langsung ke platform, tanpa melalui pintu masuk atau pathway |
| `kosong` | Akses tidak terdefinisi |

---


### Panduan Pengisian `stops.txt` — LRT Jakarta Lin Selatan Fase 1A

---

**`Daftar Stasiun`**

| No | Nama Stasiun | Short Name | Kode Stasiun |
|:---:|---|:---:|:---:|
| 1 | Pegangsaan Dua | PGD | S01 |
| 2 | Boulevard Utara Summarecon Mall | BVU | S02 |
| 3 | Boulevard Selatan | BVS | S03 |
| 4 | Pulomas | PUM | S04 |
| 5 | Equestrian | EQS | S05 |
| 6 | Velodrome | VEL | S06 |

---

**`stop_id`**

Format penamaan terdiri dari 3 jenis sesuai `location_type`:

| Jenis | Format | Contoh | Keterangan |
|---|---|---|---|
| Parent station | `S(kode)` | `SPGD`, `SBVU`, `SBVS` | Satu per stasiun |
| Platform | `G(kode)(urut 2 digit)` | `GPGD01`, `GPGD02` | Urut mulai dari `01` per platform |
| Entrance | `E(kode)(urut 2 digit)` | `EPGD01`, `EPGD02` | Urut mulai dari `01`, sesuai label Akses A, B, C, … |

---

**`stop_name`**

Diisi dengan nama lengkap stasiun sesuai daftar resmi di atas, dengan prefiks **"Stasiun LRT"**. Untuk entrance ditambahkan sufiks label akses.

| Jenis | Format | Contoh |
|---|---|---|
| Parent station | `Stasiun LRT (nama stasiun)` | `Stasiun LRT Boulevard Utara Summarecon Mall` |
| Platform | `Stasiun LRT (nama stasiun)` | `Stasiun LRT Boulevard Utara Summarecon Mall` |
| Entrance | `Stasiun LRT (nama stasiun) Akses (label)` | `Stasiun LRT Boulevard Utara Summarecon Mall Akses A` |

---

**`location_type`**

`location_type` disesuaikan dengan kaidah dari dokumentasi GTFS, , untuk parent_stasion=1, untuk platform=0, dan untuk entrance=2.

| Jenis | Nilai | Contoh `stop_id` |
|---|:---:|---|
| Parent station | `1` | `SPGD`, `SBVU`, `SBVS`, … |
| Platform | `0` | `GPGD1`, `GBVU1`, … |
| Entrance | `2` | `EPGD01`, `EBVU01`, … |

---

**`parent_station`**

Diisi dengan nilai `stop_id` dari stasiun induk dari setiap `location_type` dengan nilai 0 (platform) dan 2 (entrance) yaitu (SPGD, SBVU, SBVS, SPUM, SEQS, SVEL)

| Jenis | Isi | Contoh |
|---|---|---|
| Parent station | **Kosong** — dilarang diisi per spesifikasi GTFS | — |
| Platform | `stop_id` dari parent station stasiun bersangkutan | `GPGD01` → `SPGD` |
| Entrance | `stop_id` dari parent station stasiun bersangkutan | `EPGD01` → `SPGD` |

---

**`platform_code`**

Diisi hanya untuk baris **platform** (`location_type=0`). Cukup nomor urutnya saja tanpa kata "peron" atau "jalur".

| Contoh `stop_id` | Nilai `platform_code` |
|---|:---:|
| `GPGD01` | `1` |
| `GPGD02` | `2` |
| `GPGD03` | `3` |
| Parent station & entrance | *(kosong)* |

---

**`stop_lat & stop_lon`**

Koordinat diambil dari node OpenStreetMap menggunakan dua API berikut secara berurutan:

| Prioritas | API | Keterangan |
|:---:|---|---|
| 1 | **OSM API** — `api.openstreetmap.org` | Sumber utama |
| 2 | **Overpass API** — `overpass-api.de` | Fallback jika OSM API gagal |

Node OSM yang digunakan mencakup parent station, platform dan entrance, untuk lokasi stop yang belum memiliki node OSM, koordinat diisi secara manual. Berikut ini daftar node stops dari LRT Jakarta yang tersedia di OpenStreetMap:

| Nama Stop | Tipe | Platform | Node OSM |
|---|---|:---:|---|
| Stasiun LRT Pegangsaan Dua | Parent station | — | https://www.openstreetmap.org/node/5984702931 |
| Stasiun LRT Pegangsaan Dua | Platform | 1 | https://www.openstreetmap.org/node/11630425911 |
| Stasiun LRT Pegangsaan Dua | Platform | 2 | https://www.openstreetmap.org/node/11630425912 |
| Stasiun LRT Pegangsaan Dua Akses A | Entrance | — | https://www.openstreetmap.org/node/13751649255 |
| Stasiun LRT Pegangsaan Dua Akses B | Entrance | — | https://www.openstreetmap.org/node/13751649256 |
| Stasiun LRT Boulevard Utara Summarecon Mall | Parent station | — | https://www.openstreetmap.org/node/5984702926 |
| Stasiun LRT Boulevard Utara Summarecon Mall | Platform | 1 | https://www.openstreetmap.org/node/6196137582 |
| Stasiun LRT Boulevard Utara Summarecon Mall | Platform | 2 | https://www.openstreetmap.org/node/6196137583 |
| Stasiun LRT Boulevard Utara Summarecon Mall Akses A | Entrance | — | https://www.openstreetmap.org/node/13751619261 |
| Stasiun LRT Boulevard Utara Summarecon Mall Akses B | Entrance | — | https://www.openstreetmap.org/node/13751633899 |
| Stasiun LRT Boulevard Utara Summarecon Mall Akses C | Entrance | — | https://www.openstreetmap.org/node/13751461695 |
| Stasiun LRT Boulevard Utara Summarecon Mall Elevator Akses B | Entrance | — | https://www.openstreetmap.org/node/13760418627 |
| Stasiun LRT Boulevard Utara Summarecon Mall Elevator Akses C | Entrance | — | https://www.openstreetmap.org/node/13751625978 |
| Stasiun LRT Bouelevard Selatan | Parent station | — | https://www.openstreetmap.org/node/5984702927 |
| Stasiun LRT Bouelevard Selatan | Platform | 1 | https://www.openstreetmap.org/node/6196137584 |
| Stasiun LRT Bouelevard Selatan | Platform | 2 | https://www.openstreetmap.org/node/6196138085 |
| Stasiun LRT Bouelevard Selatan Akses A | Entrance | — | https://www.openstreetmap.org/node/13751664244 |
| Stasiun LRT Bouelevard Selatan Akses B | Entrance | — | https://www.openstreetmap.org/node/13751675920 |
| Stasiun LRT Bouelevard Selatan Akses C | Entrance | — | https://www.openstreetmap.org/node/13751660024 |
| Stasiun LRT Bouelevard Selatan Akses D | Entrance | — | https://www.openstreetmap.org/node/13751676817 |
| Stasiun LRT Bouelevard Selatan Elevator Akses A | Entrance | — | https://www.openstreetmap.org/node/13751677149 |
| Stasiun LRT Bouelevard Selatan Elevator Akses D | Entrance | — | https://www.openstreetmap.org/node/13751682520 |
| Stasiun LRT Pulomas | Parent station | — | https://www.openstreetmap.org/node/5984702928 |
| Stasiun LRT Pulomas | Platform | 1 | https://www.openstreetmap.org/node/6196137581 |
| Stasiun LRT Pulomas | Platform | 2 | https://www.openstreetmap.org/node/6196137580 |
| Stasiun LRT Pulomas Akses A | Entrance | — | https://www.openstreetmap.org/node/13751682584 |
| Stasiun LRT Pulomas Akses C | Entrance | — | https://www.openstreetmap.org/node/13751626000 |
| Stasiun LRT Pulomas Akses D | Entrance | — | https://www.openstreetmap.org/node/13751690455 |
| Stasiun LRT Pulomas Elevator Akses A | Entrance | — | https://www.openstreetmap.org/node/13762425663 |
| Stasiun LRT Pulomas Elevator Akses C-D | Entrance | — | https://www.openstreetmap.org/node/13762425664 |
| Stasiun LRT Equestrian | Parent station | — | https://www.openstreetmap.org/node/5984702929 |
| Stasiun LRT Equestrian | Platform | 1 | https://www.openstreetmap.org/node/6196137578 |
| Stasiun LRT Equestrian | Platform | 2 | https://www.openstreetmap.org/node/6196137579 |
| Stasiun LRT Equestrian Akses A | Entrance | — | https://www.openstreetmap.org/node/13751699386 |
| Stasiun LRT Equestrian Akses B | Entrance | — | https://www.openstreetmap.org/node/13751704290 |
| Stasiun LRT Equestrian Akses C | Entrance | — | https://www.openstreetmap.org/node/13751724801 |
| Stasiun LRT Equestrian Akses D | Entrance | — | https://www.openstreetmap.org/node/13751699397 |
| Stasiun LRT Equestrian Elevator Akses A-B | Entrance | — | https://www.openstreetmap.org/node/13762468136 |
| Stasiun LRT Equestrian Elevator Akses C-D | Entrance | — | https://www.openstreetmap.org/node/13762460658 |
| Stasiun LRT Velodrome | Parent station | — | https://www.openstreetmap.org/node/5984702930 |
| Stasiun LRT Velodrome | Platform | 1 | https://ww.openstreetmap.org/node/6196137576 |
| Stasiun LRT Velodrome | Platform | 2 | https://www.openstreetmap.org/node/6196137577 |
| Stasiun LRT Velodrome Akses A | Entrance | — | https://www.openstreetmap.org/node/13751707969 |
| Stasiun LRT Velodrome Akses B | Entrance | — | https://www.openstreetmap.org/node/13752179018 |
| Stasiun LRT Velodrome Akses C | Entrance | — | https://www.openstreetmap.org/node/13752169406 |
| Stasiun LRT Velodrome Akses D | Entrance | — | https://www.openstreetmap.org/node/13752145698 |
| Stasiun LRT Velodrome Akses E Skybridge Halte Pemuda | Entrance | — | https://www.openstreetmap.org/node/6776108884 |
| Stasiun LRT Velodrome Elevator Akses C-D | Entrance | — | https://www.openstreetmap.org/node/13752169172 |

---

**`wheelchair_boarding`**

Diisi berdasarkan informasi aksesibilitas dari website resmi LRT Jakarta. Nilai yang berlaku:

| Nilai | Arti | Berlaku untuk |
|:---:|---|---|
| `1` | Aksesibel kursi roda | Parent station yang memiliki fasilitas aksesibel |
| `0` | Mewarisi nilai dari parent station | Platform & entrance (jika mengikuti induknya) |
| `2` | Tidak aksesibel kursi roda | Jika ada platform/entrance yang tidak aksesibel |

In [2]:
# Import Library
import requests
import csv
import time

# Konfigurasi API OpenStreetMap
OSM_API      = "https://api.openstreetmap.org/api/0.6/node/{node_id}.json"
OVERPASS_API = "https://overpass-api.de/api/interpreter"
HEADERS      = {"User-Agent": "gtfs-stops-generator/1.0"}

_cache = {}  # simpan hasil fetch agar tidak request ulang

In [3]:
# Function request titik koordinat (stop_lat & stop_lon) OSM API
def get_osm_coords(label, node_id):
    """Fetch (lat, lon) dari OSM node ID. Fallback ke Overpass jika OSM gagal."""
    if node_id is None:
        return None, None
    if node_id in _cache:
        return _cache[node_id]

    # Request OSM API
    try:
        url  = OSM_API.format(node_id=node_id)
        resp = requests.get(url, timeout=10, headers=HEADERS)
        resp.raise_for_status()
        elem = resp.json()["elements"][0]
        lat, lon = elem["lat"], elem["lon"]
        _cache[node_id] = (lat, lon)
        print(f"  ✓ [OSM] {label} (node_id={node_id}) → ({lat}, {lon})")
        time.sleep(0.2)
        return lat, lon
    except Exception as e:
        print(f"  … OSM gagal ({e}), coba Overpass …")

    # Request Overpass API
    try:
        query = f"[out:json];node({node_id});out;"
        resp  = requests.post(OVERPASS_API, data={"data": query},
                              timeout=15, headers=HEADERS)
        resp.raise_for_status()
        elem  = resp.json()["elements"][0]
        lat, lon = elem["lat"], elem["lon"]
        _cache[node_id] = (lat, lon)
        print(f"  ✓ [Overpass] {label} {node_id} → ({lat}, {lon})")
        time.sleep(0.5)
        return lat, lon
    except Exception as e:
        print(f"  ✗ Overpass juga gagal: {e}")

    _cache[node_id] = (None, None)
    return None, None

In [4]:
# Pengisian Data disesuaikan dengan field panduan diatas 

STATION_DATA = [
    {
        "stop_id_prefix": "PGD", #Kode Stasiun
        "stop_name"     : "Stasiun LRT Pegangsaan Dua", # stop_name
        "stop_url"      : "", #URL Stasiun
        "stop_code"     : "S01", # stop_code biasanya tertampil di peta informasi
        "parent_wc"     : 1, # wheelchair_boarding parent station
        "parent_node_id": 5984702931, # node-id parent stasiun
        "platforms": [  
            #(platform_code dan node-id platform stasiun)
            ("1", 11630425911), # platform_code dan node-id platform stasiun
            ("2", 11630425912),
        ],
        # (label, osm_node_id) — isi None jika belum ada node OSM
        "entrances": [
            #(nama entrance, node-id entrance stasiun, wheelchair_boarding entrance)
            ("Akses A", 13751649255, 1),
            ("Akses B", 13751649256, 1)
        ],
    },
    {
        "stop_id_prefix": "BVU",
        "stop_name"     : "Stasiun LRT Boulevard Utara Summarecon Mall",
        "stop_url"      : "",
        "stop_code"     : "S02",
        "parent_wc"     : 1,
        "parent_node_id": 5984702926,
        "platforms": [
            ("1", 6196137582),
            ("2", 6196137583),
        ],
        "entrances": [
            ("Akses A", 13751619261, 2), 
            ("Akses B", 13751633899, 2), 
            ("Akses C", 13751461695, 2), 
            ("Elevator Akses B", 13760418627, 1),
            ("Elevator Akses C", 13751625978, 1)
        ],
    },
    {
        "stop_id_prefix": "BVS",
        "stop_name"     : "Stasiun LRT Bouelevard Selatan",
        "stop_url"      : "",
        "stop_code"     : "S03",
        "parent_wc"     : 1,
        "parent_node_id": 5984702927,
        "platforms": [
            ("1", 6196137584),
            ("2", 6196138085),
        ],
        "entrances": [
            ("Akses A", 13751664244, 2), 
            ("Akses B", 13751675920, 2), 
            ("Akses C", 13751660024, 2),
            ("Akses D", 13751676817, 2), 
            ("Elevator Akses A", 13751677149, 1),
            ("Elevator Akses D", 13751682520, 1)
        ],
    },
    {
        "stop_id_prefix": "PUM",
        "stop_name"     : "Stasiun LRT Pulomas",
        "stop_url"      : "",
        "stop_code"     : "S04",
        "parent_wc"     : 1,
        "parent_node_id": 5984702928,
        "platforms": [
            ("1", 6196137581),
            ("2", 6196137580),
        ],
        "entrances": [
            ("Akses A", 13751682584, 2), 
            ("Akses C", 13751626000, 2), 
            ("Akses D", 13751690455, 2), 
            ("Elevator Akses A", 13762425663, 1),
            ("Elevator Akses C-D", 13762425664, 1)
        ],
    },
    {
        "stop_id_prefix": "EQS",
        "stop_name"     : "Stasiun LRT Equestrian",
        "stop_url"      : "",
        "stop_code"     : "S05",
        "parent_wc"     : 1,
        "parent_node_id": 5984702929,
        "platforms": [
            ("1", 6196137578),
            ("2", 6196137579),
        ],
        "entrances": [
            ("Akses A", 13751699386, 2), 
            ("Akses B", 13751704290, 2), 
            ("Akses C", 13751724801, 2), 
            ("Akses D", 13751699397, 2),
            ("Elevator Akses A-B", 13762468136, 1),
            ("Elevator Akses C-D", 13762460658, 1)
        ],
    },
    {
        "stop_id_prefix": "VEL",
        "stop_name"     : "Stasiun LRT Velodrome",
        "stop_url"      : "",
        "stop_code"     : "S06",
        "parent_wc"     : 1,
        "parent_node_id": 5984702930,
        "platforms": [
            ("1", 6196137576),
            ("2", 6196137577),
        ],
        "entrances": [
            ("Akses A", 13751707969, 2),
            ("Akses B", 13752179018, 2), 
            ("Akses C", 13752169406, 2),
            ("Akses D", 13752145698, 2), 
            ("Akses E Skybridge Halte Pemuda", 6776108884, 2),
            ("Elevator Akses C-D", 13752169172, 1),
        ],
    },
]

print(f"✅ {len(STATION_DATA)} stasiun terdefinisi.")

✅ 6 stasiun terdefinisi.


In [5]:
FIELDNAMES = [
    "stop_id", "stop_code", "stop_name", "stop_url", "stop_lat", "stop_lon",
    "zone_id","location_type", "parent_station", "platform_code",
    "wheelchair_boarding"
]

def build_rows():
    rows = []
    for st in STATION_DATA:
        pfx       = st["stop_id_prefix"]
        name      = st["stop_name"]
        parent_id = f"S{pfx}"

        print(f"\n{'─'*50}")
        print(f"📍 {name}")

        # Parent station (location_type = 1)
        print(f"   Parent node: {st['parent_node_id']}")
        lat, lon = get_osm_coords(name, st["parent_node_id"])
        rows.append({
            "stop_id"            : parent_id,
            "stop_code"          : st["stop_code"],
            "stop_name"          : name,
            "stop_url"           : st["stop_url"],
            "stop_lat"           : lat if lat is not None else "",
            "stop_lon"           : lon if lon is not None else "",
            "zone_id"            : pfx,
            "location_type"      : 1,
            "parent_station"     : "",
            "platform_code"      : "",
            "wheelchair_boarding": st["parent_wc"]
        })

        # Platform (location_type = 0)
        for idx, (pcode, node_id) in enumerate(st["platforms"], start=1):
            num = str(idx).zfill(2)
            lat, lon = get_osm_coords(f"{name} Platform {pcode}", node_id)
            rows.append({
                "stop_id"            : f"G{pfx}{num}",
                "stop_code"          : "",
                "stop_name"          : name,
                "stop_url"           : "",
                "stop_lat"           : lat if lat is not None else "",
                "stop_lon"           : lon if lon is not None else "",
                "zone_id"            : pfx,
                "location_type"      : 0,
                "parent_station"     : parent_id,
                "platform_code"      : pcode,
                "wheelchair_boarding": 0
            })

        # Entrance (location_type = 2)
        for idx, (label, node_id, wheelchair) in enumerate(st["entrances"], start=1):
            num = str(idx).zfill(2)
            lat, lon = get_osm_coords(f"{name} {label}", node_id)
            rows.append({
                "stop_id"            : f"E{pfx}{num}",
                "stop_code"          : "",
                "stop_name"          : f"{name} {label}",
                "stop_url"           : "",
                "stop_lat"           : lat if lat is not None else "",
                "stop_lon"           : lon if lon is not None else "",
                "zone_id"            : pfx,
                "location_type"      : 2,
                "parent_station"     : parent_id,
                "platform_code"      : "",
                "wheelchair_boarding": wheelchair
            })

    return rows

print("✅ Fungsi build_rows() siap.")

✅ Fungsi build_rows() siap.


In [6]:
_cache.clear()  # reset cache jika cell dijalankan ulang
all_rows = build_rows()

print(f"\n{'='*50}")
print(f"✅ Total baris dihasilkan: {len(all_rows)}")


──────────────────────────────────────────────────
📍 Stasiun LRT Pegangsaan Dua
   Parent node: 5984702931
  ✓ [OSM] Stasiun LRT Pegangsaan Dua (node_id=5984702931) → (-6.1572135, 106.9142091)
  ✓ [OSM] Stasiun LRT Pegangsaan Dua Platform 1 (node_id=11630425911) → (-6.15721, 106.9142841)
  ✓ [OSM] Stasiun LRT Pegangsaan Dua Platform 2 (node_id=11630425912) → (-6.1572054, 106.9141377)
  ✓ [OSM] Stasiun LRT Pegangsaan Dua Akses A (node_id=13751649255) → (-6.1574445, 106.9137291)
  ✓ [OSM] Stasiun LRT Pegangsaan Dua Akses B (node_id=13751649256) → (-6.1581905, 106.9137875)

──────────────────────────────────────────────────
📍 Stasiun LRT Boulevard Utara Summarecon Mall
   Parent node: 5984702926
  ✓ [OSM] Stasiun LRT Boulevard Utara Summarecon Mall (node_id=5984702926) → (-6.1594366, 106.9059815)
  ✓ [OSM] Stasiun LRT Boulevard Utara Summarecon Mall Platform 1 (node_id=6196137582) → (-6.1596205, 106.9058073)
  ✓ [OSM] Stasiun LRT Boulevard Utara Summarecon Mall Platform 2 (node_id=619613

In [7]:
import pandas as pd

df_stops = pd.DataFrame(all_rows, columns=FIELDNAMES)
df_stops

,stop_id,stop_code,stop_name,stop_url,stop_lat,stop_lon,zone_id,location_type,parent_station,platform_code,wheelchair_boarding
0,SPGD,S01,Stasiun LRT Pegangsaan Dua,,-6.157214,106.914209,PGD,1,,,1
1,GPGD01,,Stasiun LRT Pegangsaan Dua,,-6.157210,106.914284,PGD,0,SPGD,1,0
2,GPGD02,,Stasiun LRT Pegangsaan Dua,,-6.157205,106.914138,PGD,0,SPGD,2,0
3,EPGD01,,Stasiun LRT Pegangsaan Dua Akses A,,-6.157445,106.913729,PGD,2,SPGD,,1
4,EPGD02,,Stasiun LRT Pegangsaan Dua Akses B,,-6.158190,106.913787,PGD,2,SPGD,,1
5,SBVU,S02,Stasiun LRT Boulevard Utara Summarecon Mall,,-6.159437,106.905981,BVU,1,,,1
6,GBVU01,,Stasiun LRT Boulevard Utara Summarecon Mall,,-6.159620,106.905807,BVU,0,SBVU,1,0
7,GBVU02,,Stasiun LRT Boulevard Utara Summarecon Mall,,-6.159661,106.905876,BVU,0,SBVU,2,0
8,EBVU01,,Stasiun LRT Boulevard Utara Summarecon Mall Ak...,,-6.158968,106.906456,BVU,2,SBVU,,2
9,EBVU02,,Stasiun LRT Boulevard Utara Summarecon Mall Ak...,,-6.159997,106.905814,BVU,2,SBVU,,2


In [8]:
# Validasi titik koordinat setiap stops dengan build graph folium
import pandas as pd
import folium

# pastikan lat/lon numeric
df_stops["stop_lat"] = pd.to_numeric(df_stops["stop_lat"], errors="coerce")
df_stops["stop_lon"] = pd.to_numeric(df_stops["stop_lon"], errors="coerce")

# drop yang invalid
df_stops = df_stops.dropna(subset=["stop_lat", "stop_lon"])

# center map (pakai rata-rata koordinat)
center_lat = df_stops["stop_lat"].mean()
center_lon = df_stops["stop_lon"].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=12)

# plot semua stop
for _, row in df_stops.iterrows():
    folium.CircleMarker(
        location=[row["stop_lat"], row["stop_lon"]],
        radius=4,
        popup=f"""
        ID: {row['stop_id']}<br>
        Name: {row['stop_name']}<br>
        Type: {row.get('location_type', '')}<br>
        Parent: {row.get('parent_station', '')}
        """,
        fill=True,
    ).add_to(m)

# simpan ke html
# m.save("stops_map.html")
m

In [9]:
df_stops.to_csv(
    "stops.txt",
    index=False,
    encoding='utf-8',        
    lineterminator='\n',         
    quoting=csv.QUOTE_MINIMAL   
)

In [10]:
df_stops = pd.read_csv(
    "stops.txt",
    dtype={
        "stop_id"             : str,
        "stop_code"           : str,
        "stop_name"           : str,
        "stop_url"            : str,
        "stop_lat"            : float,
        "stop_lon"            : float,
        "zone_id"             : str,
        "location_type"       : "Int64",
        "parent_station"      : str,
        "platform_code"       : str,
        "wheelchair_boarding" : "Int64"
    },
    keep_default_na=False
)
print(f"Shape: {df_stops.shape[0]} baris × {df_stops.shape[1]} kolom")
df_stops.head(50)

Shape: 48 baris × 11 kolom


,stop_id,stop_code,stop_name,stop_url,stop_lat,stop_lon,zone_id,location_type,parent_station,platform_code,wheelchair_boarding
0,SPGD,S01,Stasiun LRT Pegangsaan Dua,,-6.157214,106.914209,PGD,1,,,1
1,GPGD01,,Stasiun LRT Pegangsaan Dua,,-6.157210,106.914284,PGD,0,SPGD,1,0
2,GPGD02,,Stasiun LRT Pegangsaan Dua,,-6.157205,106.914138,PGD,0,SPGD,2,0
3,EPGD01,,Stasiun LRT Pegangsaan Dua Akses A,,-6.157445,106.913729,PGD,2,SPGD,,1
4,EPGD02,,Stasiun LRT Pegangsaan Dua Akses B,,-6.158190,106.913787,PGD,2,SPGD,,1
5,SBVU,S02,Stasiun LRT Boulevard Utara Summarecon Mall,,-6.159437,106.905981,BVU,1,,,1
6,GBVU01,,Stasiun LRT Boulevard Utara Summarecon Mall,,-6.159620,106.905807,BVU,0,SBVU,1,0
7,GBVU02,,Stasiun LRT Boulevard Utara Summarecon Mall,,-6.159661,106.905876,BVU,0,SBVU,2,0
8,EBVU01,,Stasiun LRT Boulevard Utara Summarecon Mall Ak...,,-6.158968,106.906456,BVU,2,SBVU,,2
9,EBVU02,,Stasiun LRT Boulevard Utara Summarecon Mall Ak...,,-6.159997,106.905814,BVU,2,SBVU,,2


## routes.txt (Required)

File `routes.txt` berisi informasi tentang identitas jalur atau rute layanan transportasi tersebut seperti nama rute, operator, jenis moda yang digunakan, warna identitas rute. File ini menjadi penghubung utama antara identitas layanan dengan jadwal perjalanan aktual.

**Field yang Direferensikan di File Lain**

| Field | Direferensikan di | Sebagai |
|---|---|---|
| `route_id` | `trips.txt` | Foreign ID — setiap trip merujuk ke rute induknya |
| `route_id` | `fare_rules.txt` | Foreign ID — tarif dikaitkan ke rute tertentu |
| `route_id` | `attributions.txt` | Foreign ID — atribusi data per rute |
| `agency_id` | `agency.txt` | Foreign ID — merujuk ke operator |

---

**Skema kolom routes.txt**

| Field | Tipe Data | Status | Deskripsi |
|---|---|---|---|
| `route_id` | Unique ID | **Wajib** | ID unik pengenal rute. Direferensikan di `trips.txt` dan `fare_rules.txt`. |
| `agency_id` | Foreign ID → `agency.agency_id` | **Kondisional** | ID operator rute. **Wajib** jika dataset berisi lebih dari satu operator. |
| `route_short_name` | Text | **Kondisional** | Nama singkat rute yang dikenal penumpang, maksimal 12 karakter. Contoh: `NS`, `32`, `Green`. **Wajib** jika `route_long_name` kosong. |
| `route_long_name` | Text | **Kondisional** | Nama lengkap rute, biasanya mencantumkan tujuan atau deskripsi jalur. **Wajib** jika `route_short_name` kosong. |
| `route_desc` | Text | Opsional | Deskripsi informatif rute. Tidak boleh duplikasi dari `route_short_name` atau `route_long_name`. |
| `route_type` | Enum | **Wajib** | Jenis moda transportasi. Lihat tabel jenis rute di bawah. |
| `route_url` | URL | Opsional | URL halaman web khusus rute ini. Harus berbeda dari `agency_url`. |
| `route_color` | Color | Opsional | Warna latar rute dalam format hex 6 digit tanpa `#`. Default putih (`FFFFFF`). |
| `route_text_color` | Color | Opsional | Warna teks di atas `route_color` dalam format hex 6 digit tanpa `#`. Default hitam (`000000`). Harus kontras dengan `route_color`. |
| `route_sort_order` | Non-negative integer | Opsional | Urutan tampilan rute — nilai lebih kecil ditampilkan lebih dulu. |
| `continuous_pickup` | Enum | **Kondisional Terlarang** | Apakah penumpang bisa naik di sembarang titik sepanjang rute. Lihat tabel nilai enum di bawah. **Dilarang** jika `stop_times.start_pickup_drop_off_window` atau `end_pickup_drop_off_window` terdefinisi. |
| `continuous_drop_off` | Enum | **Kondisional Terlarang** | Apakah penumpang bisa turun di sembarang titik sepanjang rute. Aturan sama dengan `continuous_pickup`. |
| `network_id` | ID | **Kondisional Terlarang** | ID grup rute — beberapa rute bisa punya `network_id` yang sama. **Dilarang** jika file `route_networks.txt` atau `networks.txt` ada. |
| `cemv_support` | Enum | Opsional | Dukungan pembayaran contactless EMV (kartu/perangkat Visa, Mastercard, dst) untuk rute ini. Jika diisi bersamaan dengan `agency.cemv_support`, nilai di sini yang **lebih diprioritaskan**. |

---

**Jenis Rute (`route_type`)**

| Nilai | Jenis | 
|:---:|---|
| `0` | Tram / Light Rail | 
| `1` | Subway / Metro |
| `2` | Kereta jarak jauh |
| `3` | Bus | 
| `4` | Ferry |
| `5` | Cable tram |
| `6` | Aerial lift / Gondola | 
| `7` | Funicular |
| `11` | Trolleybus |
| `12` | Monorail | 

---

**Nilai `continuous_pickup` & `continuous_drop_off`**

| Nilai | Arti |
|:---:|---|
| `0` | Bisa naik/turun di sembarang titik sepanjang rute |
| `1` atau kosong | Tidak bisa — hanya di halte/stasiun yang ditentukan |
| `2` | Perlu telepon operator terlebih dahulu |
| `3` | Perlu koordinasi langsung dengan pengemudi |

### Panduan Pengisian `routes.txt` — LRT Jakarta Lin Utara-Selatan Fase 1

---

**`route_id`**

Diisi sesuai dengan singkatan dari LRT Jakarta Lin Selatan (South Line) yaitu NS Line: `LRTJ_S`

---

**`agency_id`**

Disesuaikan dengan `agency_id` yang direferensikan dari file agency.txt yaitu `LRTJ`

---

**`route_short_name`**

Diisi dengan singkatan dari South Line yaitu `S`

---

**`route_long_name`**

Diisi dengan `Lin Selatan`

---

**`route_desc`**

Diisi dengan deskripsi singkat yaitu: `LRT Jakarta Lin Selatan Fase 1A (Pegangsaan Dua-Velodrome)`

---

**`route_type`**

Diisi dengan nilai `0` karena jenis moda dari LRT Jakarta adalah Light Rail

---

**`route_color`**

Diisi dengan nilai kode warna rute untuk LRT Jakarta Lin Selatan sesuai dengan peta resmi: [TransportForJakarta](https://transportforjakarta.or.id/lrtjakarta/) dan Logo LRT Jakarta Lin Selatan [Logo](https://upload.wikimedia.org/wikipedia/commons/thumb/d/d9/Jakarta_-_LRT_Jakarta_S_Line_Icon.svg/60px-Jakarta_-_LRT_Jakarta_S_Line_Icon.svg.png) yaitu kode warna Oranye: `F1611E` 

**`route_text_color`**

Diisi dengan nilai warna teks di atas route_color, disesuaikan dengan Logo LRT Jakarta Lin Selatan [Logo](https://upload.wikimedia.org/wikipedia/commons/thumb/d/d9/Jakarta_-_LRT_Jakarta_S_Line_Icon.svg/60px-Jakarta_-_LRT_Jakarta_S_Line_Icon.svg.png) warna nya putih: `FFFFFF`

In [11]:
import pandas as pd
import csv

# Data routes LRT Jakarta Lin Selatan Fase 1A
routes_data = {
    'route_id'         : ['LRTJ_S'],
    'agency_id'        : ['LRTJ'],
    'route_short_name' : ['S'],
    'route_long_name'  : ['Lin Selatan'],
    'route_desc'       : ['LRT Jakarta Lin Selatan Fase 1A (Pegangsaan Dua-Velodrome)'],
    'route_type'       : [0],
    'route_url'        : ['https://id.wikipedia.org/wiki/LRT_Jakarta#Lin_Selatan'],
    'route_color'      : ['F1611E'],
    'route_text_color' : ['FFFFFF'],
    'route_sort_order' : [1]
}

df_routes = pd.DataFrame(routes_data)
df_routes.to_csv(
    "routes.txt",
    index=False,
    encoding='utf-8',
    lineterminator='\n',
    quoting=csv.QUOTE_MINIMAL
)

print("✅ File routes.txt berhasil dibuat!")
print("\n📄 Preview isi file:")
df_routes

✅ File routes.txt berhasil dibuat!

📄 Preview isi file:


,route_id,agency_id,route_short_name,route_long_name,route_desc,route_type,route_url,route_color,route_text_color,route_sort_order
0,LRTJ_S,LRTJ,S,Lin Selatan,LRT Jakarta Lin Selatan Fase 1A (Pegangsaan Du...,0,https://id.wikipedia.org/wiki/LRT_Jakarta#Lin_...,F1611E,FFFFFF,1


## trips.txt (Required)

File `trips.txt` mendefinisikan setiap **perjalanan individual** yang dilakukan dalam satu rute. Jika `routes.txt` mendeskripsikan *"jalur apa"*, maka `trips.txt` mendeskripsikan *"perjalanan spesifik kapan dan ke arah mana"*. Satu rute bisa memiliki ratusan trip dalam sehari, setiap keberangkatan dari stasiun awal adalah satu trip tersendiri. Pada konteks LRT Jakarta, satu trip mewakilkan satu perjalanan dari Pegangsaan Dua ke Velodrome atau sebaliknya dari Velodrome ke Pegangsaan Dua pada waktu spesifik tertentu.
    

---

**Field yang Direferensikan di File Lain**

| Field | Direferensikan di | Sebagai |
|---|---|---|
| `trip_id` | `stop_times.txt` | Foreign ID — setiap waktu pemberhentian merujuk ke trip |
| `trip_id` | `frequencies.txt` | Foreign ID — frekuensi keberangkatan per trip |
| `trip_id` | `transfers.txt` | Foreign ID — aturan transfer antar trip |
| `service_id` | `calendar.txt` | Foreign ID — jadwal hari operasi |
| `service_id` | `calendar_dates.txt` | Foreign ID — pengecualian jadwal |
| `shape_id` | `shapes.txt` | Foreign ID — jalur geometri di peta |
| `route_id` | `routes.txt` | Foreign ID — rute induk trip |

---

**Skema kolom trips.txt** 

| Field | Tipe Data | Status | Deskripsi |
|---|---|---|---|
| `route_id` | Foreign ID → `routes.route_id` | **Wajib** | ID rute yang dilayani trip ini. Untuk LRT Jakarta diisi `LRTJ_S`. |
| `service_id` | Foreign ID → `calendar.service_id` | **Wajib** | ID jadwal operasi. Merujuk ke `calendar.txt` atau `calendar_dates.txt`. Memberikan informasi trip ini berjalan everydays atau weekdays atau weekends saja |
| `trip_id` | Unique ID | **Wajib** | ID unik setiap perjalanan. |
| `trip_headsign` | Text | Opsional | Teks tujuan yang tampil di papan depan kendaraan |
| `trip_short_name` | Text | Opsional | Nama publik trip seperti nomor kereta  |
| `direction_id` | Enum | Opsional | Arah perjalanan. Misal untuk LRT Jakarta Lin Selatan Fase 1A nilai `0` = satu arah (Pegangsaan Dua → Velodrome), `1` = arah sebaliknya. |
| `block_id` | ID | Opsional | ID blok operasi untuk menandai trip yang dilayani oleh kendaraan yang sama secara berurutan. |
| `shape_id` | Foreign ID → `shapes.shape_id` | **Kondisional** | ID bentuk jalur di peta. **Wajib** jika ada `continuous_pickup/drop_off`. Disarankan diisi untuk tampilan rute di peta. |
| `wheelchair_accessible` | Enum | Opsional | Aksesibilitas kursi roda. Lihat tabel nilai di bawah. |
| `bikes_allowed` | Enum | Opsional | Apakah sepeda diperbolehkan untuk masuk ke dalam kendaraan |
| `cars_allowed` | Enum | Opsional | Apakah kendaraan bermotor/mobil diperbolehkan masuk ke dalam kendaraan |

---

**`wheelchair_accessible`**

| Nilai | Deskripsi |
|:---:|---|
| `0` atau kosong | Tidak ada informasi |
| `1` | Bisa diakses kursi roda | 
| `2` | Tidak bisa diakses kursi roda | 

**`bikes_allowed`** & **`cars_allowed`**

| Nilai | Deskripsi | 
|:---:|---|
| `0` atau kosong | Tidak ada informasi | 
| `1` | Diperbolehkan |
| `2` | Tidak diperbolehkan |

### Panduan Pengisian `trips.txt` — LRT Jakarta Lin Selatan Fase 1A

---

Berdasarkan informasi yang didapatkan dari website resmi [Jadwal Keberangkatan LRT Jakarta](https://www.lrtjakarta.co.id/jadwal.html), untuk LRT Jakarta tidak ada perbedaan jenis operasi berdasarkan hari. Berikut rinciannya:

Everyday (Weekday/Weekend)

| Jenis Trip | Asal | Tujuan | Jumlah Trip |
|---|---|---|:---:|
| Full trip | Pegangsaan Dua | Velodrome | 102 |
| Full trip | Velodrome | Pegangsaan Dua | 102 |
| **Total** | | | **204** |

---

**`route_id`**

Diisi sesuai dengan `route_id` yang ada di `routes.txt` yaitu `LRTJ_S`.

---

**`service_id`**

Direferensikan dari `calendar.txt`. Untuk LRT Jakarta hanya terdapat satu nilai `service_id` yang menunjukkan hari operasi trip:

| `service_id` | Hari Operasi |
|---|---|
| `ED` | Everyday (Setiap Hari) — Senin s/d Minggu |

---

**`trip_id`**


Penamaan `trip_id` terdiri dari **5 komponen** yang dipisahkan tanda hubung (`-`):

```
S  -  ED  -  PGD  -  VEL  -  001
│      │      │       │       │
│      │      │       │       └── Nomor urut keberangkatan (3 digit, mulai 001)
│      │      │       └────────── Kode stasiun tujuan
│      │      └────────────────── Kode stasiun asal
│      └───────────────────────── Jenis hari operasi
└──────────────────────────────── Kode lin
```

| Komponen | Nilai | Keterangan |
|---|---|---|
| Kode lin | `S` | South Line — Lin Selatan |
| Jenis hari | `ED` | Everyday/Setiap Hari (Senin-Minggu) |
| Kode asal | `PGD` / `VEL` | Kode stasiun keberangkatan |
| Kode tujuan | `VEL` / `PGD` | Kode stasiun tujuan |
| Nomor urut | `001` s/d `102` | Urutan keberangkatan dalam satu hari |

**Daftar trip:**

| `trip_id` | Jenis | Hari | Asal | Tujuan | Jumlah |
|---|---|---|---|---|:---:|
| `S-ED-PGD-VEL-001` s/d `102` | Full trip | Everyday/Setiap Hari | Pegangsaan Dua | Velodrome | 102 |
| `S-ED-VEL-PGD-001` s/d `102` | Full trip | Everyday/Setiap Hari | Velodrome | Pegangsaan Dua | 102 |

**Catatan:**
- Nomor urut menggunakan 3 digit dengan leading zero (`001`, `002`, dst) agar urutan alfanumerik tetap konsisten.

---

**`trip_headsign`**

Diisi berdasarkan stasiun tujuan akhir aktual dari setiap trip yaitu stasiun terakhir yang dilayani kereta. Teks ini yang ditampilkan di papan depan kereta maupun aplikasi perjalanan.

| Kondisi | `trip_headsign` |
|---|---|
| Full trip ke arah Selatan | `Velodrome` |
| Full trip ke arah Utara | `Pegangsaan Dua` |

---

**`direction_id`**

Diisi berdasarkan arah operasi kereta. Tidak digunakan untuk routing oleh OTP, hanya untuk membedakan arah saat menampilkan jadwal.

| Nilai | Arah | 
|:---:|---|
| `0` | Ke Selatan (Semua trip menuju Velodrome) |
| `1` | Ke Utara (Semua trip menuju Pegangsaan Dua) |

---

**`shape_id`**

Merujuk ke `shape_id` di `shapes.txt` yang berisi koordinat jalur kereta di peta. Diisi sesuai arah trip karena jalur Selatan dan Utara memiliki urutan koordinat yang berlawanan.

| Trip | `shape_id` |
|---|---|
| Full trip PGD → VEL | `shape_PGD_VEL` |
| Full trip VEL → PGD | `shape_VEL_PGD` |

---

**`wheelchair_accessible`**

Diisi `1` karena seluruh rangkaian kereta LRT Jakarta dilengkapi fasilitas aksesibel kursi roda termasuk ruang khusus di dalam kereta.

| Nilai | Deskripsi |
|:---:|---|
| `1` | Semua trip LRT Jakarta dapat diakses kursi roda |

---

**`bikes_allowed`**

Diisi `1` karena LRT Jakarta mengizinkan penumpang membawa sepeda ke dalam kereta dengan ketentuan yang berlaku.

| Nilai | Deskripsi |
|:---:|---|
| `1` | Sepeda diperbolehkan masuk ke dalam kereta LRT Jakarta |

In [12]:
import pandas as pd
import csv

TRIP_CONFIG = [
    # (prefix trip_id, service_id, direction, headsign, shape_id, count)
    ('S-ED-PGD-VEL',    'ED', 0, 'Velodrome',  'shape_PGD_VEL', 102),
    ('S-ED-VEL-PGD',    'ED', 1, 'Pegangsaan Dua',  'shape_VEL_PGD', 102)
]

rows = []
for prefix, service_id, direction_id, headsign, shape_id, count in TRIP_CONFIG:
    for i in range(1, count + 1):
        rows.append({
            'trip_id'              : f"{prefix}-{str(i).zfill(3)}",
            'route_id'             : 'LRTJ_S', # route_id (Lin Selatan)
            'service_id'           : service_id,
            'trip_headsign'        : headsign,
            'direction_id'         : direction_id,
            'shape_id'             : shape_id,
            'wheelchair_accessible': 1,
            'bikes_allowed'        : 1,
        })

df_trips = pd.DataFrame(rows)

df_trips.to_csv(
    'trips.txt',
    index=False,
    encoding='utf-8',
    lineterminator='\n',
    quoting=csv.QUOTE_MINIMAL
)

print(f"✅ trips.txt berhasil dibuat — {len(df_trips)} trip total\n")
print("=== Ringkasan per prefix ===")
print(df_trips.groupby(['service_id', 'direction_id', 'trip_headsign'])
      .size()
      .reset_index(name='jumlah_trip')
      .to_string(index=False))
print(f"\nTotal trip: {len(df_trips)}")

df_trips

✅ trips.txt berhasil dibuat — 204 trip total

=== Ringkasan per prefix ===
service_id  direction_id  trip_headsign  jumlah_trip
        ED             0      Velodrome          102
        ED             1 Pegangsaan Dua          102

Total trip: 204


,trip_id,route_id,service_id,trip_headsign,direction_id,shape_id,wheelchair_accessible,bikes_allowed
0,S-ED-PGD-VEL-001,LRTJ_S,ED,Velodrome,0,shape_PGD_VEL,1,1
1,S-ED-PGD-VEL-002,LRTJ_S,ED,Velodrome,0,shape_PGD_VEL,1,1
2,S-ED-PGD-VEL-003,LRTJ_S,ED,Velodrome,0,shape_PGD_VEL,1,1
3,S-ED-PGD-VEL-004,LRTJ_S,ED,Velodrome,0,shape_PGD_VEL,1,1
4,S-ED-PGD-VEL-005,LRTJ_S,ED,Velodrome,0,shape_PGD_VEL,1,1
...,...,...,...,...,...,...,...,...
199,S-ED-VEL-PGD-098,LRTJ_S,ED,Pegangsaan Dua,1,shape_VEL_PGD,1,1
200,S-ED-VEL-PGD-099,LRTJ_S,ED,Pegangsaan Dua,1,shape_VEL_PGD,1,1
201,S-ED-VEL-PGD-100,LRTJ_S,ED,Pegangsaan Dua,1,shape_VEL_PGD,1,1
202,S-ED-VEL-PGD-101,LRTJ_S,ED,Pegangsaan Dua,1,shape_VEL_PGD,1,1


## calendar.txt (Conditionally Required)

File `calendar.txt` mendefinisikan **jadwal hari operasi** layanan transportasi umum, menentukan hari apa saja dalam seminggu suatu layanan beroperasi, beserta rentang tanggal berlakunya. File ini menjadi acuan bagi `trips.txt` melalui field `service_id` untuk menentukan kapan sebuah trip dijalankan.

---

**Field yang Direferensikan di File Lain**

| Field | Direferensikan di | Sebagai |
|---|---|---|
| `service_id` | `trips.txt` | Foreign ID — setiap trip merujuk ke jadwal operasi |
| `service_id` | `calendar_dates.txt` | Foreign ID — pengecualian jadwal per tanggal tertentu |

---

**Skema kolom calendar.txt**

| Field | Tipe Data | Keharusan | Deskripsi |
|---|---|---|---|
| `service_id` | Unique ID | **Wajib** | ID unik pengenal jadwal operasi. Direferensikan dari `trips.txt`.|
| `monday` | Enum | **Wajib** | Beroperasi pada hari Senin. `1` = beroperasi, `0` = tidak beroperasi. |
| `tuesday` | Enum | **Wajib** | Beroperasi pada hari Selasa. `1` = beroperasi, `0` = tidak beroperasi. |
| `wednesday` | Enum | **Wajib** | Beroperasi pada hari Rabu. `1` = beroperasi, `0` = tidak beroperasi. |
| `thursday` | Enum | **Wajib** | Beroperasi pada hari Kamis. `1` = beroperasi, `0` = tidak beroperasi. |
| `friday` | Enum | **Wajib** | Beroperasi pada hari Jumat. `1` = beroperasi, `0` = tidak beroperasi. |
| `saturday` | Enum | **Wajib** | Beroperasi pada hari Sabtu. `1` = beroperasi, `0` = tidak beroperasi. |
| `sunday` | Enum | **Wajib** | Beroperasi pada hari Minggu. `1` = beroperasi, `0` = tidak beroperasi. |
| `start_date` | Date | **Wajib** | Tanggal mulai berlakunya jadwal dalam format `YYYYMMDD`. |
| `end_date` | Date | **Wajib** | Tanggal akhir berlakunya jadwal dalam format `YYYYMMDD`. Tanggal ini termasuk dalam rentang yang berlaku. |

### Panduan Pengisian `calendar.txt` — LRT Jakarta Lin Selatan Fase 1

---

**`service_id`**

ID unik pengenal jadwal operasi yang direferensikan dari `trips.txt`. Untuk LRT Jakarta hanya terdapat satu nilai:

| `service_id` | Hari Operasi |
|---|---|
| `ED` | Everyday — Senin s/d Minggu |

---

**`monday` `tuesday` `wednesday` `thursday` `friday` `saturday` `sunday`**

Diisi dengan nilai `1` (beroperasi) atau `0` (tidak beroperasi) untuk setiap hari dalam seminggu. Disesuaikan dengan jenis `service_id`:

| `service_id` | `monday` | `tuesday` | `wednesday` | `thursday` | `friday` | `saturday` | `sunday` |
|---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| `ED` | `1` | `1` | `1` | `1` | `1` | `1` | `1` |

---

**`start_date`**

Diisi `20191201` karena LRT Jakarta mulai beroperasi secara komersial pada **1 Desember 2019**.

---

**`end_date`**

Diisi `20301231` sebagai tanggal akhir berlakunya jadwal yaitu **31 Desember 2030**. Tanggal ini bersifat sementara dan perlu diperbarui secara berkala sesuai kebijakan operasional LRT Jakarta.

---

**Catatan Penting**

- **Hari libur nasional** tidak ditangani di `calendar.txt`, pengecualian jadwal seperti libur nasional didefinisikan secara terpisah di `calendar_dates.txt`.
- Jika pada hari libur nasional LRT Jakarta beroperasi dengan **jadwal weekend**, perlu ditambahkan entri di `calendar_dates.txt`
- Jika pada hari libur nasional LRT Jakarta **tidak beroperasi sama sekali**, cukup tambahkan entri di `calendar_dates.txt` dengan `exception_type=2` untuk menonaktifkan service pada tanggal tersebut.

In [13]:
import pandas as pd
import csv

# Data calendar LRT Jakarta
calendar_data = {
    'service_id' : ['ED'],
    'monday'     : [1],
    'tuesday'    : [1],           
    'wednesday'  : [1],            
    'thursday'   : [1],            
    'friday'     : [1],           
    'saturday'   : [1],            
    'sunday'     : [1],            
    'start_date' : ['20191201'],
    'end_date'   : ['20301231']
}

df_calendar = pd.DataFrame(calendar_data)
df_calendar.to_csv(
    'calendar.txt',
    index=False,
    encoding='utf-8',
    lineterminator='\n',
    quoting=csv.QUOTE_MINIMAL
)

print("✅ calendar.txt berhasil dibuat!")
print("\n📄 Preview isi file:")
df_calendar

✅ calendar.txt berhasil dibuat!

📄 Preview isi file:


,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,ED,1,1,1,1,1,1,1,20191201,20301231


## shapes.txt (Optional)

File `shapes.txt` mendefinisikan **jalur geometri** yang dilalui kendaraan melayani sebuah rute di peta. Berupa rangkaian titik koordinat yang membentuk garis rute secara visual. File ini bersifat opsional namun **sangat direkomendasikan** untuk semua layanan berbasis rute karena digunakan oleh aplikasi trip planner seperti OTP dan Google Maps untuk menampilkan jalur rute yang akurat di peta. Shape tidak harus melewati tepat di atas titik halte/stasiun, namun semua halte/stasiun dalam sebuah trip harus berada dalam jarak yang dekat dengan garis shape tersebut.

---

**Field yang Direferensikan di File Lain**

| Field | Direferensikan di | Sebagai |
|---|---|---|
| `shape_id` | `trips.txt` | Foreign ID — setiap trip merujuk ke shape jalurnya |
| `shape_dist_traveled` | `stop_times.txt` | Konsistensi satuan jarak antar file |

---

**Skema kolom shapes.txt**

| Field | Tipe Data | Keharusan | Deskripsi |
|---|---|---|---|
| `shape_id` | ID | **Wajib** | ID pengenal shape. Satu shape terdiri dari banyak titik koordinat yang semuanya berbagi `shape_id` yang sama. |
| `shape_pt_lat` | Latitude | **Wajib** | Koordinat lintang titik shape dalam format desimal WGS84. |
| `shape_pt_lon` | Longitude | **Wajib** | Koordinat bujur titik shape dalam format desimal WGS84. |
| `shape_pt_sequence` | Non-negative integer | **Wajib** | Urutan titik dalam shape. Nilai harus selalu meningkat sepanjang perjalanan namun tidak harus berurutan berurutan (boleh `0, 5, 10` dst). |
| `shape_dist_traveled` | Non-negative float | Opsional | Jarak aktual yang ditempuh dari titik pertama shape hingga titik ini. Digunakan trip planner untuk menampilkan porsi jalur yang benar di peta. Nilai harus selalu meningkat sesuai `shape_pt_sequence` dan satuan harus konsisten dengan `stop_times.txt`. |

### Panduan Pengisian `shapes.txt` — LRT Jakarta Lin Selatan Fase 1A

---

**`shape_id`**

Untuk LRT Jakarta Lin Selatan Fase 1A diperlukan **2 shape** sesuai variasi trip yang ada:

| `shape_id` | Digunakan untuk | Keterangan |
|---|---|---|
| `shape_PGD_VEL` | Full trip arah Selatan | Seluruh jalur PGD → VEL |
| `shape_VEL_PGD` | Full trip arah Utara | Seluruh jalur VEL → PGD |

---

**`shape_pt_lat`** & **`shape_pt_lon`**

Koordinat titik-titik jalur diambil dari **Relation OpenStreetMap** yang tersedia secara terbuka. Setiap relation OSM berisi urutan titik koordinat yang membentuk jalur LRT Jakarta Lin Selatan Fase 1A secara lengkap.

| `shape_id` | Sumber Relation OSM |
|---|---|
| `shape_PGD_VEL` | `https://www.openstreetmap.org/relation/10693160` |
| `shape_VEL_PGD` | `https://www.openstreetmap.org/relation/10693119` |

Untuk mengambil data koordinat dari OSM digunakan dua cara yaitu secara online **Overpass API** dengan beberapa mirror sebagai fallback atau secara offline yaitu ambil relation dari https://overpass-turbo.eu/ sebagai GeoJSON dengan query sebagai berikut:

**Online**
```python
OVERPASS_MIRRORS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.osm.ch/api/interpreter",
    "https://overpass.openstreetmap.ru/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]
```

**Offline**
```
1. Buka overpass-turbo.eu
2. Jalankan query:
    
    [out:json][timeout:90];
    relation(relation_id);
    out body;
    >;
    out skel qt;
    
3. Klik Export → GeoJSON
4. Simpan sebagai: relation_{relation_id}.geojson
```
---

**`shape_pt_sequence`**

Diisi dengan angka urut yang **selalu meningkat** untuk setiap titik dalam satu shape dimulai dari `0` untuk titik pertama. Nilai tidak harus berurutan berurutan (boleh `0, 1, 2` atau `0, 10, 20` dst) selama nilainya selalu bertambah.

| Titik | `shape_pt_sequence` |
|---|---|
| Titik pertama (stasiun asal) | `0` |
| Titik berikutnya | `1`, `2`, `3`, ... |
| Titik terakhir (stasiun tujuan) | `n` |

---

Berikut versi yang sudah dirapikan, konsisten, dan siap untuk dokumentasi markdown:

---

**`shape_dist_traveled` pada `shapes.txt`**

`shape_dist_traveled` merepresentasikan **jarak kumulatif aktual** yang ditempuh sepanjang shape, dihitung dari titik pertama hingga setiap titik berikutnya.

Nilai ini dihitung berdasarkan jarak antar pasangan titik koordinat:

* `shape_pt_lat`
* `shape_pt_lon`

yang disusun sesuai `shape_pt_sequence`.

**Pendekatan Perhitungan**

Terdapat **dua pendekatan umum** dalam menghitung `shape_dist_traveled`:


**1. Menggunakan Rumus Haversine (Manual)**

Pendekatan ini menghitung jarak antar dua titik di permukaan bumi menggunakan model bola bumi.

```python
def haversine_m(lat1, lon1, lat2, lon2):
    """Jarak antara dua titik koordinat dalam meter (Haversine)."""
    R = 6371000.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))
```

Perhitungan dilakukan secara **kumulatif**:

* Titik 0 → `0`
* Titik 1 → jarak(0 → 1)
* Titik 2 → jarak(0 → 1) + jarak(1 → 2)
* dan seterusnya

**Cocok untuk:**

* Implementasi manual (pandas / numpy)
* Tanpa dependency GIS


**2. Menggunakan `gtfs_kit.append_dist_to_shapes()`**

Fungsi dari `gtfs_kit` menghitung jarak menggunakan pendekatan berbasis proyeksi:

**Proses:**

1. Mengubah koordinat dari **WGS84 (lat/lon)** ke sistem proyeksi berbasis meter (**UTM**)
2. Menghitung jarak antar titik menggunakan **Euclidean distance (planar)**
3. Menjumlahkannya secara **kumulatif** sepanjang shape

Sehingga metode yang digunakan adalah:

**`Projected distance (UTM) + Euclidean distance`**

**Keunggulan**

* Lebih cepat dibanding Haversine
* Akurat untuk jarak pendek (seperti shape GTFS)
* Konsisten dengan pendekatan GIS umum

**Catatan**

* Kedua metode **valid dalam spesifikasi GTFS**
* Hasil perhitungan dapat sedikit berbeda (umumnya sangat kecil)
* Hal yang paling penting:

  * Nilai harus **monoton meningkat**
  * Menggunakan **satuan yang konsisten** (meter atau kilometer)

---

**Catatan Penting**

- Satuan `shape_dist_traveled` harus **konsisten** dengan satuan yang digunakan di `stop_times.txt`.
- Shape `shape_VEL_PGD` adalah **urutan koordinat yang dibalik** dari `shape_PGD_VEL`, titik pertama menjadi titik terakhir dan `shape_dist_traveled` dihitung ulang dari arah VEL.
- `shape_dist_traveled` **sangat disarankan diisi** untuk LRT Jakarta agar OTP dapat menampilkan posisi kereta dan progres perjalanan yang akurat di peta.

### **Build `shape_PGD_VEL` & `shape_VEL_PGD`**

#### Build `shape_PGD_VEL` & `shape_VEL_PGD` with gtfs_kit for `shape_dist_traveled`

In [14]:
"""
Build file shapes.txt menggunakan data geometri jalur rel LRT Jakarta yang diperoleh dari GeoJSON lokal atau melalui API Overpass (OpenStreetMap).

Shape yang dihasilkan:
  - shape_PGD_VEL : Pegangsaan Dua → Velodrome
  - shape_VEL_PGD : Velodrome → Pegangsaan Dua

Catatan:
  - File GeoJSON bisa didapatkan dari https://overpass-turbo.eu/ lakukan query untuk relation_id jalur tertentu kemudian download sebagai GeoJSON.
  - Terdapat proses trimming untuk memotong segmen pada stasiun terminus (awal & akhir) agar jalur rel tepat dimulai dari stop_id pertama.
  - Jika koordinat hasil parsing terbalik (arah OSM berlawanan dengan arah trip), aktifkan reverse=True pada konfigurasi RELATIONS.
  - shape_dist_traveled tidak dihitung di sini, akan dihitung menggunakan library gtfs_kit.
  - Urutan proses per shape: trim terminus → reverse (jika aktif) → tulis shapes.txt
  - Perhitungan shape_dist_traveled menggunakan gtfs_kit.append_dist_to_shapes(feed) setelah file ini dibuat.

MODE OPERASI (otomatis dipilih berdasarkan file yang tersedia):
  1. GeoJSON lokal  → jika file .geojson tersedia (dari overpass-turbo.eu)
  2. Online         → fetch dari Overpass API
"""

import requests
import csv
import math
import json
import os
import time
from datetime import datetime

# File path
OUTPUT_SHAPES = "shapes.txt"
STOPS_FILE    = "stops.txt"

# Terminus stop_id per shape
# first_stop_id : stop pertama yang dilayani trip (platform)
# last_stop_id  : stop terakhir yang dilayani trip (platform)
# reverse       : True jika koordinat OSM berlawanan arah dengan trip
RELATIONS = [
    {
        "relation_id"  : 10693160, # relation_id OSM untuk jalur LRT Jakarta PGD-VEL
        "shape_id"     : "shape_PGD_VEL", # shape_id untuk jalur LRT Jakarta PGD-VEL
        "label"        : "PGD → VEL", 
        "geojson_file" : "data/shapes/relation_10693160.geojson", # File GeoJSON untuk jalur LRT Jakarta PGD-VEL
        "first_stop_id": "GPGD01", # platform PGD arah VEL
        "last_stop_id" : "GVEL01", # platform VEL arah dari PGD
        "reverse"      : True,    # koordinat OSM terbalik
    },
    {
        "relation_id"  : 10693119, # relation_id OSM untuk jalur LRT Jakarta VEL-PGD
        "shape_id"     : "shape_VEL_PGD", # shape_id untuk jalur LRT Jakarta VEL-PGD
        "label"        : "VEL → PGD",
        "geojson_file" : "data/shapes/relation_10693119.geojson", # File GeoJSON untuk jalur LRT Jakarta VEL-PGD
        "first_stop_id": "GVEL02", # platform VEL arah PGD
        "last_stop_id" : "GPGD01", # platform PGD arah dari VEL
        "reverse"      : False,   # koordinat OSM sudah benar
    },
]

OVERPASS_MIRRORS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.osm.ch/api/interpreter",
    "https://overpass.openstreetmap.ru/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]

REQUEST_HEADERS = {
    "User-Agent"  : "GTFS-Builder/1.0 (LRT Jakarta GTFS)",
    "Accept"      : "application/json",
    "Content-Type": "application/x-www-form-urlencoded",
}


# Rumus Haversine 
def haversine_m(lat1, lon1, lat2, lon2):
    """Jarak antara dua titik koordinat dalam METER — dipakai untuk chaining."""
    R  = 6371000.0
    p1 = math.radians(lat1)
    p2 = math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a  = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))


# Load koordinat stop 
def load_stop_coords(stops_file, stop_id):
    """
    Baca (lat, lon, stop_name) dari stops.txt berdasarkan stop_id.
    Raise ValueError jika stop_id tidak ditemukan.
    """
    with open(stops_file, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row["stop_id"] == stop_id:
                return float(row["stop_lat"]), float(row["stop_lon"]), row.get("stop_name", stop_id)
    raise ValueError(f"stop_id '{stop_id}' tidak ditemukan di {stops_file}")


# Trim terminus 
def find_nearest_index(coords, target_lat, target_lon):
    """Cari indeks titik di coords yang paling dekat dengan (target_lat, target_lon)."""
    best_idx  = 0
    best_dist = float("inf")
    for i, (lat, lon) in enumerate(coords):
        d = haversine_m(lat, lon, target_lat, target_lon)
        if d < best_dist:
            best_dist = d
            best_idx  = i
    return best_idx, best_dist


def trim_shape_to_terminus(coords, first_lat, first_lon, last_lat, last_lon):
    """
    Potong coords agar:
      - Dimulai dari titik terdekat ke stop pertama (first_stop)
      - Berakhir di titik terdekat ke stop terakhir (last_stop)

    Menghindari:
      - Garis jalur sebelum stasiun pertama (tampilan peta bersih)
      - trip_distance_exceeds_shape_distance warning di validator GTFS

    Return: (coords_trimmed, idx_first, idx_last, dist_first_m, dist_last_m)
    """
    idx_first, dist_first = find_nearest_index(coords, first_lat, first_lon)
    idx_last,  dist_last  = find_nearest_index(coords, last_lat,  last_lon)

    if idx_first > idx_last:
        idx_first, idx_last = idx_last, idx_first

    return coords[idx_first : idx_last + 1], idx_first, idx_last, dist_first, dist_last


# Reverse koordinat 
def reverse_coords(coords):
    """
    Balik urutan koordinat.

    Digunakan jika koordinat hasil parsing OSM berlawanan arah dengan trip.
    Proses ini dilakukan SETELAH trim terminus agar:
      1. Trim tetap bekerja dengan benar menggunakan indeks koordinat asli
      2. shape_pt_sequence di-reset dari 0 setelah reverse oleh write_shapes_txt
      3. gtfs_kit.append_dist_to_shapes() menghitung dist dari titik awal yang benar

    Contoh untuk shape_PGD_VEL:
      Sebelum reverse: VEL(0) → ... → PGD(n)  ← arah OSM
      Setelah reverse: PGD(0) → ... → VEL(n)  ← arah trip PGD→VEL ✅
    """
    return list(reversed(coords))


# GeoJSON Parser (mode offline)
def extract_coords_from_geojson(geojson_path):
    """
    Parse GeoJSON dari overpass-turbo.eu → list of (lat, lon).

    GeoJSON dari overpass-turbo berisi FeatureCollection dengan:
      - LineString       : satu segmen way
      - MultiLineString  : beberapa segmen way
      - Point            : node (diabaikan)
    """
    print(f"  → Membaca: {geojson_path}")
    with open(geojson_path, "r", encoding="utf-8") as f:
        geojson = json.load(f)

    segments = []
    n_point  = 0

    for feature in geojson.get("features", []):
        geom  = feature.get("geometry", {})
        gtype = geom.get("type", "")

        if gtype == "LineString":
            segments.append([(c[1], c[0]) for c in geom["coordinates"]])
        elif gtype == "MultiLineString":
            for line in geom["coordinates"]:
                segments.append([(c[1], c[0]) for c in line])
        elif gtype == "Point":
            n_point += 1

    if not segments:
        raise ValueError("Tidak ada LineString/MultiLineString di GeoJSON!")

    print(f"  ✅ {len(segments)} segmen ditemukan, {n_point} point diabaikan")
    ordered = _chain_segments(segments)
    print(f"  ✅ Chain selesai → {len(ordered):,} titik koordinat")
    return ordered


def _chain_segments(segments):
    """
    Sambungkan list segmen menjadi satu jalur berurutan.
    Algoritma greedy berdasarkan jarak ujung-ke-ujung terdekat.
    Gap > 50m diberi warning tapi tetap disambung.
    """
    SNAP_THRESHOLD_M = 50.0
    remaining = [list(seg) for seg in segments]
    result    = remaining.pop(0)

    while remaining:
        tail      = result[-1]
        best_idx  = None
        best_dist = float("inf")
        best_rev  = False

        for i, seg in enumerate(remaining):
            d_head = haversine_m(tail[0], tail[1], seg[0][0],  seg[0][1])
            d_tail = haversine_m(tail[0], tail[1], seg[-1][0], seg[-1][1])
            if d_head < best_dist:
                best_dist, best_idx, best_rev = d_head, i, False
            if d_tail < best_dist:
                best_dist, best_idx, best_rev = d_tail, i, True

        seg = remaining.pop(best_idx)
        if best_rev:
            seg = list(reversed(seg))
        if best_dist > SNAP_THRESHOLD_M:
            print(f"  ⚠️  Gap {best_dist:.1f}m — tetap disambung")

        skip = 1 if haversine_m(tail[0], tail[1], seg[0][0], seg[0][1]) < 1.0 else 0
        result.extend(seg[skip:])

    return result


# Overpass API (mode online)
def try_request(mirror, query):
    """POST raw dulu, fallback ke GET params."""
    try:
        print(f"    [POST raw] ...")
        resp = requests.post(mirror, data=query, headers=REQUEST_HEADERS, timeout=90)
        resp.raise_for_status()
        return resp.json()
    except Exception as e_post:
        print(f"    ⚠️  POST gagal: {e_post}")

    try:
        print(f"    [GET params] ...")
        resp = requests.get(mirror, params={"data": query}, headers=REQUEST_HEADERS, timeout=90)
        resp.raise_for_status()
        return resp.json()
    except Exception as e_get:
        print(f"    ⚠️  GET gagal: {e_get}")
        raise e_get


def fetch_relation_online(relation_id):
    """Fetch relation dari Overpass API, coba semua mirror secara berurutan."""
    query = (
        f"[out:json][timeout:90];"
        f"relation({relation_id});"
        f"out body;>;"
        f"out skel qt;"
    )
    last_err = None
    for mirror in OVERPASS_MIRRORS:
        print(f"  → Mencoba {mirror} ...")
        try:
            data = try_request(mirror, query)
            print(f"  ✅ Berhasil! {len(data['elements'])} elements")
            return data
        except Exception as e:
            print(f"  ✗  Mirror gagal\n")
            last_err = e
            time.sleep(3)
    raise RuntimeError(f"Semua mirror gagal: {last_err}")


def extract_coords_from_overpass(data):
    """Parse Overpass JSON → list of (lat, lon)."""
    nodes = {e["id"]: (e["lat"], e["lon"])
             for e in data["elements"] if e["type"] == "node"}
    ways  = {e["id"]: e["nodes"]
             for e in data["elements"] if e["type"] == "way"}
    rel   = next((e for e in data["elements"] if e["type"] == "relation"), None)

    if not rel:
        raise ValueError("Relation tidak ditemukan!")

    tags        = rel.get("tags", {})
    way_members = [m for m in rel["members"] if m["type"] == "way"]
    print(f"\n  📋 Relation: {tags.get('name','N/A')} | {len(way_members)} ways")

    ordered_coords = []
    last_node_id   = None
    skipped        = 0

    for member in way_members:
        way_id = member["ref"]
        role   = member.get("role", "")

        if way_id not in ways or not ways[way_id]:
            skipped += 1
            continue

        way_nodes  = ways[way_id]
        first_node = way_nodes[0]
        last_node  = way_nodes[-1]

        if last_node_id is None:
            reverse = (role == "backward")
        else:
            if last_node_id == first_node:
                reverse = False
            elif last_node_id == last_node:
                reverse = True
            else:
                if first_node in nodes and last_node in nodes and last_node_id in nodes:
                    c       = nodes[last_node_id]
                    d_first = haversine_m(c[0], c[1], nodes[first_node][0], nodes[first_node][1])
                    d_last  = haversine_m(c[0], c[1], nodes[last_node][0],  nodes[last_node][1])
                    reverse = d_last < d_first
                else:
                    reverse = (role == "backward")

        seq = list(reversed(way_nodes)) if reverse else way_nodes
        for j, node_id in enumerate(seq):
            if node_id not in nodes:
                continue
            coord = nodes[node_id]
            if ordered_coords and j == 0 and ordered_coords[-1] == coord:
                continue
            ordered_coords.append(coord)
        last_node_id = seq[-1]

    if skipped:
        print(f"  ⚠️  {skipped} ways dilewati")

    return ordered_coords


# Selector source data
def get_coords(rel_cfg):
    """
    Pilih sumber data secara otomatis:
      1. GeoJSON lokal → jika file .geojson ada
      2. Online        → fetch dari Overpass API
    """
    geojson_file = rel_cfg.get("geojson_file", "")
    if geojson_file and os.path.exists(geojson_file):
        print(f"  📂 Mode: GeoJSON lokal")
        return extract_coords_from_geojson(geojson_file)

    print(f"  🌐 Mode: Online Overpass API")
    data = fetch_relation_online(rel_cfg["relation_id"])
    return extract_coords_from_overpass(data)


# Output
def write_shapes_txt(all_shape_data, output_file):
    """
    Tulis shapes.txt tanpa kolom shape_dist_traveled.
    shape_dist_traveled akan dihitung oleh gtfs_kit.append_dist_to_shapes().
    shape_pt_sequence di-reset dari 0 untuk setiap shape.
    """
    fieldnames = [
        "shape_id", "shape_pt_lat", "shape_pt_lon", "shape_pt_sequence"
    ]
    rows = []
    for shape_id, coords in all_shape_data.items():
        for seq, (lat, lon) in enumerate(coords):
            rows.append({
                "shape_id"         : shape_id,
                "shape_pt_lat"     : f"{lat:.7f}",
                "shape_pt_lon"     : f"{lon:.7f}",
                "shape_pt_sequence": seq,
            })

    with open(output_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    return rows


def save_json_ref(shape_id, relation_id, coords):
    """Simpan JSON referensi koordinat per shape (untuk keperluan debug/cek)."""
    total_m = sum(
        haversine_m(coords[i-1][0], coords[i-1][1],
                    coords[i][0],   coords[i][1])
        for i in range(1, len(coords))
    )
    json_path = f"shape_ref_{shape_id}.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump({
            "shape_id"     : shape_id,
            "relation_id"  : relation_id,
            "total_points" : len(coords),
            "total_dist_m" : round(total_m, 1),
            "total_dist_km": round(total_m / 1000, 3),
            "coords"       : [{"lat": lat, "lon": lon} for lat, lon in coords]
        }, f, indent=2)
    return json_path, total_m

def main():
    print("=" * 60)
    print(" OSM → GTFS shapes.txt  |  LRT Jakarta Lin Selatan Fase 1A")
    print(f" {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f" shape_dist_traveled : tidak dihitung — gunakan gtfs_kit setelahnya")
    print(f" Reverse             : per konfigurasi RELATIONS")
    print("=" * 60)
    print()
    print(" Prioritas sumber data:")
    print("   1. GeoJSON lokal (overpass-turbo.eu) — jika file ada")
    print("   2. Online Overpass API               — jika tidak ada")
    print()
    for rel in RELATIONS:
        ada     = os.path.exists(rel["geojson_file"])
        status  = "✅ ada → pakai GeoJSON" if ada else "⬜ tidak ada → fetch online"
        rev_str = "🔄 reverse=True" if rel.get("reverse") else "  reverse=False"
        print(f"   {rel['label']:<12} : {rel['geojson_file']:<40} {status}  |  {rev_str}")
    print()

    stops_available = os.path.exists(STOPS_FILE)
    if not stops_available:
        print(f"⚠️  {STOPS_FILE} tidak ditemukan — trim terminus dinonaktifkan")
    else:
        print(f"📂 stops.txt ditemukan — trim terminus aktif")
    print()

    all_shape_data = {}

    for rel_cfg in RELATIONS:
        shape_id   = rel_cfg["shape_id"]
        label      = rel_cfg["label"]
        do_reverse = rel_cfg.get("reverse", False)

        print(f"{'─'*55}")
        print(f"🔄  Memproses : {label}  {'(reverse aktif)' if do_reverse else ''}")
        print(f"{'─'*55}")

        try:
            coords      = get_coords(rel_cfg)
            n_raw       = len(coords)
            total_m_raw = sum(
                haversine_m(coords[i-1][0], coords[i-1][1],
                            coords[i][0],   coords[i][1])
                for i in range(1, len(coords))
            )

            # ── Step 1: Trim terminus ──────────────────────────────────────────
            trimmed = False
            if stops_available and rel_cfg.get("first_stop_id") and rel_cfg.get("last_stop_id"):
                try:
                    f_lat, f_lon, f_name = load_stop_coords(STOPS_FILE, rel_cfg["first_stop_id"])
                    l_lat, l_lon, l_name = load_stop_coords(STOPS_FILE, rel_cfg["last_stop_id"])

                    coords_trimmed, idx_f, idx_l, d_f, d_l = trim_shape_to_terminus(
                        coords, f_lat, f_lon, l_lat, l_lon
                    )
                    trimmed = True

                    print(f"\n  ✂️  Trim terminus:")
                    print(f"     Stop pertama : {rel_cfg['first_stop_id']} — {f_name}")
                    print(f"       indeks {idx_f} | jarak ke shape: {d_f:.1f} m")
                    print(f"     Stop terakhir: {rel_cfg['last_stop_id']} — {l_name}")
                    print(f"       indeks {idx_l} | jarak ke shape: {d_l:.1f} m")
                    print(f"     Titik dipotong: {idx_f} di awal, {n_raw - 1 - idx_l} di akhir")

                    coords = coords_trimmed

                except ValueError as e:
                    print(f"  ⚠️  Trim gagal: {e} — pakai koordinat penuh")

            # ── Step 2: Reverse (setelah trim, sebelum tulis) ──────────────────
            if do_reverse:
                coords = reverse_coords(coords)
                print(f"\n  🔄  Reverse selesai:")
                print(f"     Titik pertama (baru) : {coords[0]}")
                print(f"     Titik terakhir (baru): {coords[-1]}")

            if not coords:
                print(f"  ❌ Tidak ada koordinat!")
                continue

            all_shape_data[shape_id] = coords

            total_m = sum(
                haversine_m(coords[i-1][0], coords[i-1][1],
                            coords[i][0],   coords[i][1])
                for i in range(1, len(coords))
            )

            print(f"\n  📊 Hasil:")
            print(f"     Shape ID       : {shape_id}")
            print(f"     Titik (raw)    : {n_raw:,}")
            print(f"     Titik (final)  : {len(coords):,}  "
                  f"({'trim+reverse' if trimmed and do_reverse else 'trim' if trimmed else 'reverse' if do_reverse else 'tanpa trim/reverse'})")
            print(f"     Jarak raw      : {total_m_raw:,.1f} m  ({total_m_raw/1000:.3f} km)")
            print(f"     Jarak final    : {total_m:,.1f} m  ({total_m/1000:.3f} km)")
            print(f"     Titik pertama  : {coords[0]}")
            print(f"     Titik terakhir : {coords[-1]}")

            json_path, _ = save_json_ref(shape_id, rel_cfg["relation_id"], coords)
            print(f"     JSON referensi : {json_path}")

        except Exception as e:
            print(f"  ❌ Error: {e}")
            import traceback; traceback.print_exc()

        time.sleep(1)

    if not all_shape_data:
        print("\n⚠️  Tidak ada shape yang berhasil. shapes.txt tidak dibuat.")
        return

    rows = write_shapes_txt(all_shape_data, OUTPUT_SHAPES)

    print(f"\n{'='*55}")
    print(f"✅ shapes.txt berhasil dibuat!")
    print(f"   Path    : {OUTPUT_SHAPES}")
    print(f"   Kolom   : shape_id, shape_pt_lat, shape_pt_lon, shape_pt_sequence")
    print(f"   Rows    : {len(rows):,}")
    print()
    print(f"   {'Shape ID':<22} {'Titik':>8}  {'Jarak (m)':>12}  {'Jarak (km)':>12}  {'Reverse':>8}")
    print(f"   {'─'*65}")
    for rel_cfg in RELATIONS:
        sid    = rel_cfg["shape_id"]
        coords = all_shape_data.get(sid)
        if not coords:
            continue
        total_m = sum(
            haversine_m(coords[i-1][0], coords[i-1][1],
                        coords[i][0],   coords[i][1])
            for i in range(1, len(coords))
        )
        rev_str = "✅" if rel_cfg.get("reverse") else "—"
        print(f"   {sid:<22} {len(coords):>8,}  "
              f"{total_m:>12,.1f}  {total_m/1000:>12.3f}  {rev_str:>8}")

if __name__ == "__main__":
    main()

 OSM → GTFS shapes.txt  |  LRT Jakarta Lin Selatan Fase 1A
 2026-04-26 21:34:15
 shape_dist_traveled : tidak dihitung — gunakan gtfs_kit setelahnya
 Reverse             : per konfigurasi RELATIONS

 Prioritas sumber data:
   1. GeoJSON lokal (overpass-turbo.eu) — jika file ada
   2. Online Overpass API               — jika tidak ada

   PGD → VEL    : data/shapes/relation_10693160.geojson    ✅ ada → pakai GeoJSON  |  🔄 reverse=True
   VEL → PGD    : data/shapes/relation_10693119.geojson    ✅ ada → pakai GeoJSON  |    reverse=False

📂 stops.txt ditemukan — trim terminus aktif

───────────────────────────────────────────────────────
🔄  Memproses : PGD → VEL  (reverse aktif)
───────────────────────────────────────────────────────
  📂 Mode: GeoJSON lokal
  → Membaca: data/shapes/relation_10693160.geojson
  ✅ 1 segmen ditemukan, 6 point diabaikan
  ✅ Chain selesai → 74 titik koordinat

  ✂️  Trim terminus:
     Stop pertama : GPGD01 — Stasiun LRT Pegangsaan Dua
       indeks 1 | jarak ke sh

In [15]:
# Build field shape_dist_traveled with gtfs_kit (gtfs_kit.append_dist_to_shapes(feed))
import gtfs_kit as gk

feed = gk.read_feed('.', dist_units='m') # Pastikan satuan jarak dalam meter secara konsisten
feed = gk.append_dist_to_shapes(feed)
feed.shapes.to_csv(
    "shapes.txt",
    index=False,
    encoding='utf-8',        
    lineterminator='\n',         
    quoting=csv.QUOTE_MINIMAL   
)

feed.shapes

,shape_id,shape_pt_lat,shape_pt_lon,shape_pt_sequence,shape_dist_traveled
0,shape_PGD_VEL,-6.157210,106.914284,0,0.000000
1,shape_PGD_VEL,-6.155913,106.914301,1,143.443630
2,shape_PGD_VEL,-6.155562,106.914307,2,182.315632
3,shape_PGD_VEL,-6.155466,106.914305,3,192.892201
4,shape_PGD_VEL,-6.155379,106.914297,4,202.631535
...,...,...,...,...,...
141,shape_VEL_PGD,-6.155350,106.914141,68,5339.392650
142,shape_VEL_PGD,-6.155423,106.914150,69,5347.526646
143,shape_VEL_PGD,-6.155495,106.914152,70,5355.504928
144,shape_VEL_PGD,-6.155911,106.914150,71,5401.494549


#### Cek shapes.txt untuk **`shape_PGD_VEL`** 

In [16]:
import pandas as pd
import folium

#File path
SHAPES_FILE = 'shapes.txt'
TRIPS_FILE  = 'trips.txt'

# Ambil salah satu trip dengan shape_PGD_VEL
TARGET_TRIP_ID = 'S-ED-PGD-VEL-001'

shapes = pd.read_csv(SHAPES_FILE)
trips  = pd.read_csv(TRIPS_FILE)

# Ambil shape_id dari trips.txt
trip_row = trips[trips['trip_id'] == TARGET_TRIP_ID]

if len(trip_row) == 0:
    print(f"❌ Trip {TARGET_TRIP_ID} tidak ditemukan di trips.txt")
else:
    shape_id = trip_row['shape_id'].values[0]
    print(f"✅ Trip  : {TARGET_TRIP_ID}")
    print(f"✅ Shape : {shape_id}")

    # Ambil titik shape
    shape_data = shapes[shapes['shape_id'] == shape_id].sort_values('shape_pt_sequence')

    if len(shape_data) == 0:
        print(f"❌ Shape {shape_id} tidak ditemukan di shapes.txt")
    else:
        print(f"✅ Jumlah titik shape: {len(shape_data)}")

        coords = list(zip(shape_data['shape_pt_lat'], shape_data['shape_pt_lon']))

        # Buat peta
        center_lat = shape_data['shape_pt_lat'].mean()
        center_lon = shape_data['shape_pt_lon'].mean()
        m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

        # Gambar jalur shape
        folium.PolyLine(
            coords,
            color='blue',
            weight=4,
            opacity=0.8,
            tooltip=f"Shape: {shape_id} | Trip: {TARGET_TRIP_ID}"
        ).add_to(m)

        # Titik awal shape
        folium.Marker(
            coords[0],
            popup=f"START – {shape_id}",
            icon=folium.Icon(color='green', icon='play')
        ).add_to(m)

        # Titik akhir shape
        folium.Marker(
            coords[-1],
            popup=f"END – {shape_id}",
            icon=folium.Icon(color='red', icon='stop')
        ).add_to(m)

        # Simpan
        output_file = f'map_shape_{TARGET_TRIP_ID}.html'
        # m.save(output_file)
        # print(f"\n✅ Peta disimpan: {output_file}")
        # print(f"   Buka di browser untuk melihat rute")

✅ Trip  : S-ED-PGD-VEL-001
✅ Shape : shape_PGD_VEL
✅ Jumlah titik shape: 73


In [17]:
m

#### Cek shapes.txt untuk **`shape_VEL_PGD`** 

In [18]:
import pandas as pd
import folium

#File path
SHAPES_FILE = 'shapes.txt'
TRIPS_FILE  = 'trips.txt'

# Ambil salah satu trip dengan shape_VEL_PGD
TARGET_TRIP_ID = 'S-ED-VEL-PGD-001'

shapes = pd.read_csv(SHAPES_FILE)
trips  = pd.read_csv(TRIPS_FILE)

# Ambil shape_id dari trips.txt
trip_row = trips[trips['trip_id'] == TARGET_TRIP_ID]

if len(trip_row) == 0:
    print(f"❌ Trip {TARGET_TRIP_ID} tidak ditemukan di trips.txt")
else:
    shape_id = trip_row['shape_id'].values[0]
    print(f"✅ Trip  : {TARGET_TRIP_ID}")
    print(f"✅ Shape : {shape_id}")

    # Ambil titik shape
    shape_data = shapes[shapes['shape_id'] == shape_id].sort_values('shape_pt_sequence')

    if len(shape_data) == 0:
        print(f"❌ Shape {shape_id} tidak ditemukan di shapes.txt")
    else:
        print(f"✅ Jumlah titik shape: {len(shape_data)}")

        coords = list(zip(shape_data['shape_pt_lat'], shape_data['shape_pt_lon']))

        # Buat peta
        center_lat = shape_data['shape_pt_lat'].mean()
        center_lon = shape_data['shape_pt_lon'].mean()
        m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

        # Gambar jalur shape
        folium.PolyLine(
            coords,
            color='blue',
            weight=4,
            opacity=0.8,
            tooltip=f"Shape: {shape_id} | Trip: {TARGET_TRIP_ID}"
        ).add_to(m)

        # Titik awal shape
        folium.Marker(
            coords[0],
            popup=f"START – {shape_id}",
            icon=folium.Icon(color='green', icon='play')
        ).add_to(m)

        # Titik akhir shape
        folium.Marker(
            coords[-1],
            popup=f"END – {shape_id}",
            icon=folium.Icon(color='red', icon='stop')
        ).add_to(m)

        # Simpan
        output_file = f'map_shape_{TARGET_TRIP_ID}.html'
        # m.save(output_file)
        # print(f"\n✅ Peta disimpan: {output_file}")
        # print(f"   Buka di browser untuk melihat rute")

✅ Trip  : S-ED-VEL-PGD-001
✅ Shape : shape_VEL_PGD
✅ Jumlah titik shape: 73


In [19]:
m

#### Build `shape_PGD_VEL` & `shape_VEL_PGD` with haversine formula for `shape_dist_traveled`

In [20]:
"""
Build file shapes.txt menggunakan data geometri jalur rel LRT Jakarta yang diperoleh dari GeoJSON lokal atau melalui API Overpass (OpenStreetMap).

Shape yang dihasilkan:
  - shape_PGD_VEL : Pegangsaan Dua → Velodrome
  - shape_VEL_PGD : Velodrome → Pegangsaan Dua

Catatan:
  - File GeoJSON bisa didapatkan dari https://overpass-turbo.eu/ lakukan query untuk relation_id jalur tertentu kemudian download sebagai GeoJSON.
  - Terdapat proses trimming untuk memotong segmen pada stasiun terminus (awal & akhir) agar jalur rel tepat dimulai dari stop_id pertama.
  - Jika koordinat hasil parsing terbalik (arah OSM berlawanan dengan arah trip), aktifkan reverse=True pada konfigurasi RELATIONS.
  - shape_dist_traveled dihitung menggunakan rumus haversine dengan satuan Meter SETELAH proses reverse (jika aktif).

MODE OPERASI (otomatis dipilih berdasarkan file yang tersedia):
  1. GeoJSON lokal  → jika file .geojson tersedia (dari overpass-turbo.eu)
  2. Online         → fetch dari Overpass API
"""

import requests
import csv
import math
import json
import os
import time
from datetime import datetime

# File path
OUTPUT_SHAPES = "shapes.txt"
STOPS_FILE    = "stops.txt"

# Terminus stop_id per shape
# first_stop_id : stop pertama yang dilayani trip (platform)
# last_stop_id  : stop terakhir yang dilayani trip (platform)
# reverse       : True jika koordinat OSM berlawanan arah dengan trip
RELATIONS = [
    {
        "relation_id"  : 10693160, # relation_id OSM untuk jalur LRT Jakarta PGD-VEL
        "shape_id"     : "shape_PGD_VEL", # shape_id untuk jalur LRT Jakarta PGD-VEL
        "label"        : "PGD → VEL",
        "geojson_file" : "data/shapes/relation_10693160.geojson", # File GeoJSON untuk jalur LRT Jakarta PGD-VEL
        "first_stop_id": "GPGD01", # platform PGD arah VEL
        "last_stop_id" : "GVEL01", # platform VEL arah dari PGD
        "reverse"      : True,    # koordinat OSM terbalik
    },
    {
        "relation_id"  : 10693119, # relation_id OSM untuk jalur LRT Jakarta VEL-PGD
        "shape_id"     : "shape_VEL_PGD", # shape_id untuk jalur LRT Jakarta VEL-PGD
        "label"        : "VEL → PGD",
        "geojson_file" : "data/shapes/relation_10693119.geojson", # File GeoJSON untuk jalur LRT Jakarta VEL-PGD
        "first_stop_id": "GVEL02", # platform VEL arah PGD
        "last_stop_id" : "GPGD01", # platform PGD arah dari VEL
        "reverse"      : False,   # koordinat OSM sudah benar
    },
]

OVERPASS_MIRRORS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.osm.ch/api/interpreter",
    "https://overpass.openstreetmap.ru/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]

REQUEST_HEADERS = {
    "User-Agent"  : "GTFS-Builder/1.0 (LRT Jakarta GTFS)",
    "Accept"      : "application/json",
    "Content-Type": "application/x-www-form-urlencoded",
}

# Rumus Haversine 
def haversine_m(lat1, lon1, lat2, lon2):
    """Jarak antara dua titik koordinat dalam METER."""
    R  = 6371000.0
    p1 = math.radians(lat1)
    p2 = math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a  = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))


# Load stop koordinat 
def load_stop_coords(stops_file, stop_id):
    """
    Baca (lat, lon, stop_name) dari stops.txt berdasarkan stop_id.
    Raise ValueError jika stop_id tidak ditemukan.
    """
    with open(stops_file, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row["stop_id"] == stop_id:
                return (
                    float(row["stop_lat"]),
                    float(row["stop_lon"]),
                    row.get("stop_name", stop_id),
                )
    raise ValueError(f"stop_id '{stop_id}' tidak ditemukan di {stops_file}")


# Trim terminus 
def find_nearest_index(coords, target_lat, target_lon):
    """Cari indeks titik di coords yang paling dekat ke (target_lat, target_lon)."""
    best_idx  = 0
    best_dist = float("inf")
    for i, (lat, lon) in enumerate(coords):
        d = haversine_m(lat, lon, target_lat, target_lon)
        if d < best_dist:
            best_dist = d
            best_idx  = i
    return best_idx, best_dist


def trim_shape_to_terminus(coords, first_lat, first_lon, last_lat, last_lon):
    """
    Potong coords agar:
      - Dimulai dari titik terdekat ke stop pertama (first_stop)
      - Berakhir di titik terdekat ke stop terakhir (last_stop)

    Return: (coords_trimmed, idx_first, idx_last, dist_first_m, dist_last_m)
    """
    idx_first, dist_first = find_nearest_index(coords, first_lat, first_lon)
    idx_last,  dist_last  = find_nearest_index(coords, last_lat,  last_lon)

    if idx_first > idx_last:
        idx_first, idx_last = idx_last, idx_first

    return (
        coords[idx_first : idx_last + 1],
        idx_first, idx_last,
        dist_first, dist_last,
    )


# Reverse koordinat 
def reverse_coords(coords):
    """
    Balik urutan koordinat dan reset shape_pt_sequence dari 0.

    Digunakan jika koordinat hasil parsing OSM berlawanan arah dengan trip.
    Proses ini dilakukan SETELAH trim terminus dan SEBELUM hitung shape_dist_traveled
    agar:
      1. shape_pt_sequence tetap ascending dari 0
      2. shape_dist_traveled dihitung dari titik awal yang benar
      3. Titik pertama = platform terminus awal trip

    Contoh:
      Sebelum reverse: VEL(0) → ... → PGD(n)  ← arah OSM
      Setelah reverse: PGD(0) → ... → VEL(n)  ← arah trip PGD→VEL
    """
    return list(reversed(coords))


# GeoJSON Parser (mode offline) 
def extract_coords_from_geojson(geojson_path):
    """
    Parse GeoJSON dari overpass-turbo.eu → list of (lat, lon).

    GeoJSON dari overpass-turbo berisi FeatureCollection dengan:
      - LineString       : satu segmen way
      - MultiLineString  : beberapa segmen way
      - Point            : node (diabaikan)
    """
    print(f"  → Membaca: {geojson_path}")
    with open(geojson_path, "r", encoding="utf-8") as f:
        geojson = json.load(f)

    segments = []
    n_point  = 0

    for feature in geojson.get("features", []):
        geom  = feature.get("geometry", {})
        gtype = geom.get("type", "")

        if gtype == "LineString":
            segments.append([(c[1], c[0]) for c in geom["coordinates"]])
        elif gtype == "MultiLineString":
            for line in geom["coordinates"]:
                segments.append([(c[1], c[0]) for c in line])
        elif gtype == "Point":
            n_point += 1

    if not segments:
        raise ValueError("Tidak ada LineString/MultiLineString di GeoJSON!")

    print(f"  ✅ {len(segments)} segmen ditemukan, {n_point} point diabaikan")
    ordered = _chain_segments(segments)
    print(f"  ✅ Chain selesai → {len(ordered):,} titik koordinat")
    return ordered


def _chain_segments(segments):
    """
    Sambungkan list segmen menjadi satu jalur berurutan.
    Algoritma greedy berdasarkan jarak ujung-ke-ujung terdekat.
    Gap > 50m diberi warning tapi tetap disambung.
    """
    SNAP_THRESHOLD_M = 50.0
    remaining = [list(seg) for seg in segments]
    result    = remaining.pop(0)

    while remaining:
        tail      = result[-1]
        best_idx  = None
        best_dist = float("inf")
        best_rev  = False

        for i, seg in enumerate(remaining):
            d_head = haversine_m(tail[0], tail[1], seg[0][0],  seg[0][1])
            d_tail = haversine_m(tail[0], tail[1], seg[-1][0], seg[-1][1])

            if d_head < best_dist:
                best_dist, best_idx, best_rev = d_head, i, False
            if d_tail < best_dist:
                best_dist, best_idx, best_rev = d_tail, i, True

        seg = remaining.pop(best_idx)
        if best_rev:
            seg = list(reversed(seg))
        if best_dist > SNAP_THRESHOLD_M:
            print(f"  ⚠️  Gap {best_dist:.1f}m — tetap disambung")

        skip = 1 if haversine_m(tail[0], tail[1], seg[0][0], seg[0][1]) < 1.0 else 0
        result.extend(seg[skip:])

    return result


#  Overpass API OSM (mode online)
def try_request(mirror, query):
    """POST raw dulu, fallback ke GET params."""
    try:
        print(f"    [POST raw] ...")
        resp = requests.post(mirror, data=query, headers=REQUEST_HEADERS, timeout=90)
        resp.raise_for_status()
        return resp.json()
    except Exception as e_post:
        print(f"    ⚠️  POST gagal: {e_post}")

    try:
        print(f"    [GET params] ...")
        resp = requests.get(mirror, params={"data": query}, headers=REQUEST_HEADERS, timeout=90)
        resp.raise_for_status()
        return resp.json()
    except Exception as e_get:
        print(f"    ⚠️  GET gagal: {e_get}")
        raise e_get


def fetch_relation_online(relation_id):
    """Fetch relation dari Overpass API, coba semua mirror secara berurutan."""
    query = (
        f"[out:json][timeout:90];"
        f"relation({relation_id});"
        f"out body;>;"
        f"out skel qt;"
    )
    last_err = None
    for mirror in OVERPASS_MIRRORS:
        print(f"  → Mencoba {mirror} ...")
        try:
            data = try_request(mirror, query)
            print(f"  ✅ Berhasil! {len(data['elements'])} elements")
            return data
        except Exception as e:
            print(f"  ✗  Mirror gagal\n")
            last_err = e
            time.sleep(3)
    raise RuntimeError(f"Semua mirror gagal: {last_err}")


def extract_coords_from_overpass(data):
    """Parse Overpass JSON → list of (lat, lon)."""
    nodes = {e["id"]: (e["lat"], e["lon"])
             for e in data["elements"] if e["type"] == "node"}
    ways  = {e["id"]: e["nodes"]
             for e in data["elements"] if e["type"] == "way"}
    rel   = next((e for e in data["elements"] if e["type"] == "relation"), None)

    if not rel:
        raise ValueError("Relation tidak ditemukan!")

    tags        = rel.get("tags", {})
    way_members = [m for m in rel["members"] if m["type"] == "way"]
    print(f"\n  📋 Relation: {tags.get('name','N/A')} | {len(way_members)} ways")

    ordered_coords = []
    last_node_id   = None
    skipped        = 0

    for member in way_members:
        way_id = member["ref"]
        role   = member.get("role", "")

        if way_id not in ways or not ways[way_id]:
            skipped += 1
            continue

        way_nodes  = ways[way_id]
        first_node = way_nodes[0]
        last_node  = way_nodes[-1]

        if last_node_id is None:
            reverse = (role == "backward")
        else:
            if last_node_id == first_node:
                reverse = False
            elif last_node_id == last_node:
                reverse = True
            else:
                if first_node in nodes and last_node in nodes and last_node_id in nodes:
                    c       = nodes[last_node_id]
                    d_first = haversine_m(c[0], c[1], nodes[first_node][0], nodes[first_node][1])
                    d_last  = haversine_m(c[0], c[1], nodes[last_node][0],  nodes[last_node][1])
                    reverse = d_last < d_first
                else:
                    reverse = (role == "backward")

        seq = list(reversed(way_nodes)) if reverse else way_nodes
        for j, node_id in enumerate(seq):
            if node_id not in nodes:
                continue
            coord = nodes[node_id]
            if ordered_coords and j == 0 and ordered_coords[-1] == coord:
                continue
            ordered_coords.append(coord)
        last_node_id = seq[-1]

    if skipped:
        print(f"  ⚠️  {skipped} ways dilewati")

    return ordered_coords


# Selector source data 
def get_coords(rel_cfg):
    """
    Pilih sumber data secara otomatis:
      1. GeoJSON lokal → jika file .geojson ada
      2. Online        → fetch dari Overpass API
    """
    geojson_file = rel_cfg.get("geojson_file", "")
    if geojson_file and os.path.exists(geojson_file):
        print(f"  📂 Mode: GeoJSON lokal")
        return extract_coords_from_geojson(geojson_file)

    print(f"  🌐 Mode: Online Overpass API")
    data = fetch_relation_online(rel_cfg["relation_id"])
    return extract_coords_from_overpass(data)


# Output 
def write_shapes_txt(all_shape_data, output_file):
    """
    Tulis shapes.txt dengan kolom shape_dist_traveled.

    Urutan proses per shape:
      1. Trim terminus (sudah dilakukan sebelumnya di main)
      2. Reverse jika dikonfigurasi (sudah dilakukan sebelumnya di main)
      3. Reset shape_pt_sequence dari 0
      4. Hitung shape_dist_traveled kumulatif dari titik pertama (dist=0)

    Satuan: meter, presisi penuh (tidak dibulatkan).
    """
    fieldnames = [
        "shape_id", "shape_pt_lat", "shape_pt_lon",
        "shape_pt_sequence", "shape_dist_traveled",
    ]
    rows = []
    for shape_id, coords in all_shape_data.items():
        dist = 0.0
        prev = None
        for seq, (lat, lon) in enumerate(coords):
            if prev:
                dist += haversine_m(prev[0], prev[1], lat, lon)
            rows.append({
                "shape_id"           : shape_id,
                "shape_pt_lat"       : f"{lat:.7f}",
                "shape_pt_lon"       : f"{lon:.7f}",
                "shape_pt_sequence"  : seq,
                "shape_dist_traveled": f"{dist}",
            })
            prev = (lat, lon)

    with open(output_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    return rows


def save_json_ref(shape_id, relation_id, coords, total_m):
    """Simpan JSON referensi koordinat per shape."""
    json_path = f"shape_ref_{shape_id}.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump({
            "shape_id"     : shape_id,
            "relation_id"  : relation_id,
            "total_points" : len(coords),
            "total_dist_m" : round(total_m, 1),
            "total_dist_km": round(total_m / 1000, 3),
            "coords"       : [{"lat": lat, "lon": lon} for lat, lon in coords]
        }, f, indent=2)
    return json_path


# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    print("=" * 60)
    print(" OSM → GTFS shapes.txt  |  LRT Jakarta Lin Selatan Fase 1A")
    print(f" {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f" Satuan shape_dist_traveled : Meter (haversine)")
    print(f" Trim terminus              : aktif jika stops.txt tersedia")
    print(f" Reverse                    : per konfigurasi RELATIONS")
    print("=" * 60)
    print()
    print(" Prioritas sumber data:")
    print("   1. GeoJSON lokal (overpass-turbo.eu) — jika file ada")
    print("   2. Online Overpass API               — jika tidak ada")
    print()
    for rel in RELATIONS:
        ada     = os.path.exists(rel["geojson_file"])
        status  = "✅ ada → pakai GeoJSON" if ada else "⬜ tidak ada → fetch online"
        rev_str = "🔄 reverse=True" if rel.get("reverse") else "  reverse=False"
        print(f"   {rel['label']:<12} : {rel['geojson_file']:<40} {status}  |  {rev_str}")
    print()

    stops_available = os.path.exists(STOPS_FILE)
    if stops_available:
        print(f"📂 {STOPS_FILE} ditemukan — trim terminus aktif")
    else:
        print(f"⚠️  {STOPS_FILE} tidak ditemukan — trim terminus dinonaktifkan")
    print()

    all_shape_data = {}

    for rel_cfg in RELATIONS:
        shape_id   = rel_cfg["shape_id"]
        label      = rel_cfg["label"]
        do_reverse = rel_cfg.get("reverse", False)

        print(f"{'─'*55}")
        print(f"🔄  Memproses : {label}  {'(reverse aktif)' if do_reverse else ''}")
        print(f"{'─'*55}")

        try:
            coords      = get_coords(rel_cfg)
            n_raw       = len(coords)
            total_m_raw = sum(
                haversine_m(coords[i-1][0], coords[i-1][1],
                            coords[i][0],   coords[i][1])
                for i in range(1, len(coords))
            )

            # Step 1: Trim terminus 
            trimmed = False
            if (stops_available
                    and rel_cfg.get("first_stop_id")
                    and rel_cfg.get("last_stop_id")):
                try:
                    f_lat, f_lon, f_name = load_stop_coords(
                        STOPS_FILE, rel_cfg["first_stop_id"]
                    )
                    l_lat, l_lon, l_name = load_stop_coords(
                        STOPS_FILE, rel_cfg["last_stop_id"]
                    )
                    coords, idx_f, idx_l, d_f, d_l = trim_shape_to_terminus(
                        coords, f_lat, f_lon, l_lat, l_lon
                    )
                    trimmed = True

                    print(f"\n  ✂️  Trim terminus:")
                    print(f"     Stop pertama  : {rel_cfg['first_stop_id']} — {f_name}")
                    print(f"       idx={idx_f} | jarak ke shape: {d_f:.1f} m")
                    print(f"     Stop terakhir : {rel_cfg['last_stop_id']} — {l_name}")
                    print(f"       idx={idx_l} | jarak ke shape: {d_l:.1f} m")
                    print(f"     Titik dipotong: {idx_f} di awal, "
                          f"{n_raw - 1 - idx_l} di akhir")

                except ValueError as e:
                    print(f"  ⚠️  Trim gagal: {e} — pakai koordinat penuh")

            # Step 2: Reverse (setelah trim, sebelum hitung dist)
            if do_reverse:
                coords = reverse_coords(coords)
                print(f"\n  🔄  Reverse selesai:")
                print(f"     Titik pertama (baru) : {coords[0]}")
                print(f"     Titik terakhir (baru): {coords[-1]}")

            # Step 3: Hitung total jarak (setelah reverse) 
            if not coords:
                print(f"  ❌ Tidak ada koordinat!")
                continue

            all_shape_data[shape_id] = coords

            total_m = sum(
                haversine_m(coords[i-1][0], coords[i-1][1],
                            coords[i][0],   coords[i][1])
                for i in range(1, len(coords))
            )

            print(f"\n  📊 Hasil:")
            print(f"     Shape ID        : {shape_id}")
            print(f"     Titik raw       : {n_raw:,}")
            print(f"     Titik final     : {len(coords):,}  "
                  f"({'trim+reverse' if trimmed and do_reverse else 'trim' if trimmed else 'reverse' if do_reverse else 'tanpa trim/reverse'})")
            print(f"     Jarak raw       : {total_m_raw:,.1f} m  ({total_m_raw/1000:.3f} km)")
            print(f"     Jarak final     : {total_m:,.1f} m  ({total_m/1000:.3f} km)")
            print(f"     Titik pertama   : {coords[0]}")
            print(f"     Titik terakhir  : {coords[-1]}")

            json_path = save_json_ref(shape_id, rel_cfg["relation_id"], coords, total_m)
            print(f"     JSON referensi  : {json_path}")

        except Exception as e:
            print(f"  ❌ Error: {e}")
            import traceback; traceback.print_exc()

        time.sleep(1)

    if not all_shape_data:
        print("\n⚠️  Tidak ada shape yang berhasil. shapes.txt tidak dibuat.")
        return

    rows = write_shapes_txt(all_shape_data, OUTPUT_SHAPES)

    print(f"\n{'='*55}")
    print(f"✅ shapes.txt berhasil dibuat!")
    print(f"   Path    : {OUTPUT_SHAPES}")
    print(f"   Kolom   : shape_id, shape_pt_lat, shape_pt_lon,")
    print(f"             shape_pt_sequence, shape_dist_traveled")
    print(f"   Rows    : {len(rows):,}")
    print()
    print(f"   {'Shape ID':<22} {'Titik':>8}  {'Jarak (m)':>12}  {'Jarak (km)':>12}  {'Reverse':>8}")
    print(f"   {'─'*65}")
    for rel_cfg in RELATIONS:
        sid    = rel_cfg["shape_id"]
        coords = all_shape_data.get(sid)
        if not coords:
            continue
        total_m = sum(
            haversine_m(coords[i-1][0], coords[i-1][1],
                        coords[i][0],   coords[i][1])
            for i in range(1, len(coords))
        )
        rev_str = "✅" if rel_cfg.get("reverse") else "—"
        print(f"   {sid:<22} {len(coords):>8,}  "
              f"{total_m:>12,.1f}  {total_m/1000:>12.3f}  {rev_str:>8}")


if __name__ == "__main__":
    main()

 OSM → GTFS shapes.txt  |  LRT Jakarta Lin Selatan Fase 1A
 2026-04-26 21:34:18
 Satuan shape_dist_traveled : Meter (haversine)
 Trim terminus              : aktif jika stops.txt tersedia
 Reverse                    : per konfigurasi RELATIONS

 Prioritas sumber data:
   1. GeoJSON lokal (overpass-turbo.eu) — jika file ada
   2. Online Overpass API               — jika tidak ada

   PGD → VEL    : data/shapes/relation_10693160.geojson    ✅ ada → pakai GeoJSON  |  🔄 reverse=True
   VEL → PGD    : data/shapes/relation_10693119.geojson    ✅ ada → pakai GeoJSON  |    reverse=False

📂 stops.txt ditemukan — trim terminus aktif

───────────────────────────────────────────────────────
🔄  Memproses : PGD → VEL  (reverse aktif)
───────────────────────────────────────────────────────
  📂 Mode: GeoJSON lokal
  → Membaca: data/shapes/relation_10693160.geojson
  ✅ 1 segmen ditemukan, 6 point diabaikan
  ✅ Chain selesai → 74 titik koordinat

  ✂️  Trim terminus:
     Stop pertama  : GPGD01 — Stasiun 

#### Cek shapes.txt untuk **`shape_PGD_VEL`** 

In [21]:
import pandas as pd
import folium

#File path
SHAPES_FILE = 'shapes.txt'
TRIPS_FILE  = 'trips.txt'

# Ambil salah satu trip dengan shape_PGD_VEL
TARGET_TRIP_ID = 'S-ED-PGD-VEL-001'

shapes = pd.read_csv(SHAPES_FILE)
trips  = pd.read_csv(TRIPS_FILE)

# Ambil shape_id dari trips.txt
trip_row = trips[trips['trip_id'] == TARGET_TRIP_ID]

if len(trip_row) == 0:
    print(f"❌ Trip {TARGET_TRIP_ID} tidak ditemukan di trips.txt")
else:
    shape_id = trip_row['shape_id'].values[0]
    print(f"✅ Trip  : {TARGET_TRIP_ID}")
    print(f"✅ Shape : {shape_id}")

    # Ambil titik shape
    shape_data = shapes[shapes['shape_id'] == shape_id].sort_values('shape_pt_sequence')

    if len(shape_data) == 0:
        print(f"❌ Shape {shape_id} tidak ditemukan di shapes.txt")
    else:
        print(f"✅ Jumlah titik shape: {len(shape_data)}")

        coords = list(zip(shape_data['shape_pt_lat'], shape_data['shape_pt_lon']))

        # Buat peta
        center_lat = shape_data['shape_pt_lat'].mean()
        center_lon = shape_data['shape_pt_lon'].mean()
        m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

        # Gambar jalur shape
        folium.PolyLine(
            coords,
            color='blue',
            weight=4,
            opacity=0.8,
            tooltip=f"Shape: {shape_id} | Trip: {TARGET_TRIP_ID}"
        ).add_to(m)

        # Titik awal shape
        folium.Marker(
            coords[0],
            popup=f"START – {shape_id}",
            icon=folium.Icon(color='green', icon='play')
        ).add_to(m)

        # Titik akhir shape
        folium.Marker(
            coords[-1],
            popup=f"END – {shape_id}",
            icon=folium.Icon(color='red', icon='stop')
        ).add_to(m)

        # Simpan
        output_file = f'map_shape_{TARGET_TRIP_ID}.html'
        # m.save(output_file)
        # print(f"\n✅ Peta disimpan: {output_file}")
        # print(f"   Buka di browser untuk melihat rute")

✅ Trip  : S-ED-PGD-VEL-001
✅ Shape : shape_PGD_VEL
✅ Jumlah titik shape: 73


In [22]:
m

#### Cek shapes.txt untuk **`shape_VEL_PGD`** 

In [23]:
import pandas as pd
import folium

#File path
SHAPES_FILE = 'shapes.txt'
TRIPS_FILE  = 'trips.txt'

# Ambil salah satu trip dengan shape_VEL_PGD
TARGET_TRIP_ID = 'S-ED-VEL-PGD-001'

shapes = pd.read_csv(SHAPES_FILE)
trips  = pd.read_csv(TRIPS_FILE)

# Ambil shape_id dari trips.txt
trip_row = trips[trips['trip_id'] == TARGET_TRIP_ID]

if len(trip_row) == 0:
    print(f"❌ Trip {TARGET_TRIP_ID} tidak ditemukan di trips.txt")
else:
    shape_id = trip_row['shape_id'].values[0]
    print(f"✅ Trip  : {TARGET_TRIP_ID}")
    print(f"✅ Shape : {shape_id}")

    # Ambil titik shape
    shape_data = shapes[shapes['shape_id'] == shape_id].sort_values('shape_pt_sequence')

    if len(shape_data) == 0:
        print(f"❌ Shape {shape_id} tidak ditemukan di shapes.txt")
    else:
        print(f"✅ Jumlah titik shape: {len(shape_data)}")

        coords = list(zip(shape_data['shape_pt_lat'], shape_data['shape_pt_lon']))

        # Buat peta
        center_lat = shape_data['shape_pt_lat'].mean()
        center_lon = shape_data['shape_pt_lon'].mean()
        m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

        # Gambar jalur shape
        folium.PolyLine(
            coords,
            color='blue',
            weight=4,
            opacity=0.8,
            tooltip=f"Shape: {shape_id} | Trip: {TARGET_TRIP_ID}"
        ).add_to(m)

        # Titik awal shape
        folium.Marker(
            coords[0],
            popup=f"START – {shape_id}",
            icon=folium.Icon(color='green', icon='play')
        ).add_to(m)

        # Titik akhir shape
        folium.Marker(
            coords[-1],
            popup=f"END – {shape_id}",
            icon=folium.Icon(color='red', icon='stop')
        ).add_to(m)

        # Simpan
        output_file = f'map_shape_{TARGET_TRIP_ID}.html'
        # m.save(output_file)
        # print(f"\n✅ Peta disimpan: {output_file}")
        # print(f"   Buka di browser untuk melihat rute")

✅ Trip  : S-ED-VEL-PGD-001
✅ Shape : shape_VEL_PGD
✅ Jumlah titik shape: 73


In [24]:
m

## stop_times.txt (Required)

File `stop_times.txt` mendefinisikan **waktu kedatangan dan keberangkatan moda transportasi di setiap stasiun/stasiun/stops untuk setiap perjalanan (trip)**. File ini adalah file terbesar dan terpenting dalam GTFS karena menghubungkan tiga entitas utama yaitu trip, stop, dan waktu menjadi jadwal yang lengkap. Setiap baris mewakili satu pemberhentian dalam satu perjalanan.

---

**Field yang Direferensikan di File Lain**

| Field | Direferensikan dari | Keterangan |
|---|---|---|
| `trip_id` | `trips.txt` | Setiap baris merujuk ke perjalanan tertentu |
| `stop_id` | `stops.txt` | Hanya boleh merujuk ke `location_type=0` (platform) |
| `shape_dist_traveled` | `shapes.txt` | Satuan harus konsisten dengan `shapes.shape_dist_traveled` |

---

**Skema kolom stop_times.txt**

| Field | Tipe Data | Status | Deskripsi |
|---|---|---|---|
| `trip_id` | Foreign ID → `trips.trip_id` | **Wajib** | ID perjalanan yang dilayani pada baris ini. |
| `arrival_time` | Time | **Kondisional** | Waktu tiba di stasiun dalam format `HH:MM:SS`. Untuk waktu melewati tengah malam gunakan jam lebih dari 24 — contoh `25:35:00` untuk 01:35 dini hari. **Wajib** untuk stasiun pertama dan terakhir dalam trip, serta jika `timepoint=1`. |
| `departure_time` | Time | **Kondisional** | Waktu berangkat dari stasiun dalam format `HH:MM:SS`. Aturan sama dengan `arrival_time`. Jika waktu tiba dan berangkat sama, isi kedua field dengan nilai yang sama. |
| `stop_id` | Foreign ID → `stops.stop_id` | **Kondisional** | ID stasiun yang dilayani. Harus merujuk ke `location_type=0` (platform). **Wajib** jika `location_group_id` dan `location_id` tidak didefinisikan. |
| `location_group_id` | Foreign ID → `location_groups` | **Kondisional** | ID grup lokasi untuk layanan on-demand. **Dilarang** jika `stop_id` atau `location_id` sudah didefinisikan. |
| `location_id` | Foreign ID → `locations.geojson` | **Kondisional** | ID lokasi GeoJSON untuk layanan on-demand. **Dilarang** jika `stop_id` atau `location_group_id` sudah didefinisikan. |
| `stop_sequence` | Non-negative integer | **Wajib** | Urutan stasiun dalam perjalanan. Nilai harus selalu meningkat sepanjang trip namun tidak harus berurutan berurutan. |
| `stop_headsign` | Text | Opsional | Teks tujuan yang ditampilkan di papan kendaraan khusus untuk stasiun ini — menggantikan `trips.trip_headsign` jika berbeda di tengah perjalanan. |
| `pickup_type` | Enum | **Kondisional** | Metode naik penumpang. Lihat tabel nilai enum di bawah. |
| `drop_off_type` | Enum | **Kondisional** | Metode turun penumpang. Lihat tabel nilai enum di bawah. |
| `continuous_pickup` | Enum | **Kondisional** | Apakah penumpang bisa naik di sembarang titik dari stasiun ini ke stasiun berikutnya. Jika diisi, menggantikan nilai di `routes.txt`. |
| `continuous_drop_off` | Enum | **Kondisional** | Apakah penumpang bisa turun di sembarang titik dari stasiun ini ke stasiun berikutnya. Jika diisi, menggantikan nilai di `routes.txt`. |
| `shape_dist_traveled` | Non-negative float | Opsional | Jarak aktual yang ditempuh dari stasiun pertama hingga stasiun ini, dalam satuan yang sama dengan `shapes.txt`. Nilai harus selalu meningkat sesuai `stop_sequence`. |
| `timepoint` | Enum | Opsional | Tingkat keakuratan waktu. `1` = waktu pasti, `0` = waktu perkiraan/interpolasi. Jika tidak diisi semua waktu dianggap pasti. |
| `pickup_booking_rule_id` | Foreign ID → `booking_rules` | Opsional | ID aturan pemesanan untuk naik penumpang. Disarankan jika `pickup_type=2`. |
| `drop_off_booking_rule_id` | Foreign ID → `booking_rules` | Opsional | ID aturan pemesanan untuk turun penumpang. Disarankan jika `drop_off_type=2`. |

---

**`pickup_type`** & **`drop_off_type`**

| Nilai | Arti |
|:---:|---|
| `0` atau kosong | Naik/turun terjadwal reguler | 
| `1` | Tidak tersedia |
| `2` | Perlu telepon operator |
| `3` | Perlu koordinasi dengan pengemudi | 

**`timepoint`**

| Nilai | Arti |
|:---:|---|
| `1` atau kosong | Waktu **pasti** | 
| `0` | Waktu **perkiraan** / interpolasi |

### Panduan Pengisian `stop_times.txt` — LRT Jakarta Lin Selatan Fase 1A

---

**`trip_id`**

Diisi sesuai dengan `trip_id` yang ada di `trips.txt`. Terdiri dari 2 jenis trip yaitu:

- `S-ED-PGD-VEL-xxx`
- `S-ED-VEL-PGD-xxx`

---

**`stop_id`**

Diisi dengan `stop_id` platform (`location_type=0`) dari setiap stasiun pemberhentian. LRT Jakarta memiliki dua arah perjalanan dan 2 jenis trip dengan mapping platform sebagai berikut:

**Full trip PGD → VEL** (`S-ED-PGD-VEL-xxx`)

| Stasiun | `stop_id` |
|---|---|
| PGD | `GPGD01` |
| BVU | `GBVU01` |
| BVS | `GBVS01` |
| PUM | `GPUM01` |
| EQS | `GEQS01` |
| VEL | `GVEL01` |

**Full trip VEL → PGD** (`S-ED-VEL-PGD-xxx`)

| Stasiun | `stop_id` |
|---|---|
| VEL | `GVEL02` |
| EQS | `GEQS02` |
| PUM | `GPUM02` |
| BVS | `GBVS02` |
| BVU | `GBVU02` |
| PGD | `GPGD01` |

---

**`arrival_time`** & **`departure_time`**

Referensi jadwal diambil dari website resmi [LRT Jakarta](https://www.lrtjakarta.co.id/jadwal.html) yang hanya menyediakan `departure_time`. Dwell time aktual LRT Jakarta adalah **30 detik**, sehingga:

| Kondisi | `arrival_time` |
|---|---|
| Stasiun pertama (`stop_sequence=0`) | `= departure_time` |
| Stasiun tengah | `= departure_time - 30 detik` |
| Stasiun terakhir | `= departure_time` |

Format waktu mengikuti spesifikasi GTFS untuk waktu melewati tengah malam jam tetap bertambah dan tidak direset ke `00`, contoh `25:05:00` untuk pukul 01:05 dini hari.

---

**`stop_sequence`**

Diisi secara berurutan dimulai dari `0` untuk setiap trip, mengikuti urutan stasiun sesuai 2 jenis trip yang sudah didefinisikan di atas.

| Stop | `stop_sequence` |
|---|:---:|
| Stasiun pertama | `0` |
| Stasiun kedua | `1` |
| Stasiun ketiga | `2` |
| ... | ... |

---

**`pickup_type`** & **`drop_off_type`**

Diisi `0` untuk semua baris naik dan turun penumpang hanya di stasiun resmi sesuai jadwal.

---

**`continuous_pickup`** & **`continuous_drop_off`**

Dikosongkan nilai default `1` berlaku otomatis, artinya penumpang tidak bisa naik maupun turun di sembarang titik sepanjang jalur, hanya di stasiun resmi.

---

**`timepoint`**

Diisi `1` untuk semua baris karena LRT Jakarta memiliki ketepatan waktu presisi dengan jadwal yang bersifat pasti bukan perkiraan.

---

**`stop_headsign`**

Diisi berdasarkan arah perjalanan trip:

| Arah | `stop_headsign` |
|---|---|
| Ke Selatan (PGD → VEL) | `Velodrome` |
| Ke Utara (VEL → PGD) | `Pegangsaan Dua` |

---

**`shape_dist_traveled`**

`shape_dist_traveled` merepresentasikan **jarak kumulatif aktual** yang ditempuh kendaraan dari titik awal shape hingga suatu `stop_id`.

Nilai ini dihitung secara **berurutan mengikuti stop_sequence**, sehingga setiap baris menunjukkan posisi relatif stop terhadap keseluruhan jalur. Nilai ini dihitung berdasarkan jarak antar `stop_id` ke `stop_id` yang berurutan.

**Pendekatan Perhitungan**

Terdapat **dua pendekatan umum** dalam menghitung `shape_dist_traveled`:

**1. Menggunakan Rumus Haversine (Manual)**

Pendekatan ini menghitung jarak antar dua titik koordinat di permukaan bumi berdasarkan model bola bumi.

```python
def haversine_m(lat1, lon1, lat2, lon2):
    """Jarak antara dua titik koordinat dalam meter (Haversine)."""
    R = 6371000.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))
```

Perhitungan dilakukan secara **kumulatif**:

* Stop 0 → `0`
* Stop 1 → jarak(0 → 1)
* Stop 2 → jarak(0 → 1) + jarak(1 → 2)
* dan seterusnya

**Kelebihan:**

* Sederhana dan mudah diimplementasikan (pandas / numpy)
* Tidak memerlukan library GIS
* Lebih stabil terhadap perbedaan presisi (minim mismatch dengan `shapes.txt`)


**2. Menggunakan `gtfs_kit.append_dist_to_stop_times()`**

Fungsi dari library `gtfs_kit` menghitung `shape_dist_traveled` dengan pendekatan spasial yang lebih akurat:

**Proses:**

* Koordinat diubah dari **WGS84 (lat/lon)** ke sistem proyeksi berbasis meter (UTM)
* Dibentuk:

  * LineString untuk shape
  * Point untuk setiap stop
    
* Jarak dihitung menggunakan:

  * `LineString.project(Point)`
  * menghasilkan posisi jarak kumulatif sepanjang shape

**Validasi hasil:**

* Jika jarak monotonik meningkat → digunakan langsung
* Jika tidak valid → diperbaiki dengan interpolasi berbasis waktu (fallback)

**Catatan & Kekurangan**

Pendekatan `gtfs_kit` dapat menghasilkan **perbedaan sangat kecil (floating point precision)** antara:

* `shape_dist_traveled` maksimum di `shapes.txt`
* `shape_dist_traveled` pada stop terakhir di `stop_times.txt`

Contoh:

```
14609.29264816948   (stop_times)
14609.292648169478  (shapes)
```

Walaupun selisihnya sangat kecil, hal ini dapat memicu warning:

**`trip_distance_exceeds_shape_distance_below_threshold`**

`Jarak antara titik terakhir shape dan stop terakhir lebih dari 0 tetapi kurang dari threshold (±11.1 meter)`

**Solusi yang Disarankan**

Untuk menghindari warning tersebut, lakukan **penyesuaian (force snap)**:

* Set nilai `shape_dist_traveled` pada **stop terakhir**
* Menjadi **persis sama** dengan nilai maksimum pada `shapes.txt`

Pendekatan ini memastikan:

* Konsistensi antar file GTFS
* Tidak ada mismatch akibat presisi floating point
* Validator GTFS tidak menghasilkan warning


**Catatan**

* Kedua metode diatas valid dalam GTFS
* Hasil bisa sedikit berbeda (biasanya sangat kecil)
* Yang penting:
  * nilai **monoton meningkat**
  * konsisten satuan (meter/km)

### Build `stop_times.txt` with gtfs_kit for shape_dist_traveled

In [25]:
"""
Build stop_times.txt berdasarkan jadwal keberangkatan LRT Jakarta

departure_time  : langsung dari file jadwal keberangkatan
arrival_time    : departure_time - DWELL_TIME (detik), kecuali stasiun pertama & terakhir: arrival = departure

Format waktu arrival_time & departure_time pada stop_times.txt:
  - Waktu TIDAK direset ke 00:xx:xx saat melewati tengah malam
  - Melainkan terus bertambah: 24:xx:xx, 25:xx:xx, dst
  - Contoh: kereta berangkat 23:40 tiba 00:01 → ditulis 24:01:xx

Catatan:
  - shape_dist_traveled tidak dihitung di sini, akan dihitung menggunakan library gtfs_kit
  - Perhitungan shape_dist_traveled menggunakan gtfs_kit.append_dist_to_stop_times(feed) setelah file ini dibuat
  - Nilai shape_dist_traveled pada stop_times.txt perlu disesuaikan dengan shapes.txt, khususnya pada stop terakhir setiap trip agar sama dengan panjang 
    total shape, sehingga menghindari selisih kecil (floating point) dan warning pada validator GTFS (trip_distance_exceeds_shape_distance_below_threshold)
"""

import pandas as pd
import csv
import os
from datetime import datetime

# File path
OUTPUT_FILE = "stop_times.txt"

# Dwell time dalam detik
# arrival = departure - DWELL_TIME
# kecuali stasiun pertama & terakhir: arrival = departure
DWELL_TIME  = 30   # detik

# File jadwal keberangkatan lRT Jakarta
SCHEDULE_FILES = {
    "PGD_VEL_ED": "data/jadwal-keberangkatan/PegangsaanDua_Velodrome.csv",
    "VEL_PGD_ED": "data/jadwal-keberangkatan/Velodrome_PegangsaanDua.csv",
}

# Mapping kode stasiun → stop_id platform
PLATFORM_MAP = {
    "PGD_VEL": {
        "PGD": "GPGD01", "BVU": "GBVU01", "BVS": "GBVS01",
        "PUM": "GPUM01", "EQS": "GEQS01", "VEL": "GVEL01",
    },
    "VEL_PGD": {
        "VEL": "GVEL02", "EQS": "GEQS02", "PUM": "GPUM02",
        "BVS": "GBVS02", "BVU": "GBVU02", "PGD": "GPGD01",
    },
}


# Time utils
def parse_time(time_str):
    """
    Parse waktu HH:MM:SS ke total detik.
    Support jam > 24 untuk layanan melewati tengah malam (GTFS compliant).

    Contoh:
        "05:12:00" →  18720
        "23:40:30" →  85230
        "24:00:50" →  86450
    """
    parts = str(time_str).strip().split(":")
    h = int(parts[0])
    m = int(parts[1])
    s = int(parts[2]) if len(parts) > 2 else 0
    return h * 3600 + m * 60 + s


def format_time(total_seconds):
    """
    Format total detik ke HH:MM:SS sesuai spesifikasi GTFS.
    Jam TIDAK direset ke 00 saat melewati tengah malam melainkan terus bertambah (24:xx:xx, 25:xx:xx, dst).
    Contoh:
        85230 → "23:40:30"
        86450 → "24:00:50"  ✅ bukan "00:00:50"
        90000 → "25:00:00"
    """
    h = total_seconds // 3600
    m = (total_seconds % 3600) // 60
    s = total_seconds % 60
    return f"{h:02d}:{m:02d}:{s:02d}"


def fix_midnight_rollover(current_sec, prev_sec):
    """
    Koreksi midnight rollover sesuai spesifikasi GTFS.

    GTFS mengharuskan waktu selalu ASCENDING dalam satu trip.
    Jika current_sec < prev_sec, berarti CSV menyimpan waktu sebagai
    00:xx:xx setelah tengah malam — dikoreksi dengan +86400 detik.

    Contoh:
        prev    = 23:40:30 → 85230 detik
        current = 00:00:50 →    50 detik  ← lebih kecil → rollover!
        koreksi = 50 + 86400 = 86450      → "24:00:50"
    """
    while current_sec < prev_sec:
        current_sec += 86400
    return current_sec


def calc_arrival(departure_sec, stop_sequence, total_stops):
    """
    Hitung arrival_time dalam detik.

    Aturan:
      - Stasiun pertama  (seq=0)        : arrival = departure
      - Stasiun terakhir (seq=total-1)  : arrival = departure
      - Stasiun lainnya                 : arrival = departure - DWELL_TIME
    """
    if stop_sequence == 0 or stop_sequence == total_stops - 1:
        return departure_sec
    return departure_sec - DWELL_TIME


# trip
def get_trip_info(trip_id):
    """
    Tentukan platform_dir dan stop_headsign berdasarkan trip_id.
    Return: (platform_dir, stop_headsign)
    """
    if "PGD-VEL" in trip_id:
        return "PGD_VEL", "Velodrome"
    elif "VEL-PGD" in trip_id:
        return "VEL_PGD", "Pegangsaan Dua"
    else:
        raise ValueError(f"Tidak bisa mendeteksi jenis trip dari: {trip_id}")


# Generate stop_times
def generate_stop_times(schedule_df, platform_dir):
    """
    Generate baris stop_times dari satu DataFrame jadwal.

    Proses per trip:
      1. Iterasi setiap stasiun secara berurutan
      2. Parse departure_time dari file jadwal keberangkatan LRT Jakarta
      3. Koreksi midnight rollover jika perlu (00:xx → 24:xx)
      4. Hitung arrival_time berdasarkan DWELL_TIME

    Return: (list of dict, int rollover_count)
    """
    rows                    = []
    plat_map                = PLATFORM_MAP[platform_dir]
    station_cols            = [c for c in schedule_df.columns
                               if c not in ("trip_id", "Full Trip")]
    midnight_rollover_count = 0

    for _, trip_row in schedule_df.iterrows():
        trip_id                  = trip_row["trip_id"]
        _, stop_headsign         = get_trip_info(trip_id)

        # Filter stasiun yang ada jadwalnya (tidak kosong / NaN)
        active_stations = [
            s for s in station_cols
            if pd.notna(trip_row[s]) and str(trip_row[s]).strip() != ""
        ]
        total_stops  = len(active_stations)
        prev_dep_sec = -1   # sentinel: belum ada stop sebelumnya

        for seq, stn in enumerate(active_stations):
            raw_dep_sec = parse_time(trip_row[stn])

            # Koreksi midnight rollover 
            if prev_dep_sec >= 0:
                corrected_dep_sec = fix_midnight_rollover(raw_dep_sec, prev_dep_sec)
                if corrected_dep_sec != raw_dep_sec:
                    midnight_rollover_count += 1
            else:
                corrected_dep_sec = raw_dep_sec

            arr_sec = calc_arrival(corrected_dep_sec, seq, total_stops)

            rows.append({
                "trip_id"             : trip_id, #id trip berdasarkan trips.txt
                "arrival_time"        : format_time(arr_sec), #waktu kedatangan berdasarkan 30 detik sebelum keberangkatan
                "departure_time"      : format_time(corrected_dep_sec), #waktu keberangkatan berdasarkan jadwal
                "stop_id"             : plat_map.get(stn, ""), #id setiap stasiun (platform) per trip
                "stop_sequence"       : seq, #urutan stasiun per trip
                "stop_headsign"       : stop_headsign, #sesuai arah trip
                "pickup_type"         : 0, #naik/turun di stasiun sesuai jadwal trip
                "drop_off_type"       : 0, #naik/turun di stasiun sesuai jadwal trip
                "continuous_pickup"   : 1, #naik/turun di stasiun resmi
                "continuous_drop_off" : 1, #naik/turun di stasiun resmi
                "timepoint"           : 1, #tepat Waktu (1)
            })

            prev_dep_sec = corrected_dep_sec

    return rows, midnight_rollover_count


# Validasi ascending per trip
def validate_ascending(df_stop_times):
    """
    Validasi bahwa departure_time selalu ascending dalam setiap trip.
    Return: list of dict issue (kosong = semua OK)
    """
    issues = []
    for trip_id, group in df_stop_times.groupby("trip_id"):
        group    = group.sort_values("stop_sequence")
        dep_secs = group["departure_time"].apply(parse_time).tolist()
        for i in range(1, len(dep_secs)):
            if dep_secs[i] < dep_secs[i - 1]:
                issues.append({
                    "trip_id"   : trip_id,
                    "stop_seq"  : group.iloc[i]["stop_sequence"],
                    "stop_id"   : group.iloc[i]["stop_id"],
                    "prev_time" : group.iloc[i - 1]["departure_time"],
                    "curr_time" : group.iloc[i]["departure_time"],
                })
    return issues

def main():
    print("=" * 60)
    print(" Build stop_times.txt  |  LRT Jakarta Lin Selatan Fase 1A")
    print(f" {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f" Dwell time   : {DWELL_TIME} detik")
    print(f" Format waktu : HH:MM:SS (>24 jam untuk lewat tengah malam)")
    print(f" shape_dist   : tidak dihitung — gunakan gtfs_kit setelahnya")
    print("=" * 60)

    # Proses setiap file jadwal
    all_rows             = []
    total_rollover_count = 0
    seen_trip_ids        = set()

    for key, filename in SCHEDULE_FILES.items():
        if not os.path.exists(filename):
            print(f"\n⚠️  File tidak ditemukan: {filename} — dilewati")
            continue

        platform_dir = "PGD_VEL" if key.startswith("PGD") else "VEL_PGD"

        print(f"\n{'─'*55}")
        print(f"📋 Memproses : {filename}")
        print(f"   Arah      : {platform_dir}")

        df = pd.read_csv(filename)
        if "Full Trip" in df.columns:
            df = df.drop(columns=["Full Trip"])

        # Guard duplikat trip_id antar file
        incoming_ids  = set(df["trip_id"].unique())
        duplicate_ids = incoming_ids & seen_trip_ids
        if duplicate_ids:
            print(f"   ⚠️  {len(duplicate_ids)} trip_id duplikat dilewati:")
            for tid in sorted(duplicate_ids)[:5]:
                print(f"      {tid}")
            df = df[~df["trip_id"].isin(duplicate_ids)]
        seen_trip_ids.update(set(df["trip_id"].unique()))

        rows, rollover_count = generate_stop_times(df, platform_dir)
        all_rows.extend(rows)
        total_rollover_count += rollover_count

        print(f"   Trip              : {df['trip_id'].nunique()}")
        print(f"   Baris             : {len(rows)}")
        print(f"   Midnight rollover : {rollover_count} koreksi")

    # Save stop_times.txt
    if not all_rows:
        print("\n⚠️  Tidak ada baris yang dihasilkan.")
        return

    fieldnames = [
        "trip_id", "arrival_time", "departure_time",
        "stop_id", "stop_sequence", "stop_headsign",
        "pickup_type", "drop_off_type",
        "continuous_pickup", "continuous_drop_off", "timepoint",
    ]

    df_out = pd.DataFrame(all_rows, columns=fieldnames)
    df_out.to_csv(
        OUTPUT_FILE,
        index=False,
        encoding="utf-8",
        lineterminator="\n",
        quoting=csv.QUOTE_MINIMAL,
    )

    # Validasi ascending
    print(f"\n{'─'*55}")
    print("🔍 Validasi ascending time per trip ...")
    issues = validate_ascending(df_out)
    if issues:
        print(f"   ⚠️  {len(issues)} waktu tidak ascending:")
        for iss in issues[:10]:
            print(f"      trip={iss['trip_id']}  seq={iss['stop_seq']}"
                  f"  stop={iss['stop_id']}"
                  f"  {iss['prev_time']} → {iss['curr_time']}")
        if len(issues) > 10:
            print(f"      ... dan {len(issues) - 10} lainnya")
    else:
        print("   ✅ Semua trip ascending — tidak ada masalah waktu!")

    print(f"\n{'='*55}")
    print(f"✅ stop_times.txt berhasil dibuat!")
    print(f"   Path              : {OUTPUT_FILE}")
    print(f"   Total baris       : {len(df_out):,}")
    print(f"   Total trip        : {df_out['trip_id'].nunique():,}")
    print(f"   Midnight rollover : {total_rollover_count} koreksi total")

    print(f"\n📄 Preview 5 baris pertama:")
    print(df_out.head().to_string(index=False))

    print(f"\n📄 Preview 5 baris terakhir (cek format >24 jam):")
    print(df_out.tail().to_string(index=False))


if __name__ == "__main__":
    main()

 Build stop_times.txt  |  LRT Jakarta Lin Selatan Fase 1A
 2026-04-26 21:34:20
 Dwell time   : 30 detik
 Format waktu : HH:MM:SS (>24 jam untuk lewat tengah malam)
 shape_dist   : tidak dihitung — gunakan gtfs_kit setelahnya

───────────────────────────────────────────────────────
📋 Memproses : data/jadwal-keberangkatan/PegangsaanDua_Velodrome.csv
   Arah      : PGD_VEL
   Trip              : 102
   Baris             : 612
   Midnight rollover : 0 koreksi

───────────────────────────────────────────────────────
📋 Memproses : data/jadwal-keberangkatan/Velodrome_PegangsaanDua.csv
   Arah      : VEL_PGD
   Trip              : 102
   Baris             : 612
   Midnight rollover : 0 koreksi

───────────────────────────────────────────────────────
🔍 Validasi ascending time per trip ...
   ✅ Semua trip ascending — tidak ada masalah waktu!

✅ stop_times.txt berhasil dibuat!
   Path              : stop_times.txt
   Total baris       : 1,224
   Total trip        : 204
   Midnight rollover : 0 ko

In [26]:
# Build field shape_dist_traveled with gtfs_kit (gtfs_kit.append_dist_to_stop_time(feed))

import gtfs_kit as gk
import pandas as pd
import numpy as np

feed = gk.read_feed(".", dist_units="m")
feed = gk.append_dist_to_stop_times(feed)

shape_max = (
    feed.shapes
    .groupby("shape_id")["shape_dist_traveled"]
    .max()
    .to_dict()
)

# Lookup: trip_id → shape_id
trip_to_shape = feed.trips.set_index("trip_id")["shape_id"].to_dict()

# Force snap: stop terakhir = max shape dist (nilai exact)
st = feed.stop_times.copy()

# Indeks baris stop terakhir per trip
last_idx = st.groupby("trip_id")["stop_sequence"].idxmax()

snapped = 0
for trip_id, idx in last_idx.items():
    shape_id = trip_to_shape.get(trip_id)
    if not shape_id or shape_id not in shape_max:
        continue
    exact_val                      = shape_max[shape_id]
    st.at[idx, "shape_dist_traveled"] = exact_val
    snapped += 1

feed.stop_times = st

print(f"✅ Force snap selesai: {snapped} trip dikoreksi")

# Verifikasi: pastikan selisih = 0
print("\n=== Verifikasi (5 trip pertama) ===")
issues = 0
for trip_id, idx in list(last_idx.items())[:5]:
    shape_id   = trip_to_shape.get(trip_id)
    val_stop   = st.at[idx, "shape_dist_traveled"]
    val_shape  = shape_max.get(shape_id)
    match      = val_stop == val_shape   # harus True (identik bit-for-bit)
    status     = "✅" if match else "❌"
    issues    += 0 if match else 1
    print(f"  {status} {trip_id}")
    print(f"     stop_times  : {val_stop:.15f}")
    print(f"     shapes      : {val_shape:.15f}")
    print(f"     selisih     : {abs(val_stop - val_shape):.2e}")

if issues == 0:
    print("\n✅ Semua stop terakhir identik dengan shapes")
else:
    print(f"\n⚠️  {issues} trip masih ada selisih")

# Save
feed.stop_times.to_csv(
    "stop_times.txt",
    index=False,
    encoding='utf-8',        
    lineterminator='\n',         
    quoting=csv.QUOTE_MINIMAL   
)
feed.stop_times

✅ Force snap selesai: 204 trip dikoreksi

=== Verifikasi (5 trip pertama) ===
  ✅ S-ED-PGD-VEL-001
     stop_times  : 5593.213483889900999
     shapes      : 5593.213483889900999
     selisih     : 0.00e+00
  ✅ S-ED-PGD-VEL-002
     stop_times  : 5593.213483889900999
     shapes      : 5593.213483889900999
     selisih     : 0.00e+00
  ✅ S-ED-PGD-VEL-003
     stop_times  : 5593.213483889900999
     shapes      : 5593.213483889900999
     selisih     : 0.00e+00
  ✅ S-ED-PGD-VEL-004
     stop_times  : 5593.213483889900999
     shapes      : 5593.213483889900999
     selisih     : 0.00e+00
  ✅ S-ED-PGD-VEL-005
     stop_times  : 5593.213483889900999
     shapes      : 5593.213483889900999
     selisih     : 0.00e+00

✅ Semua stop terakhir identik dengan shapes


,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,continuous_pickup,continuous_drop_off,timepoint,shape_dist_traveled
0,S-ED-PGD-VEL-001,05:30:00,05:30:00,GPGD01,0,Velodrome,0,0,1,1,1,0.000000
1,S-ED-PGD-VEL-001,05:33:30,05:34:00,GBVU01,1,Velodrome,0,0,1,1,1,1473.368196
2,S-ED-PGD-VEL-001,05:36:30,05:37:00,GBVS01,2,Velodrome,0,0,1,1,1,2692.496575
3,S-ED-PGD-VEL-001,05:38:30,05:39:00,GPUM01,3,Velodrome,0,0,1,1,1,3887.548857
4,S-ED-PGD-VEL-001,05:40:30,05:41:00,GEQS01,4,Velodrome,0,0,1,1,1,4681.129606
...,...,...,...,...,...,...,...,...,...,...,...,...
1219,S-ED-VEL-PGD-102,22:41:30,22:42:00,GEQS02,1,Pegangsaan Dua,0,0,1,1,1,893.288449
1220,S-ED-VEL-PGD-102,22:43:30,22:44:00,GPUM02,2,Pegangsaan Dua,0,0,1,1,1,1683.787443
1221,S-ED-VEL-PGD-102,22:45:30,22:46:00,GBVS02,3,Pegangsaan Dua,0,0,1,1,1,2878.246572
1222,S-ED-VEL-PGD-102,22:48:30,22:49:00,GBVU02,4,Pegangsaan Dua,0,0,1,1,1,4097.370048


### Validate shape_dist_traveled Consistency (Stop Times vs Shapes) result gtfs_kit

In [27]:
import pandas as pd

SHAPES_FILE     = "shapes.txt"
STOP_TIMES_FILE = "stop_times.txt"
TRIPS_FILE      = "trips.txt"

print("="*60)
print(" VALIDASI shape_dist_traveled (FULL PRECISION)")
print("="*60)

# Load data
shapes = pd.read_csv(SHAPES_FILE)
stop_times = pd.read_csv(STOP_TIMES_FILE)
trips = pd.read_csv(TRIPS_FILE)

# shape max
shape_max = (
    shapes
    .groupby("shape_id")["shape_dist_traveled"]
    .max()
    .to_dict()
)

# TRIP → SHAPE
trip_to_shape = trips.set_index("trip_id")["shape_id"].to_dict()

# LAST STOP PER TRIP
last_stops = stop_times.loc[
    stop_times.groupby("trip_id")["stop_sequence"].idxmax()
].copy()

# VALIDATION
results = []
mismatch_count = 0

for _, row in last_stops.iterrows():
    trip_id  = row["trip_id"]
    shape_id = trip_to_shape.get(trip_id)

    if not shape_id or shape_id not in shape_max:
        continue

    stop_val  = row["shape_dist_traveled"]
    shape_val = shape_max[shape_id]

    # STRICT COMPARISON (tanpa tolerance)
    match = stop_val == shape_val

    if not match:
        mismatch_count += 1

    results.append({
        "trip_id": trip_id,
        "shape_id": shape_id,
        "stop_times_last": stop_val,
        "shapes_max": shape_val,
        "diff": stop_val - shape_val,
        "status": "OK" if match else "MISMATCH"
    })

df_result = pd.DataFrame(results)

# ================= SUMMARY =================
print(f"\n📊 Total trip dicek : {len(df_result):,}")
print(f"❌ Mismatch         : {mismatch_count:,}")
print(f"✅ Match            : {len(df_result) - mismatch_count:,}")

def print_full_precision(df, n=10):
    print("\n📄 Sample hasil (FULL PRECISION):")
    print(
        f"{'trip_id':<20} {'shape_id':<18} "
        f"{'stop_times_last':<26} {'shapes_max':<26} "
        f"{'diff':<26} {'status'}"
    )

    for _, r in df.head(n).iterrows():
        print(
            f"{r['trip_id']:<20} "
            f"{r['shape_id']:<18} "
            f"{repr(r['stop_times_last']):<26} "
            f"{repr(r['shapes_max']):<26} "
            f"{repr(r['diff']):<26} "
            f"{r['status']}"
        )

print_full_precision(df_result)

# DETAIL MISMATCH
if mismatch_count > 0:
    print("\n⚠️ Detail mismatch (FULL PRECISION):")

    df_mismatch = (
        df_result[df_result["status"] == "MISMATCH"]
        .copy()
    )

    # sort by absolute diff (tanpa ubah nilai)
    df_mismatch["abs_diff"] = df_mismatch["diff"].abs()
    df_mismatch = df_mismatch.sort_values("abs_diff", ascending=False)

    print(
        f"{'trip_id':<20} {'shape_id':<18} "
        f"{'stop_times_last':<26} {'shapes_max':<26} "
        f"{'diff':<26}"
    )

    for _, r in df_mismatch.head(20).iterrows():
        print(
            f"{r['trip_id']:<20} "
            f"{r['shape_id']:<18} "
            f"{repr(r['stop_times_last']):<26} "
            f"{repr(r['shapes_max']):<26} "
            f"{repr(r['diff'])}"
        )
else:
    print("\n✅ Semua trip IDENTIK")

 VALIDASI shape_dist_traveled (FULL PRECISION)

📊 Total trip dicek : 204
❌ Mismatch         : 0
✅ Match            : 204

📄 Sample hasil (FULL PRECISION):
trip_id              shape_id           stop_times_last            shapes_max                 diff                       status
S-ED-PGD-VEL-001     shape_PGD_VEL      5593.213483889901          5593.213483889901          0.0                        OK
S-ED-PGD-VEL-002     shape_PGD_VEL      5593.213483889901          5593.213483889901          0.0                        OK
S-ED-PGD-VEL-003     shape_PGD_VEL      5593.213483889901          5593.213483889901          0.0                        OK
S-ED-PGD-VEL-004     shape_PGD_VEL      5593.213483889901          5593.213483889901          0.0                        OK
S-ED-PGD-VEL-005     shape_PGD_VEL      5593.213483889901          5593.213483889901          0.0                        OK
S-ED-PGD-VEL-006     shape_PGD_VEL      5593.213483889901          5593.213483889901          0.0

#### Cek shapes.txt dan stop_times.txt untuk **`shape_PGD_VEL`** 

In [28]:
import pandas as pd
import folium

shapes     = pd.read_csv('shapes.txt')
trips      = pd.read_csv('trips.txt')
stops      = pd.read_csv('stops.txt')
stop_times = pd.read_csv('stop_times.txt')

target_trip_id = 'S-ED-PGD-VEL-001'

# Ambil shape_id dari trips.txt
trip_row = trips[trips['trip_id'] == target_trip_id]

if len(trip_row) == 0:
    print(f"❌ Trip {target_trip_id} tidak ditemukan di trips.txt")
else:
    shape_id = trip_row['shape_id'].values[0]
    print(f"✅ Trip   : {target_trip_id}")
    print(f"✅ Shape  : {shape_id}")

    # Ambil titik shape
    shape_data = shapes[shapes['shape_id'] == shape_id].sort_values('shape_pt_sequence')

    if len(shape_data) == 0:
        print(f"❌ Shape {shape_id} tidak ditemukan di shapes.txt")
    else:
        print(f"✅ Jumlah titik shape: {len(shape_data)}")

        # Titik tengah peta
        center_lat = shape_data['shape_pt_lat'].mean()
        center_lon = shape_data['shape_pt_lon'].mean()

        # Buat peta
        m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

        # Gambar jalur shape
        coords = list(zip(shape_data['shape_pt_lat'], shape_data['shape_pt_lon']))
        folium.PolyLine(
            coords,
            color='blue',
            weight=4,
            opacity=0.8,
            tooltip=f"Shape: {shape_id} | Trip: {target_trip_id}"
        ).add_to(m)

        # Titik awal dan akhir shape
        folium.Marker(
            coords[0],
            popup=f"START shape {shape_id}",
            icon=folium.Icon(color='green', icon='play')
        ).add_to(m)

        folium.Marker(
            coords[-1],
            popup=f"END shape {shape_id}",
            icon=folium.Icon(color='red', icon='stop')
        ).add_to(m)

        # Gambar stop jika ada di stop_times
        st = stop_times[stop_times['trip_id'] == target_trip_id]

        if len(st) == 0:
            print(f"⚠️  Trip {target_trip_id} tidak ada di stop_times.txt")
            print(f"   Jalur shape tetap digambar tanpa titik stop")
        else:
            st = st.sort_values('stop_sequence').merge(
                stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']],
                on='stop_id', how='left'
            )
            for _, stop in st.iterrows():
                folium.CircleMarker(
                    location=[stop['stop_lat'], stop['stop_lon']],
                    radius=6,
                    color='orange',
                    fill=True,
                    fill_color='orange',
                    fill_opacity=0.9,
                    popup=f"seq:{stop['stop_sequence']} | {stop['stop_id']} | {stop['stop_name']}"
                ).add_to(m)
            print(f"✅ Jumlah stop digambar: {len(st)}")

        # Simpan
        output_file = f'map_Lrt_jakarta_{target_trip_id}.html'
        # m.save(output_file)
        # print(f"\n✅ Peta disimpan: {output_file}")
        # print(f"   Buka di browser untuk melihat rute")

✅ Trip   : S-ED-PGD-VEL-001
✅ Shape  : shape_PGD_VEL
✅ Jumlah titik shape: 73
✅ Jumlah stop digambar: 6


In [29]:
m

#### Cek shapes.txt dan stop_times.txt untuk **`shape_VEL_PGD`** 

In [30]:
import pandas as pd
import folium

shapes     = pd.read_csv('shapes.txt')
trips      = pd.read_csv('trips.txt')
stops      = pd.read_csv('stops.txt')
stop_times = pd.read_csv('stop_times.txt')

target_trip_id = 'S-ED-VEL-PGD-001'

# Ambil shape_id dari trips.txt
trip_row = trips[trips['trip_id'] == target_trip_id]

if len(trip_row) == 0:
    print(f"❌ Trip {target_trip_id} tidak ditemukan di trips.txt")
else:
    shape_id = trip_row['shape_id'].values[0]
    print(f"✅ Trip   : {target_trip_id}")
    print(f"✅ Shape  : {shape_id}")

    # Ambil titik shape
    shape_data = shapes[shapes['shape_id'] == shape_id].sort_values('shape_pt_sequence')

    if len(shape_data) == 0:
        print(f"❌ Shape {shape_id} tidak ditemukan di shapes.txt")
    else:
        print(f"✅ Jumlah titik shape: {len(shape_data)}")

        # Titik tengah peta
        center_lat = shape_data['shape_pt_lat'].mean()
        center_lon = shape_data['shape_pt_lon'].mean()

        # Buat peta
        m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

        # Gambar jalur shape
        coords = list(zip(shape_data['shape_pt_lat'], shape_data['shape_pt_lon']))
        folium.PolyLine(
            coords,
            color='blue',
            weight=4,
            opacity=0.8,
            tooltip=f"Shape: {shape_id} | Trip: {target_trip_id}"
        ).add_to(m)

        # Titik awal dan akhir shape
        folium.Marker(
            coords[0],
            popup=f"START shape {shape_id}",
            icon=folium.Icon(color='green', icon='play')
        ).add_to(m)

        folium.Marker(
            coords[-1],
            popup=f"END shape {shape_id}",
            icon=folium.Icon(color='red', icon='stop')
        ).add_to(m)

        # Gambar stop jika ada di stop_times
        st = stop_times[stop_times['trip_id'] == target_trip_id]

        if len(st) == 0:
            print(f"⚠️  Trip {target_trip_id} tidak ada di stop_times.txt")
            print(f"   Jalur shape tetap digambar tanpa titik stop")
        else:
            st = st.sort_values('stop_sequence').merge(
                stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']],
                on='stop_id', how='left'
            )
            for _, stop in st.iterrows():
                folium.CircleMarker(
                    location=[stop['stop_lat'], stop['stop_lon']],
                    radius=6,
                    color='orange',
                    fill=True,
                    fill_color='orange',
                    fill_opacity=0.9,
                    popup=f"seq:{stop['stop_sequence']} | {stop['stop_id']} | {stop['stop_name']}"
                ).add_to(m)
            print(f"✅ Jumlah stop digambar: {len(st)}")

        # Simpan
        output_file = f'map_lrt_jakarta_{target_trip_id}.html'
        # m.save(output_file)
        # print(f"\n✅ Peta disimpan: {output_file}")
        # print(f"   Buka di browser untuk melihat rute")

✅ Trip   : S-ED-VEL-PGD-001
✅ Shape  : shape_VEL_PGD
✅ Jumlah titik shape: 73
✅ Jumlah stop digambar: 6


In [31]:
m

### Build `stop_times.txt` with formula haversine for shape_dist_traveled

In [32]:
"""
Build stop_times.txt berdasarkan jadwal keberangkatan LRT Jakarta

departure_time  : langsung dari file jadwal keberangkatan
arrival_time    : departure_time - DWELL_TIME (detik), kecuali stasiun pertama & terakhir: arrival = departure

Satuan shape_dist_traveled : METER (konsisten dengan shapes.txt)

Format waktu GTFS:
  - Waktu TIDAK direset ke 00:xx:xx saat melewati tengah malam
  - Melainkan terus bertambah: 24:xx:xx, 25:xx:xx, dst
  - Contoh: kereta berangkat 23:40 tiba 00:01 → ditulis 24:01:xx

Catatan:
  - shape_dist_traveled dihitung menggunakan rumus haversine dengan satuan Meter
"""

import pandas as pd
import csv
import math
import os
from datetime import datetime

# File path 
SHAPES_FILE     = "shapes.txt"
STOPS_FILE      = "stops.txt"
OUTPUT_FILE     = "stop_times.txt"

# Dwell time dalam detik
# arrival = departure - DWELL_TIME
# kecuali stasiun pertama & terakhir: arrival = departure
DWELL_TIME      = 30   # detik

# File jadwal keberangkatan lRT Jakarta
SCHEDULE_FILES = {
    "PGD_VEL_ED": "data/jadwal-keberangkatan/PegangsaanDua_Velodrome.csv",
    "VEL_PGD_ED": "data/jadwal-keberangkatan/Velodrome_PegangsaanDua.csv",
}

# Mapping kode stasiun → stop_id platform
PLATFORM_MAP = {
    "PGD_VEL": {
        "PGD": "GPGD01", "BVU": "GBVU01", "BVS": "GBVS01",
        "PUM": "GPUM01", "EQS": "GEQS01", "VEL": "GVEL01",
    },
    "VEL_PGD": {
        "VEL": "GVEL02", "EQS": "GEQS02", "PUM": "GPUM02",
        "BVS": "GBVS02", "BVU": "GBVU02", "PGD": "GPGD01",
    },
}

# shape_id per jenis trip
SHAPE_MAP = {
    "PGD_VEL"   : "shape_PGD_VEL",
    "VEL_PGD"   : "shape_VEL_PGD"
}


# Formula Haversine 
def haversine_m(lat1, lon1, lat2, lon2):
    """Jarak antara dua titik koordinat dalam METER."""
    R  = 6371000.0
    p1 = math.radians(lat1)
    p2 = math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a  = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))


# Load shapes
def load_shapes(shapes_file):
    """
    Load shapes.txt → dict shape_id → DataFrame (lat, lon, shape_dist_traveled)
    diurutkan berdasarkan shape_pt_sequence.
    """
    df = pd.read_csv(shapes_file)
    shapes = {}
    for sid in df["shape_id"].unique():
        shapes[sid] = (
            df[df["shape_id"] == sid]
            .sort_values("shape_pt_sequence")
            .reset_index(drop=True)
        )
    return shapes


def get_shape_dist(shape_df, stop_lat, stop_lon):
    """
    Cari shape_dist_traveled pada titik shape terdekat ke (stop_lat, stop_lon).
    Return nilai shape_dist_traveled dalam meter.
    """
    dists = shape_df.apply(
        lambda r: haversine_m(stop_lat, stop_lon,
                              r["shape_pt_lat"], r["shape_pt_lon"]),
        axis=1
    )
    nearest_idx = dists.idxmin()
    return float(shape_df.loc[nearest_idx, "shape_dist_traveled"])


# Load stops
def load_stops(stops_file):
    """Load stops.txt → dict stop_id → (stop_lat, stop_lon)."""
    df = pd.read_csv(stops_file, dtype={"stop_id": str})
    return {
        row["stop_id"]: (float(row["stop_lat"]), float(row["stop_lon"]))
        for _, row in df.iterrows()
        if pd.notna(row.get("stop_lat")) and str(row.get("stop_lat", "")).strip() != ""
    }


# Time utils
def parse_time(time_str):
    """
    Parse waktu HH:MM:SS ke total detik.
    Support jam > 24 untuk layanan melewati tengah malam (GTFS compliant).

    Contoh:
        "05:12:00" →  18720
        "23:40:30" →  85230
        "24:00:50" →  86450  (sudah >24, langsung parse tanpa modifikasi)
    """
    parts = str(time_str).strip().split(":")
    h = int(parts[0])
    m = int(parts[1])
    s = int(parts[2]) if len(parts) > 2 else 0
    return h * 3600 + m * 60 + s


def format_time(total_seconds):
    """
    Format total detik ke HH:MM:SS sesuai spesifikasi GTFS.

    Jam TIDAK direset ke 00 saat melewati tengah malam —
    melainkan terus bertambah (24:xx:xx, 25:xx:xx, dst).

    Contoh:
        85230  → "23:40:30"
        86450  → "24:00:50"  ✅ bukan "00:00:50"
        90000  → "25:00:00"
    """
    h = total_seconds // 3600
    m = (total_seconds % 3600) // 60
    s = total_seconds % 60
    return f"{h:02d}:{m:02d}:{s:02d}"


def fix_midnight_rollover(current_sec, prev_sec):
    """
    Koreksi midnight rollover sesuai spesifikasi GTFS.

    GTFS mengharuskan waktu selalu ASCENDING dalam satu trip.
    Kalau current_sec < prev_sec, artinya CSV menyimpan waktu
    sebagai 00:xx:xx setelah melewati tengah malam — harus
    dikoreksi dengan menambah 86400 detik (24 jam).

    Menggunakan while loop untuk mengakomodasi edge case trip
    yang melewati tengah malam lebih dari satu kali.

    Contoh:
        prev    = 23:40:30 → 85230 detik
        current = 00:00:50 →    50 detik  ← lebih kecil → rollover!
        koreksi = 50 + 86400 = 86450      → format_time → "24:00:50" ✅

    Args:
        current_sec : waktu stop saat ini dalam detik
        prev_sec    : waktu stop sebelumnya dalam detik

    Returns:
        current_sec yang sudah dikoreksi (int)
    """
    while current_sec < prev_sec:
        current_sec += 86400  # +24 jam
    return current_sec


def calc_arrival(departure_sec, stop_sequence, total_stops):
    """
    Hitung arrival_time dalam detik.

    Aturan:
      - Stasiun pertama  (seq=0)        : arrival = departure (no dwell)
      - Stasiun terakhir (seq=total-1)  : arrival = departure (no dwell)
      - Stasiun lainnya                 : arrival = departure - DWELL_TIME

    Catatan: arrival tidak perlu rollover correction karena selalu
    dihitung SETELAH departure sudah dikoreksi.
    """
    if stop_sequence == 0 or stop_sequence == total_stops - 1:
        return departure_sec
    return departure_sec - DWELL_TIME


# Deteksi jenis trip
def get_trip_type(trip_id):
    """
    Tentukan jenis trip berdasarkan trip_id.
    Return: (trip_type, platform_dir)
    """
   
    if "PGD-VEL" in trip_id:
        return "PGD_VEL", "PGD_VEL", "Velodrome"
    elif "VEL-PGD" in trip_id:
        return "VEL_PGD", "VEL_PGD", "Pegangsaan Dua"
    else:
        raise ValueError(f"Tidak bisa mendeteksi jenis trip dari: {trip_id}")


# Generate stop_times 
def generate_stop_times(schedule_df, platform_dir, shapes, stops):
    """
    Generate baris stop_times dari satu DataFrame jadwal.

    Proses per trip:
      1. Iterasi setiap stasiun secara berurutan
      2. Parse departure_time dari CSV
      3. Koreksi midnight rollover jika perlu (00:xx → 24:xx) 
      4. Hitung arrival_time dengan DWELL_TIME (setelah koreksi)
      5. Map stop_id dari PLATFORM_MAP
      6. Hitung shape_dist_traveled dari shapes.txt

    Returns:
        (list of dict, int rollover_count)
    """
    rows                    = []
    plat_map                = PLATFORM_MAP[platform_dir]
    station_cols            = [c for c in schedule_df.columns
                               if c not in ("trip_id", "Full Trip")]
    midnight_rollover_count = 0

    for _, trip_row in schedule_df.iterrows():
        trip_id      = trip_row["trip_id"]
        trip_type, _, stop_headsign = get_trip_type(trip_id)
        shape_id     = SHAPE_MAP[trip_type]
        shape_df     = shapes.get(shape_id)

        # Filter stasiun yang ada jadwalnya (tidak kosong / NaN)
        active_stations = [
            s for s in station_cols
            if pd.notna(trip_row[s]) and str(trip_row[s]).strip() != ""
        ]
        total_stops = len(active_stations)

        # ── Hitung shape_dist_traveled per stasiun ────────────────
        shape_dists_raw = {}
        if shape_df is not None:
            for stn in active_stations:
                stop_id = plat_map.get(stn)
                if stop_id and stop_id in stops:
                    lat, lon = stops[stop_id]
                    shape_dists_raw[stn] = get_shape_dist(shape_df, lat, lon)
                else:
                    shape_dists_raw[stn] = None
                    
        shape_dists = {
                s: (shape_dists_raw[s]
                    if shape_dists_raw.get(s) is not None else "")
                for s in active_stations
            }

        # Iterasi per stasiun dengan midnight rollover 
        prev_dep_sec = -1  # sentinel: belum ada stop sebelumnya

        for seq, stn in enumerate(active_stations):
            raw_dep_sec = parse_time(trip_row[stn])

            # ── Koreksi midnight rollover ─────────────────────────
            # Harus dilakukan SEBELUM calc_arrival agar arrival
            # juga menggunakan waktu yang sudah dikoreksi.
            #
            # Contoh tanpa koreksi (SALAH):
            #   IST  dep=23:40:30 → 85230 detik
            #   PGD  dep=00:00:50 →    50 detik  ← lebih kecil!
            #   arr  = 50 - 30 = 20 detik → "00:00:20" ← SALAH
            #
            # Contoh dengan koreksi (BENAR):
            #   IST  dep=23:40:30 → 85230 detik
            #   PGD  dep=00:00:50 → 86450 detik  ← +86400 ✅
            #   arr  = 86450 - 0  = 86450 detik  → "24:00:50" ✅
            if prev_dep_sec >= 0:
                corrected_dep_sec = fix_midnight_rollover(
                    raw_dep_sec, prev_dep_sec
                )
                if corrected_dep_sec != raw_dep_sec:
                    midnight_rollover_count += 1
            else:
                corrected_dep_sec = raw_dep_sec

            arr_sec       = calc_arrival(corrected_dep_sec, seq, total_stops)
            dist_traveled = shape_dists.get(stn, "")

            rows.append({
                "trip_id"        : trip_id,
                "arrival_time"   : format_time(arr_sec),
                "departure_time" : format_time(corrected_dep_sec),
                "stop_id"        : plat_map.get(stn, ""),
                "stop_sequence"  : seq,
                "stop_headsign"  : stop_headsign,
                "pickup_type"    : 0,
                "drop_off_type"  : 0,
                "continuous_pickup": 1,
                "continuous_drop_off": 1,
                "timepoint": 1,
                "shape_dist_traveled": dist_traveled,
            })

            prev_dep_sec = corrected_dep_sec  # update untuk iterasi berikutnya

    return rows, midnight_rollover_count


# Validasi ascending per trip
def validate_ascending(df_stop_times):
    """
    Validasi bahwa departure_time selalu ascending dalam setiap trip.
    Return: list of dict issue (kosong = semua OK)
    """
    issues = []
    for trip_id, group in df_stop_times.groupby("trip_id"):
        group    = group.sort_values("stop_sequence")
        dep_secs = group["departure_time"].apply(parse_time).tolist()
        for i in range(1, len(dep_secs)):
            if dep_secs[i] < dep_secs[i - 1]:
                issues.append({
                    "trip_id"   : trip_id,
                    "stop_seq"  : group.iloc[i]["stop_sequence"],
                    "stop_id"   : group.iloc[i]["stop_id"],
                    "prev_time" : group.iloc[i - 1]["departure_time"],
                    "curr_time" : group.iloc[i]["departure_time"],
                })
    return issues


def main():
    print("=" * 60)
    print(" Build stop_times.txt  |  LRT Jakarta Lin Selatan Fase 1A")
    print(f" {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f" Dwell time  : {DWELL_TIME} detik")
    print(f" Satuan dist : METER")
    print(f" Format waktu: HH:MM:SS (>24 jam untuk lewat tengah malam)")
    print("=" * 60)

    # Validasi file wajib
    for f in [SHAPES_FILE, STOPS_FILE]:
        if not os.path.exists(f):
            print(f"\n❌ File tidak ditemukan: {f}")
            return

    # Load shapes & stops
    print("\n📂 Loading shapes.txt ...")
    shapes = load_shapes(SHAPES_FILE)
    print(f"   Shape tersedia : {list(shapes.keys())}")

    print("📂 Loading stops.txt ...")
    stops = load_stops(STOPS_FILE)
    print(f"   Stop tersedia  : {len(stops)} stop")

    # Proses setiap file jadwal
    all_rows             = []
    total_rollover_count = 0
    seen_trip_ids        = set()  # guard duplikat trip_id antar file

    for key, filename in SCHEDULE_FILES.items():
        if not os.path.exists(filename):
            print(f"\n⚠️  File tidak ditemukan: {filename} — dilewati")
            continue

        platform_dir = "PGD_VEL" if key.startswith("PGD") else "VEL_PGD"

        print(f"\n{'─'*55}")
        print(f"📋 Memproses : {filename}")
        print(f"   Arah      : {platform_dir}")

        df = pd.read_csv(filename)
        if "Full Trip" in df.columns:
            df = df.drop(columns=["Full Trip"])

        # Guard duplikat trip_id antar file
        incoming_ids  = set(df["trip_id"].unique())
        duplicate_ids = incoming_ids & seen_trip_ids
        if duplicate_ids:
            print(f"   ⚠️  {len(duplicate_ids)} trip_id duplikat dilewati:")
            for tid in sorted(duplicate_ids)[:5]:
                print(f"      {tid}")
            df = df[~df["trip_id"].isin(duplicate_ids)]
        seen_trip_ids.update(set(df["trip_id"].unique()))

        rows, rollover_count = generate_stop_times(
            df, platform_dir, shapes, stops
        )
        all_rows.extend(rows)
        total_rollover_count += rollover_count

        print(f"   Trip              : {df['trip_id'].nunique()}")
        print(f"   Baris             : {len(rows)}")
        print(f"   Midnight rollover : {rollover_count} koreksi")

    # Save stop_times.txt
    if not all_rows:
        print("\n⚠️  Tidak ada baris yang dihasilkan.")
        return

    fieldnames = [
        "trip_id", "arrival_time", "departure_time",
        "stop_id", "stop_sequence", "stop_headsign", "pickup_type", "drop_off_type",
        "continuous_pickup", "continuous_drop_off", "timepoint", "shape_dist_traveled"
    ]

    df_out = pd.DataFrame(all_rows, columns=fieldnames)
    df_out.to_csv(
        OUTPUT_FILE,
        index=False,
        encoding="utf-8",
        lineterminator="\n",
        quoting=csv.QUOTE_MINIMAL,
    )

    # Validasi ascending
    print(f"\n{'─'*55}")
    print("🔍 Validasi ascending time per trip ...")
    issues = validate_ascending(df_out)
    if issues:
        print(f"   ⚠️  {len(issues)} waktu tidak ascending:")
        for iss in issues[:10]:
            print(f"      trip={iss['trip_id']}  seq={iss['stop_seq']}"
                  f"  stop={iss['stop_id']}"
                  f"  {iss['prev_time']} → {iss['curr_time']}")
        if len(issues) > 10:
            print(f"      ... dan {len(issues) - 10} lainnya")
    else:
        print("   ✅ Semua trip ascending — tidak ada masalah waktu!")

    # Summary
    print(f"\n{'='*55}")
    print(f"✅ stop_times.txt berhasil dibuat!")
    print(f"   Path              : {OUTPUT_FILE}")
    print(f"   Total baris       : {len(df_out):,}")
    print(f"   Total trip        : {df_out['trip_id'].nunique():,}")
    print(f"   Midnight rollover : {total_rollover_count} koreksi total")

    print(f"\n📄 Preview 5 baris pertama:")
    print(df_out.head().to_string(index=False))

    print(f"\n📄 Preview 5 baris terakhir (cek format >24 jam):")
    print(df_out.tail().to_string(index=False))


if __name__ == "__main__":
    main()

 Build stop_times.txt  |  LRT Jakarta Lin Selatan Fase 1A
 2026-04-26 21:34:21
 Dwell time  : 30 detik
 Satuan dist : METER
 Format waktu: HH:MM:SS (>24 jam untuk lewat tengah malam)

📂 Loading shapes.txt ...
   Shape tersedia : ['shape_PGD_VEL', 'shape_VEL_PGD']
📂 Loading stops.txt ...
   Stop tersedia  : 48 stop

───────────────────────────────────────────────────────
📋 Memproses : data/jadwal-keberangkatan/PegangsaanDua_Velodrome.csv
   Arah      : PGD_VEL
   Trip              : 102
   Baris             : 612
   Midnight rollover : 0 koreksi

───────────────────────────────────────────────────────
📋 Memproses : data/jadwal-keberangkatan/Velodrome_PegangsaanDua.csv
   Arah      : VEL_PGD
   Trip              : 102
   Baris             : 612
   Midnight rollover : 0 koreksi

───────────────────────────────────────────────────────
🔍 Validasi ascending time per trip ...
   ✅ Semua trip ascending — tidak ada masalah waktu!

✅ stop_times.txt berhasil dibuat!
   Path              : stop_ti

### Validate shape_dist_traveled Consistency (Stop Times vs Shapes) result formula haversine

In [33]:
import pandas as pd

SHAPES_FILE     = "shapes.txt"
STOP_TIMES_FILE = "stop_times.txt"
TRIPS_FILE      = "trips.txt"

print("="*60)
print(" VALIDASI shape_dist_traveled (FULL PRECISION)")
print("="*60)

# Load data
shapes = pd.read_csv(SHAPES_FILE)
stop_times = pd.read_csv(STOP_TIMES_FILE)
trips = pd.read_csv(TRIPS_FILE)

# shape max
shape_max = (
    shapes
    .groupby("shape_id")["shape_dist_traveled"]
    .max()
    .to_dict()
)

# TRIP → SHAPE
trip_to_shape = trips.set_index("trip_id")["shape_id"].to_dict()

# LAST STOP PER TRIP
last_stops = stop_times.loc[
    stop_times.groupby("trip_id")["stop_sequence"].idxmax()
].copy()

# VALIDATION
results = []
mismatch_count = 0

for _, row in last_stops.iterrows():
    trip_id  = row["trip_id"]
    shape_id = trip_to_shape.get(trip_id)

    if not shape_id or shape_id not in shape_max:
        continue

    stop_val  = row["shape_dist_traveled"]
    shape_val = shape_max[shape_id]

    # STRICT COMPARISON (tanpa tolerance)
    match = stop_val == shape_val

    if not match:
        mismatch_count += 1

    results.append({
        "trip_id": trip_id,
        "shape_id": shape_id,
        "stop_times_last": stop_val,
        "shapes_max": shape_val,
        "diff": stop_val - shape_val,
        "status": "OK" if match else "MISMATCH"
    })

df_result = pd.DataFrame(results)

# ================= SUMMARY =================
print(f"\n📊 Total trip dicek : {len(df_result):,}")
print(f"❌ Mismatch         : {mismatch_count:,}")
print(f"✅ Match            : {len(df_result) - mismatch_count:,}")

def print_full_precision(df, n=10):
    print("\n📄 Sample hasil (FULL PRECISION):")
    print(
        f"{'trip_id':<20} {'shape_id':<18} "
        f"{'stop_times_last':<26} {'shapes_max':<26} "
        f"{'diff':<26} {'status'}"
    )

    for _, r in df.head(n).iterrows():
        print(
            f"{r['trip_id']:<20} "
            f"{r['shape_id']:<18} "
            f"{repr(r['stop_times_last']):<26} "
            f"{repr(r['shapes_max']):<26} "
            f"{repr(r['diff']):<26} "
            f"{r['status']}"
        )

print_full_precision(df_result)

# DETAIL MISMATCH
if mismatch_count > 0:
    print("\n⚠️ Detail mismatch (FULL PRECISION):")

    df_mismatch = (
        df_result[df_result["status"] == "MISMATCH"]
        .copy()
    )

    # sort by absolute diff (tanpa ubah nilai)
    df_mismatch["abs_diff"] = df_mismatch["diff"].abs()
    df_mismatch = df_mismatch.sort_values("abs_diff", ascending=False)

    print(
        f"{'trip_id':<20} {'shape_id':<18} "
        f"{'stop_times_last':<26} {'shapes_max':<26} "
        f"{'diff':<26}"
    )

    for _, r in df_mismatch.head(20).iterrows():
        print(
            f"{r['trip_id']:<20} "
            f"{r['shape_id']:<18} "
            f"{repr(r['stop_times_last']):<26} "
            f"{repr(r['shapes_max']):<26} "
            f"{repr(r['diff'])}"
        )
else:
    print("\n✅ Semua trip IDENTIK")

 VALIDASI shape_dist_traveled (FULL PRECISION)

📊 Total trip dicek : 204
❌ Mismatch         : 0
✅ Match            : 204

📄 Sample hasil (FULL PRECISION):
trip_id              shape_id           stop_times_last            shapes_max                 diff                       status
S-ED-PGD-VEL-001     shape_PGD_VEL      5593.213483889901          5593.213483889901          0.0                        OK
S-ED-PGD-VEL-002     shape_PGD_VEL      5593.213483889901          5593.213483889901          0.0                        OK
S-ED-PGD-VEL-003     shape_PGD_VEL      5593.213483889901          5593.213483889901          0.0                        OK
S-ED-PGD-VEL-004     shape_PGD_VEL      5593.213483889901          5593.213483889901          0.0                        OK
S-ED-PGD-VEL-005     shape_PGD_VEL      5593.213483889901          5593.213483889901          0.0                        OK
S-ED-PGD-VEL-006     shape_PGD_VEL      5593.213483889901          5593.213483889901          0.0

#### Cek shapes.txt dan stop_times.txt untuk **`shape_PGD_VEL`** 

In [34]:
import pandas as pd
import folium

shapes     = pd.read_csv('shapes.txt')
trips      = pd.read_csv('trips.txt')
stops      = pd.read_csv('stops.txt')
stop_times = pd.read_csv('stop_times.txt')

target_trip_id = 'S-ED-PGD-VEL-001'

# Ambil shape_id dari trips.txt
trip_row = trips[trips['trip_id'] == target_trip_id]

if len(trip_row) == 0:
    print(f"❌ Trip {target_trip_id} tidak ditemukan di trips.txt")
else:
    shape_id = trip_row['shape_id'].values[0]
    print(f"✅ Trip   : {target_trip_id}")
    print(f"✅ Shape  : {shape_id}")

    # Ambil titik shape
    shape_data = shapes[shapes['shape_id'] == shape_id].sort_values('shape_pt_sequence')

    if len(shape_data) == 0:
        print(f"❌ Shape {shape_id} tidak ditemukan di shapes.txt")
    else:
        print(f"✅ Jumlah titik shape: {len(shape_data)}")

        # Titik tengah peta
        center_lat = shape_data['shape_pt_lat'].mean()
        center_lon = shape_data['shape_pt_lon'].mean()

        # Buat peta
        m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

        # Gambar jalur shape
        coords = list(zip(shape_data['shape_pt_lat'], shape_data['shape_pt_lon']))
        folium.PolyLine(
            coords,
            color='blue',
            weight=4,
            opacity=0.8,
            tooltip=f"Shape: {shape_id} | Trip: {target_trip_id}"
        ).add_to(m)

        # Titik awal dan akhir shape
        folium.Marker(
            coords[0],
            popup=f"START shape {shape_id}",
            icon=folium.Icon(color='green', icon='play')
        ).add_to(m)

        folium.Marker(
            coords[-1],
            popup=f"END shape {shape_id}",
            icon=folium.Icon(color='red', icon='stop')
        ).add_to(m)

        # Gambar stop jika ada di stop_times
        st = stop_times[stop_times['trip_id'] == target_trip_id]

        if len(st) == 0:
            print(f"⚠️  Trip {target_trip_id} tidak ada di stop_times.txt")
            print(f"   Jalur shape tetap digambar tanpa titik stop")
        else:
            st = st.sort_values('stop_sequence').merge(
                stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']],
                on='stop_id', how='left'
            )
            for _, stop in st.iterrows():
                folium.CircleMarker(
                    location=[stop['stop_lat'], stop['stop_lon']],
                    radius=6,
                    color='orange',
                    fill=True,
                    fill_color='orange',
                    fill_opacity=0.9,
                    popup=f"seq:{stop['stop_sequence']} | {stop['stop_id']} | {stop['stop_name']}"
                ).add_to(m)
            print(f"✅ Jumlah stop digambar: {len(st)}")

        # Simpan
        output_file = f'map_lrt_jakarta_{target_trip_id}.html'
        # m.save(output_file)
        # print(f"\n✅ Peta disimpan: {output_file}")
        # print(f"   Buka di browser untuk melihat rute")

✅ Trip   : S-ED-PGD-VEL-001
✅ Shape  : shape_PGD_VEL
✅ Jumlah titik shape: 73
✅ Jumlah stop digambar: 6


In [35]:
m

#### Cek shapes.txt dan stop_times.txt untuk **`shape_VEL_PGD`** 

In [36]:
import pandas as pd
import folium

shapes     = pd.read_csv('shapes.txt')
trips      = pd.read_csv('trips.txt')
stops      = pd.read_csv('stops.txt')
stop_times = pd.read_csv('stop_times.txt')

target_trip_id = 'S-ED-VEL-PGD-001'

# Ambil shape_id dari trips.txt
trip_row = trips[trips['trip_id'] == target_trip_id]

if len(trip_row) == 0:
    print(f"❌ Trip {target_trip_id} tidak ditemukan di trips.txt")
else:
    shape_id = trip_row['shape_id'].values[0]
    print(f"✅ Trip   : {target_trip_id}")
    print(f"✅ Shape  : {shape_id}")

    # Ambil titik shape
    shape_data = shapes[shapes['shape_id'] == shape_id].sort_values('shape_pt_sequence')

    if len(shape_data) == 0:
        print(f"❌ Shape {shape_id} tidak ditemukan di shapes.txt")
    else:
        print(f"✅ Jumlah titik shape: {len(shape_data)}")

        # Titik tengah peta
        center_lat = shape_data['shape_pt_lat'].mean()
        center_lon = shape_data['shape_pt_lon'].mean()

        # Buat peta
        m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

        # Gambar jalur shape
        coords = list(zip(shape_data['shape_pt_lat'], shape_data['shape_pt_lon']))
        folium.PolyLine(
            coords,
            color='blue',
            weight=4,
            opacity=0.8,
            tooltip=f"Shape: {shape_id} | Trip: {target_trip_id}"
        ).add_to(m)

        # Titik awal dan akhir shape
        folium.Marker(
            coords[0],
            popup=f"START shape {shape_id}",
            icon=folium.Icon(color='green', icon='play')
        ).add_to(m)

        folium.Marker(
            coords[-1],
            popup=f"END shape {shape_id}",
            icon=folium.Icon(color='red', icon='stop')
        ).add_to(m)

        # Gambar stop jika ada di stop_times
        st = stop_times[stop_times['trip_id'] == target_trip_id]

        if len(st) == 0:
            print(f"⚠️  Trip {target_trip_id} tidak ada di stop_times.txt")
            print(f"   Jalur shape tetap digambar tanpa titik stop")
        else:
            st = st.sort_values('stop_sequence').merge(
                stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']],
                on='stop_id', how='left'
            )
            for _, stop in st.iterrows():
                folium.CircleMarker(
                    location=[stop['stop_lat'], stop['stop_lon']],
                    radius=6,
                    color='orange',
                    fill=True,
                    fill_color='orange',
                    fill_opacity=0.9,
                    popup=f"seq:{stop['stop_sequence']} | {stop['stop_id']} | {stop['stop_name']}"
                ).add_to(m)
            print(f"✅ Jumlah stop digambar: {len(st)}")

        # Simpan
        output_file = f'map_lrt_jakarta_{target_trip_id}.html'
        # m.save(output_file)
        # print(f"\n✅ Peta disimpan: {output_file}")
        # print(f"   Buka di browser untuk melihat rute")

✅ Trip   : S-ED-VEL-PGD-001
✅ Shape  : shape_VEL_PGD
✅ Jumlah titik shape: 73
✅ Jumlah stop digambar: 6


In [37]:
m

## fare_attributes.txt (Optional)

File `fare_attributes.txt` mendefinisikan **struktur tarif dan aturan pembayaran** layanan transportasi. File ini merupakan bagian dari **GTFS Fares V1**, skema tarif versi lama yang lebih sederhana. Setiap baris mewakili satu kelas tarif dengan harga, mata uang, metode pembayaran, dan ketentuan transfer.

---

**GTFS Fares V1 vs V2**

GTFS menyediakan dua skema tarif yang boleh digunakan bersamaan dalam satu dataset, namun hanya satu yang boleh digunakan oleh aplikasi konsumen:

| Aspek | **Fares V1** | **Fares V2** |
|---|---|---|
| File utama | `fare_attributes.txt` + `fare_rules.txt` | `fare_media.txt`, `fare_products.txt`, `fare_leg_rules.txt`, dst |
| Kompleksitas | Sederhana | Lebih detail dan fleksibel |
| Rekomendasi | Legacy | **V2 diprioritaskan** jika keduanya ada |
| Cocok untuk | Tarif sederhana berbasis zona/rute | Tarif kompleks multi-produk |

---

**Field yang Direferensikan di File Lain**

| Field | Direferensikan di | Keterangan |
|---|---|---|
| `fare_id` | `fare_rules.txt` | Setiap aturan tarif merujuk ke `fare_id` |
| `agency_id` | `agency.txt` | Merujuk ke operator yang menerapkan tarif |

---

**Skema kolom file fare_attributes.txt**

| Field | Tipe Data | Keharusan | Deskripsi |
|---|---|---|---|
| `fare_id` | Unique ID | **Wajib** | ID unik pengenal kelas tarif. |
| `price` | Non-negative float | **Wajib** | Nominal tarif dalam satuan mata uang yang ditentukan oleh `currency_type`.  |
| `currency_type` | Currency code | **Wajib** | Kode mata uang ISO 4217. |
| `payment_method` | Enum | **Wajib** | Metode pembayaran. `0` = bayar saat di kendaraan, `1` = bayar sebelum naik. |
| `transfers` | Enum | **Wajib** | Jumlah transfer yang diizinkan. `0` = tidak ada transfer, `1` = 1x transfer, `2` = 2x transfer, kosong = tidak terbatas. |
| `agency_id` | Foreign ID → `agency.agency_id` | **Kondisional** | ID operator yang menerapkan tarif ini. **Wajib** jika dataset berisi lebih dari satu operator. |
| `transfer_duration` | Non-negative integer | Opsional | Durasi tiket berlaku dalam detik sebelum transfer kedaluwarsa. Jika `transfers=0`, field ini bisa digunakan untuk menunjukkan berapa lama tiket valid. |

---

### Panduan Pengisian `fare_attributes.txt` — LRT Jakarta Lin Selatan Fase 1A

---

**`fare_id`**

Diisi dengan ID unik pengenal kelas tarif yang direferensikan dari `fare_rules.txt`. Untuk LRT Jakarta terdapat 1 kelas tarif yaitu flat price sebesar Rp 5.000:

| `fare_id` | Tarif |
|---|---|
| `LRTJ_FP` | Rp 5.000 |

---

**`price`**

Nominal tarif dalam satuan mata uang yang ditentukan oleh `currency_type`. Tarif LRT Jakarta untuk semua perjalanan adalah flat price sebesar Rp5.000.

| `fare_id` | `price` |
|---|---|
| `LRTJ_FP` | `5000` |

---

**`currency_type`**

Kode mata uang ISO 4217 sesuai dengan mata uang yang berlaku. Untuk LRT Jakarta diisi `IDR` (Indonesian Rupiah) karena seluruh transaksi menggunakan Rupiah.

---

**`payment_method`**

Metode pembayaran tarif. Untuk LRT Jakarta diisi `1`secara teknis penumpang melakukan tap kartu di gate sebelum masuk peron, LRT Jakarta menerima pembayaran menggunakan kartu uang elektronik dari berbagai lembaga perbankan yang tergabung dalam Himbara seperti BNI, BRI, dan Bank Mandiri, serta Bank DKI dan Bank BCA. Selain itu tersedia juga pembayaran melalui aplikasi LinkAja, JakLingko Apps, tiket QR, kartu Mastercard contactless, dan QRIS Tap.

| Nilai | Arti |
|:---:|---|
| `0` | Bayar saat di kendaraan |
| `1` | Bayar sebelum boarding |

---

**`transfers`**

Jumlah transfer yang diizinkan dalam satu tiket. Untuk LRT Jakarta diisi `0` karena setiap perjalanan LRT Jakarta berdiri sendiri tanpa transfer gratis ke rute lain dalam satu tarif. Tarif integrasi dengan moda lain (TransJakarta, KRL, LRT) dikelola secara terpisah oleh sistem Jak Lingko dan bukan bagian dari `fare_attributes.txt` LRT Jakarta.

| Nilai | Arti |
|:---:|---|
| `0` | Tidak ada transfer |
| `1` | 1x transfer |
| `2` | 2x transfer |
| kosong | Transfer tidak terbatas |

---

**`agency_id`**

Merujuk ke `agency_id` yang didefinisikan di `agency.txt`. Untuk LRT Jakarta diisi `LRTJ` sesuai dengan ID operator yang sudah ditetapkan.

---

**`transfer_duration`**

Durasi tiket berlaku dalam detik sebelum transfer kedaluwarsa. Untuk LRT Jakarta **dikosongkan** karena `transfers=0` tidak ada mekanisme transfer yang perlu dibatasi durasinya.

In [38]:
import pandas as pd
import os
import csv

# Data fare attributes LRT Jakarta
fare_attributes_data = {
    'fare_id': ['LRTJ_FP'],
    'price': [5000],
    'currency_type': ['IDR'],
    'payment_method': [0],
    'transfers': [''],
    'agency_id': ['LRTJ'],
    'transfer_duration': [''],
}

df_fare_attributes = pd.DataFrame(fare_attributes_data)

df_fare_attributes.to_csv(
    "fare_attributes.txt",
    index=False,
    encoding='utf-8',        
    lineterminator='\n',         
    quoting=csv.QUOTE_MINIMAL   
)

# Tampilkan hasil
print("✅ File fare_attributes.txt berhasil dibuat!")
print("\n📄 Preview isi file:")
df_fare_attributes

✅ File fare_attributes.txt berhasil dibuat!

📄 Preview isi file:


,fare_id,price,currency_type,payment_method,transfers,agency_id,transfer_duration
0,LRTJ_FP,5000,IDR,0,,LRTJ,


## fare_rules.txt (Optional)

File `fare_rules.txt` mendefinisikan **aturan penerapan tarif** dari `fare_attributes.txt` terhadap perjalanan penumpang. File ini menghubungkan kelas tarif dengan rute, zona asal, zona tujuan, atau zona yang dilalui sehingga sistem dapat menentukan tarif yang tepat untuk setiap kombinasi perjalanan.

Sebagian besar struktur tarif menggunakan kombinasi dari tiga pendekatan berikut:

| Pendekatan | Keterangan |
|---|---|
| **Berbasis asal & tujuan** | Tarif ditentukan dari zona asal dan zona tujuan | 
| **Berbasis zona yang dilalui** | Tarif ditentukan dari zona-zona yang dilewati selama perjalanan | 
| **Berbasis rute** | Tarif berlaku untuk rute tertentu | 

---

**Field yang Direferensikan di File Lain**

| Field | Merujuk ke | Keterangan |
|---|---|---|
| `fare_id` | `fare_attributes.fare_id` | Detail tarif (harga, mata uang, metode bayar) |
| `route_id` | `routes.route_id` | Rute yang menerapkan tarif ini |
| `origin_id` | `stops.zone_id` | Zona asal penumpang |
| `destination_id` | `stops.zone_id` | Zona tujuan penumpang |
| `contains_id` | `stops.zone_id` | Zona yang dilewati selama perjalanan |

---

**Skema kolom fare_rules.txt**

| Field | Tipe Data | Status | Deskripsi |
|---|---|---|---|
| `fare_id` | Foreign ID → `fare_attributes.fare_id` | **Wajib** | ID kelas tarif yang diterapkan.|
| `route_id` | Foreign ID → `routes.route_id` | Opsional | ID rute yang menerapkan tarif ini. Jika beberapa rute memiliki tarif yang sama, buat satu baris per rute.|
| `origin_id` | Foreign ID → `stops.zone_id` | Opsional | Zona asal perjalanan. Merujuk ke `zone_id` di `stops.txt`. Jika satu kelas tarif memiliki beberapa zona asal, buat satu baris per zona. |
| `destination_id` | Foreign ID → `stops.zone_id` | Opsional | Zona tujuan perjalanan. Merujuk ke `zone_id` di `stops.txt`. |
| `contains_id` | Foreign ID → `stops.zone_id` | Opsional | Zona yang harus dilalui agar tarif berlaku. **Semua** `contains_id` yang terdaftar untuk satu `fare_id` harus terpenuhi. Jika satu zona tidak dilalui, tarif tersebut tidak berlaku. |

### Panduan Pengisian `fare_rules.txt` — LRT Jakarta Lin Utara-Selatan Fase 1

---

**`fare_id`**

Merujuk ke `fare_id` yang sudah didefinisikan di `fare_attributes.txt`. Hanya terdapat 1 kelas tarif untuk LRT Jakarta, dengan tarif flate Rp5.000 untuk semua perjalanan:

| `fare_id` | Tarif | 
|---|---|
| `LRTJ_FP` | Rp 5.000 |

---

**`route_id`**

Diisi dengan `LRT_S` sesuai dengan `route_id` yang didefinisikan di `routes.txt`, karena seluruh aturan tarif ini hanya berlaku untuk Lin Selatan Fase 1A.

---

**`origin_id`** & **`destination_id`**

Diisi dengan kode stasiun yang menjadi zona asal dan zona tujuan perjalanan merujuk ke `zone_id` di `stops.txt`. Tarif LRT Jakarta adalah flat Rp. 5000 sehingga untuk field  **`origin_id`** & **`destination_id`** dikosong karena berlaku untuk semua trip.

---

**`contains_id`**

Tidak digunakan untuk LRT Jakarta dikosongkan pada semua baris karena tarif sudah ditentukan secara eksplisit.

In [39]:
import pandas as pd
import os
import csv

# Data fare rules LRT Jakarta
fare_rules_data = {
    'fare_id': ['LRTJ_FP'],
    'route_id': ['LRTJ_S'],
    'origin_id': [''],
    'destination_id': [''],
    'contains_id': [''],
}

df_fare_rules = pd.DataFrame(fare_rules_data)

df_fare_rules.to_csv(
    "fare_rules.txt",
    index=False,
    encoding='utf-8',        
    lineterminator='\n',         
    quoting=csv.QUOTE_MINIMAL   
)

# Tampilkan hasil
print("✅ File fare_rules.txt berhasil dibuat!")
print("\n📄 Preview isi file:")
df_fare_rules

✅ File fare_rules.txt berhasil dibuat!

📄 Preview isi file:


,fare_id,route_id,origin_id,destination_id,contains_id
0,LRTJ_FP,LRTJ_S,,,
